# NetraScan.AI: Three Transformers, One Vision
**Research and engineering notebook | V2**

Three-class cataract image classification using ViT-B/16, Swin-Tiny and MobileViT-S, with majority voting, Swin Grad-CAM, local evidence-grounded GenAI and PDF reporting.

## Scope and execution notes
This is a curated record of the development experiments, not a self-contained installation notebook. The original notebook is retained separately. Outputs have been cleared to avoid publishing image data, machine-specific logs and private reports. Selected recorded results are summarized below. The code has not been re-executed after editorial cleanup.

- Environment: Windows, Python 3.11.5, PyTorch 2.9.0+cu128, torchvision 0.24.0+cu128, timm 1.0.22.
- Replace the example workspace `C:\NetraScanWorkspace` and SSD drive `E:` with your local paths. The original training directory name `MY PROJECT E PAARVAI` is retained for compatibility with the experiment layout.
- Dataset images, CSV manifests (including valid local image paths), checkpoints, V1 source/runtime and local LLM assets are external prerequisites. They are not embedded in this notebook.
- Execute sections selectively in their intended environment. Training, downloads, package installation, workspace creation, server launch and SSD deployment have side effects. Do not use Run All on an existing installation.
- Research prototype only; internal held-out evaluation does not establish clinical validity or patient-level independence.

## 1. Environment and initial experiment
The initial ViT experiment motivated a review of cross-partition image similarity. Its validation score is not used as the final performance estimate.

### 01. Research implementation

In [ ]:
# STEP 01 — Research implementation
#cell 1
import sys
import importlib.util

print("Python version:", sys.version.split()[0])

try:
    import torch

    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA version:", torch.version.cuda)
        print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
    else:
        print("GPU acceleration unavailable")

except ImportError:
    print("PyTorch is not installed in this environment")

for package in ["timm", "torchvision", "pandas", "sklearn", "PIL", "matplotlib", "tqdm"]:
    status = "Installed" if importlib.util.find_spec(package) else "Missing"
    print(f"{package}: {status}")

### 02. Research implementation

In [ ]:
# STEP 02 — Research implementation
#cell 2
%pip install --upgrade --force-reinstall torch==2.9.0 torchvision==0.24.0 --index-url https://download.pytorch.org/whl/cu128

### 03. Research implementation

In [ ]:
# STEP 03 — Research implementation
#cell 3
%pip install timm==1.0.22

### 04. Research implementation

In [ ]:
# STEP 04 — Research implementation
#cell 4
import torch
import torchvision
import timm

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("timm:", timm.__version__)

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=3
).to("cuda")

print("ViT initialized successfully")
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

del model
torch.cuda.empty_cache()

### 05. Research implementation

In [ ]:
# STEP 05 — Research implementation
#cell 5
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\NetraScanWorkspace\MY PROJECT E PAARVAI")
METADATA_DIR = ROOT / "NetraScan_Metadata"

train_df = pd.read_csv(METADATA_DIR / "netrascan_train.csv")
val_df = pd.read_csv(METADATA_DIR / "netrascan_val.csv")
test_df = pd.read_csv(METADATA_DIR / "netrascan_test.csv")

print("TRAIN:", train_df.shape)
print("VALIDATION:", val_df.shape)
print("TEST:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nClass distribution:")
for name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\n{name}")
    print(df["class"].value_counts())

print("\nChecking image paths:")
for name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    missing = sum(not Path(p).is_file() for p in df["image_path"])
    print(f"{name}: {missing} missing images")

print("\nChecking visual-family overlap:")
train_families = set(train_df["visual_family_id"])
val_families = set(val_df["visual_family_id"])
test_families = set(test_df["visual_family_id"])

print("Train/Val:", len(train_families & val_families))
print("Train/Test:", len(train_families & test_families))
print("Val/Test:", len(val_families & test_families))

### 06. #Dataset class and preprocessing

In [ ]:
# STEP 06 — #Dataset class and preprocessing
#Dataset class and preprocessing
#cell 6


import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from timm.data import resolve_model_data_config

CLASS_NAMES = ["normal", "immature", "mature"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}

MODEL_NAME = "vit_base_patch16_224"

reference_model = timm.create_model(
    MODEL_NAME,
    pretrained=False,
    num_classes=3
)

data_config = resolve_model_data_config(reference_model)
del reference_model

image_size = data_config["input_size"][-1]
mean = data_config["mean"]
std = data_config["std"]

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomRotation(8),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(
        brightness=0.08,
        contrast=0.08,
        saturation=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

eval_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

class CataractDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image = Image.open(row["image_path"]).convert("RGB")
        image = self.transform(image)
        label = CLASS_TO_IDX[row["class"]]
        return image, label

train_dataset = CataractDataset(train_df, train_transform)
val_dataset = CataractDataset(val_df, eval_transform)
test_dataset = CataractDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Classes:", CLASS_TO_IDX)
print("Image size:", image_size)
print("Normalization mean:", mean)
print("Normalization std:", std)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

### 07. Display sample images

In [ ]:
# STEP 07 — Display sample images
#cell 7 Display sample images

import matplotlib.pyplot as plt
import numpy as np

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label indices:", labels.tolist())

fig, axes = plt.subplots(2, 4, figsize=(13, 7))

mean_array = np.array(mean).reshape(3, 1, 1)
std_array = np.array(std).reshape(3, 1, 1)

for i, ax in enumerate(axes.flat):
    image = images[i].permute(1, 2, 0).numpy()
    image = image * std_array.reshape(1, 1, 3) + mean_array.reshape(1, 1, 3)
    image = np.clip(image, 0, 1)

    ax.imshow(image)
    ax.set_title(IDX_TO_CLASS[labels[i].item()])
    ax.axis("off")

plt.suptitle("NetraScan.AI — Training Batch Inspection", fontsize=15)
plt.tight_layout()
plt.show()

### 08. Research implementation

In [ ]:
# STEP 08 — Research implementation
#CELL 8
import torch
import timm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

MODEL_NAME = "vit_base_patch16_224"

model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=3
)

model = model.to(DEVICE)

print("Device:", DEVICE)
print("Model:", MODEL_NAME)
print("Output classes:", model.num_classes)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Verify the forward pass using a small batch
model.eval()

with torch.inference_mode():
    sample_batch = images[:2].to(DEVICE)
    outputs = model(sample_batch)

print("Input shape:", sample_batch.shape)
print("Output shape:", outputs.shape)

assert outputs.shape == (2, 3)

print("Pretrained ViT-B/16 loaded successfully!")

### 09. Training configuration

In [ ]:
# STEP 09 — Training configuration
#CELL 9 Training configuration

import random
import numpy as np
import torch.nn as nn
from torch.optim import AdamW
from torch.amp import GradScaler

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 3

criterion = nn.CrossEntropyLoss()

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scaler = GradScaler("cuda")

CHECKPOINT_DIR = ROOT / "models" / "vit"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = ROOT / "results" / "vit"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Training configuration ready")
print("Device:", DEVICE)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", train_loader.batch_size)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Results directory:", RESULTS_DIR)

### 10. single step training gpu benchmark test

In [ ]:
# STEP 10 — single step training gpu benchmark test
#cell 10 single step training gpu benchmark test 

import time
from torch.amp import autocast

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model.train()

batch_images, batch_labels = next(iter(train_loader))
batch_images = batch_images.to(DEVICE, non_blocking=True)
batch_labels = batch_labels.to(DEVICE, non_blocking=True)

print("Batch shape:", batch_images.shape)
print("Starting GPU memory benchmark...")

try:
    torch.cuda.synchronize()
    start_time = time.perf_counter()

    optimizer.zero_grad(set_to_none=True)

    with autocast(device_type="cuda", dtype=torch.float16):
        outputs = model(batch_images)
        loss = criterion(outputs, batch_labels)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time

    peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)
    reserved_memory = torch.cuda.max_memory_reserved() / (1024 ** 3)

    print("\nBENCHMARK SUCCESSFUL")
    print(f"Loss: {loss.item():.4f}")
    print(f"Step time: {elapsed:.2f} seconds")
    print(f"Peak allocated VRAM: {peak_memory:.2f} GB")
    print(f"Peak reserved VRAM: {reserved_memory:.2f} GB")
    print(f"GPU total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024 ** 3):.2f} GB")

except torch.cuda.OutOfMemoryError:
    print("\nCUDA OUT OF MEMORY")
    print("Batch size 16 is too large for this configuration.")
    print("We will reduce the batch size to 8.")

finally:
    optimizer.zero_grad(set_to_none=True)

    del batch_images, batch_labels

    if "outputs" in globals():
        del outputs

    torch.cuda.empty_cache()

### 11. Training and validation functions

In [ ]:
# STEP 11 — Training and validation functions
#cell 11 Training and validation functions

import time
import torch
import numpy as np

from torch.amp import autocast
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm


def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    progress = tqdm(loader, desc="Training", leave=False)

    for batch_images, batch_labels in progress:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(batch_images)
            loss = criterion(outputs, batch_labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * batch_images.size(0)

        predictions = outputs.detach().argmax(dim=1)

        all_preds.extend(predictions.cpu().tolist())
        all_labels.extend(batch_labels.cpu().tolist())

        progress.set_postfix(loss=f"{loss.item():.4f}")

    epoch_loss = total_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(
        all_labels, all_preds, average="macro", zero_division=0
    )

    return epoch_loss, epoch_acc, epoch_f1


@torch.inference_mode()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    progress = tqdm(loader, desc="Validation", leave=False)

    for batch_images, batch_labels in progress:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)

        with autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(batch_images)
            loss = criterion(outputs, batch_labels)

        total_loss += loss.item() * batch_images.size(0)

        predictions = outputs.argmax(dim=1)

        all_preds.extend(predictions.cpu().tolist())
        all_labels.extend(batch_labels.cpu().tolist())

    epoch_loss = total_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(
        all_labels, all_preds, average="macro", zero_division=0
    )

    return epoch_loss, epoch_acc, epoch_f1


print("Training and validation functions initialized.")

### 12. Full ViT training

In [ ]:
# STEP 12 — Full ViT training
#cell 12 Full ViT training

import json
import random
import time
import pandas as pd
import numpy as np
import torch
import timm

from torch.optim import AdamW
from torch.amp import GradScaler

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Reset the model after our benchmark
del model
torch.cuda.empty_cache()

model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=3
).to(DEVICE)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scaler = GradScaler("cuda")

# Save experiment configuration
config = {
    "model": MODEL_NAME,
    "pretrained": True,
    "classes": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,
    "image_size": image_size,
    "normalization_mean": list(mean),
    "normalization_std": list(std),
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "test_samples": len(test_dataset),
    "batch_size": train_loader.batch_size,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "max_epochs": EPOCHS,
    "early_stopping_patience": PATIENCE,
    "seed": SEED,
    "optimizer": "AdamW",
    "loss": "CrossEntropyLoss",
    "mixed_precision": "FP16",
    "selection_metric": "validation_macro_f1"
}

with open(RESULTS_DIR / "experiment_config.json", "w") as f:
    json.dump(config, f, indent=4)

best_f1 = -1.0
patience_counter = 0
history = []

print("Starting ViT-B/16 fine-tuning")
print("Device:", DEVICE)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("-" * 65)

for epoch in range(EPOCHS):
    epoch_start = time.perf_counter()

    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, DEVICE
    )

    val_loss, val_acc, val_f1 = validate_one_epoch(
        model, val_loader, criterion, DEVICE
    )

    elapsed = time.perf_counter() - epoch_start

    metrics = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_macro_f1": train_f1,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1,
        "epoch_seconds": elapsed
    }

    history.append(metrics)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Train F1: {train_f1:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    # Save history after every epoch
    pd.DataFrame(history).to_csv(
        RESULTS_DIR / "training_history.csv",
        index=False
    )

    if val_f1 > best_f1 + 1e-4:
        best_f1 = val_f1
        patience_counter = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_macro_f1": val_f1,
                "class_to_idx": CLASS_TO_IDX,
                "model_name": MODEL_NAME
            },
            CHECKPOINT_DIR / "vit_best.pth"
        )

        print(f"  Best checkpoint saved — Val Macro-F1: {best_f1:.4f}")

    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{PATIENCE})")

    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

print("-" * 65)
print("Training completed.")
print(f"Best validation macro-F1: {best_f1:.4f}")
print("Checkpoint:", CHECKPOINT_DIR / "vit_best.pth")
print("History:", RESULTS_DIR / "training_history.csv")

## 2. Dataset integrity and V4 partitioning
Perceptual hashing and filename lineage identify candidate related images. Union-Find groups related samples before stratified family-level splitting. Intermediate V2/V3 grouping steps are retained because V4 depends on them. These checks address detected image-level relationships, not verified patient identities.

### 13. Audit cross-split near-duplicates

In [ ]:
# STEP 13 — Audit cross-split near-duplicates
# cell 13 Audit cross-split near-duplicates

from collections import Counter
import numpy as np
import pandas as pd

all_splits = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

hashes = np.array(
    [int(h, 16) for h in all_splits["phash"]],
    dtype=np.uint64
)

split_labels = all_splits["split"].to_numpy()

cross_split_pairs = []

for i in range(len(hashes)):
    for j in range(i + 1, len(hashes)):
        if split_labels[i] == split_labels[j]:
            continue

        distance = (
            int(hashes[i]) ^ int(hashes[j])
        ).bit_count()

        if distance <= 4:
            cross_split_pairs.append({
                "index_1": i,
                "index_2": j,
                "split_1": split_labels[i],
                "split_2": split_labels[j],
                "class_1": all_splits.iloc[i]["class"],
                "class_2": all_splits.iloc[j]["class"],
                "distance": distance
            })

audit_df = pd.DataFrame(cross_split_pairs)

print("Cross-split near-duplicate candidates:", len(audit_df))

if not audit_df.empty:
    print("\nDistance distribution:")
    print(audit_df["distance"].value_counts().sort_index())

    print("\nClass agreement:")
    print(
        (audit_df["class_1"] == audit_df["class_2"])
        .value_counts()
    )

    display(audit_df.head(10))

### 14. Cross-split visual inspection

In [ ]:
# STEP 14 — Cross-split visual inspection
#cell 14 Cross-split visual inspection

import matplotlib.pyplot as plt
from PIL import Image

# Prioritize the closest perceptual-hash matches
inspection_df = (
    audit_df
    .sort_values("distance")
    .drop_duplicates(subset=["index_1"])
    .head(12)
)

fig, axes = plt.subplots(
    len(inspection_df), 2,
    figsize=(9, len(inspection_df) * 3)
)

for row_num, (_, pair) in enumerate(inspection_df.iterrows()):
    idx1 = int(pair["index_1"])
    idx2 = int(pair["index_2"])

    record1 = all_splits.iloc[idx1]
    record2 = all_splits.iloc[idx2]

    for col, record in enumerate([record1, record2]):
        image = Image.open(record["image_path"]).convert("RGB")

        axes[row_num, col].imshow(image)
        axes[row_num, col].axis("off")

        axes[row_num, col].set_title(
            f"{record['split']} | {record['class']}\n"
            f"{record['source']} | {record['filename'][:30]}",
            fontsize=8
        )

    axes[row_num, 0].set_ylabel(
        f"pHash distance: {int(pair['distance'])}",
        fontsize=9
    )

plt.suptitle(
    "NetraScan.AI — Cross-Split Near-Duplicate Inspection",
    fontsize=14
)

plt.tight_layout()
plt.show()

### 15. Filename lineage audit

In [ ]:
# STEP 15 — Filename lineage audit
#cell 15 Filename lineage audit

import re
import pandas as pd

lineage_df = all_splits.copy()

def extract_original_stem(row):
    filename = str(row["filename"])
    source = str(row["source"])

    stem = filename.rsplit(".", 1)[0]

    if source == "cataract_v01":
        stem = re.sub(
            r"\.rf\.[a-fA-F0-9]+$",
            "",
            stem
        )

    return stem.lower()

lineage_df["original_stem"] = lineage_df.apply(
    extract_original_stem,
    axis=1
)

# Include class and source to avoid unrelated filename collisions
lineage_df["lineage_key"] = (
    lineage_df["source"].astype(str)
    + "::"
    + lineage_df["class"].astype(str)
    + "::"
    + lineage_df["original_stem"]
)

family_summary = (
    lineage_df.groupby("lineage_key")
    .agg(
        images=("filename", "size"),
        splits=("split", "nunique"),
        sources=("source", "nunique")
    )
    .reset_index()
)

cross_split_lineages = family_summary[
    family_summary["splits"] > 1
]

print("Total filename lineage groups:", len(family_summary))
print("Cross-split filename lineage groups:", len(cross_split_lineages))

print("\nImages in cross-split lineages:")
print(
    lineage_df[
        lineage_df["lineage_key"].isin(
            cross_split_lineages["lineage_key"]
        )
    ].shape[0]
)

display(
    cross_split_lineages
    .sort_values("images", ascending=False)
    .head(20)
)

### 16. Union-Find grouping with filename lineage

In [ ]:
# STEP 16 — Union-Find grouping with filename lineage
#cell 16 Union-Find grouping with filename lineage

from collections import defaultdict, Counter
import numpy as np
import pandas as pd

# Start from our existing combined dataset
corrected_df = lineage_df.copy().reset_index(drop=True)

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        root_a = self.find(a)
        root_b = self.find(b)

        if root_a == root_b:
            return

        if self.rank[root_a] < self.rank[root_b]:
            root_a, root_b = root_b, root_a

        self.parent[root_b] = root_a

        if self.rank[root_a] == self.rank[root_b]:
            self.rank[root_a] += 1


uf = UnionFind(len(corrected_df))

# 1. Preserve original pHash=0 visual families
for _, group in corrected_df.groupby("visual_family_id"):
    indices = group.index.tolist()
    for idx in indices[1:]:
        uf.union(indices[0], idx)

# 2. Merge Roboflow augmentation variants sharing original filename
roboflow_groups = corrected_df[
    corrected_df["source"] == "cataract_v01"
].groupby("lineage_key")

for _, group in roboflow_groups:
    indices = group.index.tolist()
    for idx in indices[1:]:
        uf.union(indices[0], idx)

# 3. Merge pHash distance <= 2 within matching classes
hashes = [
    int(h, 16) for h in corrected_df["phash"]
]
labels = corrected_df["class"].tolist()

merged_pairs = 0

for i in range(len(corrected_df)):
    for j in range(i + 1, len(corrected_df)):
        if labels[i] != labels[j]:
            continue

        distance = (hashes[i] ^ hashes[j]).bit_count()

        if distance <= 2:
            uf.union(i, j)
            merged_pairs += 1

# Assign new visual-family IDs
roots = [uf.find(i) for i in range(len(corrected_df))]
unique_roots = sorted(set(roots))

root_to_id = {
    root: f"VF2_{idx + 1:05d}"
    for idx, root in enumerate(unique_roots)
}

corrected_df["visual_family_id_v2"] = [
    root_to_id[root] for root in roots
]

family_sizes = corrected_df[
    "visual_family_id_v2"
].value_counts()

corrected_df["visual_family_size_v2"] = (
    corrected_df["visual_family_id_v2"].map(family_sizes)
)

print("Total images:", len(corrected_df))
print("Original visual families:", corrected_df["visual_family_id"].nunique())
print("Corrected visual families:", corrected_df["visual_family_id_v2"].nunique())
print("pHash <= 2 candidate pairs:", merged_pairs)
print("Largest corrected family:", family_sizes.max())

conflicts = (
    corrected_df.groupby("visual_family_id_v2")["class"]
    .nunique()
)

print("Families with conflicting labels:", (conflicts > 1).sum())

# Check how many corrected families span original splits
cross_split_families = (
    corrected_df.groupby("visual_family_id_v2")["split"]
    .nunique()
)

print(
    "Corrected families spanning original splits:",
    (cross_split_families > 1).sum()
)

### 17. Corrected group-aware splitting

In [ ]:
# STEP 17 — Corrected group-aware splitting
#cell 17 Corrected group-aware splitting


from sklearn.model_selection import train_test_split
import pandas as pd

# One representative record per corrected visual family
family_df = (
    corrected_df.groupby("visual_family_id_v2")
    .agg(
        class_label=("class", "first"),
        family_size=("visual_family_size_v2", "first")
    )
    .reset_index()
)

# 70% train, 30% temporary
train_families, temp_families = train_test_split(
    family_df,
    test_size=0.30,
    stratify=family_df["class_label"],
    random_state=42
)

# Divide remaining 30% into 15% validation and 15% test
val_families, test_families = train_test_split(
    temp_families,
    test_size=0.50,
    stratify=temp_families["class_label"],
    random_state=42
)

# Map family IDs to their assigned splits
split_mapping = {}

for family_id in train_families["visual_family_id_v2"]:
    split_mapping[family_id] = "train"

for family_id in val_families["visual_family_id_v2"]:
    split_mapping[family_id] = "val"

for family_id in test_families["visual_family_id_v2"]:
    split_mapping[family_id] = "test"

corrected_df["split_v2"] = (
    corrected_df["visual_family_id_v2"].map(split_mapping)
)

print("Corrected split distribution:")
print(corrected_df["split_v2"].value_counts())

print("\nClass distribution:")
print(
    pd.crosstab(
        corrected_df["split_v2"],
        corrected_df["class"]
    )
)

# Verify corrected visual families never cross splits
leakage_check = (
    corrected_df.groupby("visual_family_id_v2")["split_v2"]
    .nunique()
)

print("\nCorrected families crossing splits:",
      (leakage_check > 1).sum())

assert (leakage_check > 1).sum() == 0

print("\nPASS — No corrected visual-family overlap across splits.")

### 18. V2 split-integrity audit

In [ ]:
# STEP 18 — V2 split-integrity audit
#cell 18 V2 split-integrity audit


from collections import Counter
import pandas as pd

# Work only with the corrected in-memory dataset
v2_audit = corrected_df.reset_index(drop=True).copy()

hashes_v2 = [
    int(h, 16) for h in v2_audit["phash"]
]

remaining_pairs = []

for i in range(len(v2_audit)):
    for j in range(i + 1, len(v2_audit)):
        if v2_audit.at[i, "split_v2"] == v2_audit.at[j, "split_v2"]:
            continue

        distance = (
            hashes_v2[i] ^ hashes_v2[j]
        ).bit_count()

        if distance <= 4:
            remaining_pairs.append({
                "index_1": i,
                "index_2": j,
                "distance": distance,
                "split_1": v2_audit.at[i, "split_v2"],
                "split_2": v2_audit.at[j, "split_v2"],
                "class_1": v2_audit.at[i, "class"],
                "class_2": v2_audit.at[j, "class"],
                "source_1": v2_audit.at[i, "source"],
                "source_2": v2_audit.at[j, "source"]
            })

remaining_df = pd.DataFrame(remaining_pairs)

print("Remaining cross-split pHash candidates:", len(remaining_df))

if not remaining_df.empty:
    print("\nDistance distribution:")
    print(remaining_df["distance"].value_counts().sort_index())

    print("\nClass agreement:")
    print(
        (remaining_df["class_1"] == remaining_df["class_2"])
        .value_counts()
    )

    print("\nSource combinations:")
    print(
        pd.crosstab(
            remaining_df["source_1"],
            remaining_df["source_2"]
        )
    )

    display(remaining_df.head(15))

# Check Roboflow augmentation lineage
roboflow_v2 = v2_audit[
    v2_audit["source"] == "cataract_v01"
]

lineage_overlap = (
    roboflow_v2.groupby("lineage_key")["split_v2"]
    .nunique()
)

print(
    "\nRoboflow lineages crossing V2 splits:",
    int((lineage_overlap > 1).sum())
)

print(
    "V2 families crossing splits:",
    int(
        (
            v2_audit.groupby("visual_family_id_v2")["split_v2"]
            .nunique() > 1
        ).sum()
    )
)

### 19. Inspect remaining similarity candidates

In [ ]:
# STEP 19 — Inspect remaining similarity candidates
import matplotlib.pyplot as plt
from PIL import Image

# Select representative pairs from each dataset
inspection = []

for source in ["cataract_v01", "dataset_katarak"]:
    subset = remaining_df[
        (remaining_df["source_1"] == source) &
        (remaining_df["source_2"] == source)
    ].copy()

    subset["family_1"] = subset["index_1"].map(
        v2_audit["visual_family_id_v2"]
    )
    subset["family_2"] = subset["index_2"].map(
        v2_audit["visual_family_id_v2"]
    )

    subset["family_pair"] = subset.apply(
        lambda row: tuple(sorted([
            row["family_1"],
            row["family_2"]
        ])),
        axis=1
    )

    subset = subset.drop_duplicates("family_pair")
    inspection.append(subset.head(8))

inspection_df = pd.concat(
    inspection, ignore_index=True
)

print("Representative family pairs:", len(inspection_df))

if len(inspection_df) > 0:
    fig, axes = plt.subplots(
        len(inspection_df), 2,
        figsize=(10, len(inspection_df) * 2.8),
        squeeze=False
    )

    for row_num, (_, pair) in enumerate(inspection_df.iterrows()):
        idx1 = int(pair["index_1"])
        idx2 = int(pair["index_2"])

        for col, idx in enumerate([idx1, idx2]):
            record = v2_audit.iloc[idx]

            with Image.open(record["image_path"]) as img:
                image = img.convert("RGB")

            axes[row_num, col].imshow(image)
            axes[row_num, col].axis("off")
            axes[row_num, col].set_title(
                f"{record['source']} | {record['split_v2']}\n"
                f"{record['class']} | {record['filename'][:35]}",
                fontsize=8
            )

        axes[row_num, 0].set_ylabel(
            f"Pair {row_num + 1}\npHash = {pair['distance']}",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()

### 20. # Build V3 groups from the existing V2 groups

In [ ]:
# STEP 20 — # Build V3 groups from the existing V2 groups
# Build V3 groups from the existing V2 groups
uf_v3 = UnionFind(len(v2_audit))

# Preserve every V2 visual family
for _, group in v2_audit.groupby("visual_family_id_v2"):
    indices = group.index.tolist()

    for idx in indices[1:]:
        uf_v3.union(indices[0], idx)

# Merge all same-class pHash distance-4 candidates
merged_candidates = 0

for _, pair in remaining_df.iterrows():
    i = int(pair["index_1"])
    j = int(pair["index_2"])

    if (
        pair["distance"] == 4
        and pair["class_1"] == pair["class_2"]
    ):
        uf_v3.union(i, j)
        merged_candidates += 1

roots_v3 = [
    uf_v3.find(i)
    for i in range(len(v2_audit))
]

unique_roots_v3 = sorted(set(roots_v3))

root_to_v3 = {
    root: f"VF3_{n + 1:05d}"
    for n, root in enumerate(unique_roots_v3)
}

v3_df = v2_audit.copy()

v3_df["visual_family_id_v3"] = [
    root_to_v3[root]
    for root in roots_v3
]

v3_sizes = v3_df["visual_family_id_v3"].value_counts()

v3_df["visual_family_size_v3"] = (
    v3_df["visual_family_id_v3"].map(v3_sizes)
)

print("Total images:", len(v3_df))
print("V2 families:", v2_audit["visual_family_id_v2"].nunique())
print("V3 families:", v3_df["visual_family_id_v3"].nunique())
print("Candidate pairs grouped:", merged_candidates)
print("Largest V3 family:", v3_sizes.max())

conflicts = (
    v3_df.groupby("visual_family_id_v3")["class"]
    .nunique()
)

print("Conflicting-label families:", (conflicts > 1).sum())

assert (conflicts > 1).sum() == 0

print("\nPASS — V3 grouping constructed successfully.")

### 21. V3 stratified splitting

In [ ]:
# STEP 21 — V3 stratified splitting
#Cell 20 — V3 stratified splitting
from sklearn.model_selection import train_test_split
import pandas as pd

RANDOM_SEED = 42

# One record per V3 visual family
v3_family_df = (
    v3_df.groupby("visual_family_id_v3")
    .agg(
        class_label=("class", "first"),
        family_size=("visual_family_size_v3", "first")
    )
    .reset_index()
)

# Split families: 70% train, 30% temporary
v3_train_families, v3_temp_families = train_test_split(
    v3_family_df,
    test_size=0.30,
    stratify=v3_family_df["class_label"],
    random_state=RANDOM_SEED
)

# Split temporary families equally into validation and test
v3_val_families, v3_test_families = train_test_split(
    v3_temp_families,
    test_size=0.50,
    stratify=v3_temp_families["class_label"],
    random_state=RANDOM_SEED
)

# Build family-to-split mapping
v3_split_map = {}

for family_id in v3_train_families["visual_family_id_v3"]:
    v3_split_map[family_id] = "train"

for family_id in v3_val_families["visual_family_id_v3"]:
    v3_split_map[family_id] = "val"

for family_id in v3_test_families["visual_family_id_v3"]:
    v3_split_map[family_id] = "test"

# Assign splits to all images
v3_df["split_v3"] = (
    v3_df["visual_family_id_v3"].map(v3_split_map)
)

# Display distributions
print("V3 dataset distribution:")
print(v3_df["split_v3"].value_counts())

print("\nV3 class distribution:")
print(
    pd.crosstab(
        v3_df["split_v3"],
        v3_df["class"]
    )
)

# Verify group isolation
v3_overlap = (
    v3_df.groupby("visual_family_id_v3")["split_v3"]
    .nunique()
)

print(
    "\nV3 families crossing splits:",
    int((v3_overlap > 1).sum())
)

assert (v3_overlap > 1).sum() == 0

print("\nPASS — V3 visual-family split isolation verified.")

### 22. Final V3 integrity audit

In [ ]:
# STEP 22 — Final V3 integrity audit
#Cell 21 — Final V3 integrity audit

from pathlib import Path
from collections import Counter
import pandas as pd

audit = v3_df.reset_index(drop=True).copy()

hash_values = [int(h, 16) for h in audit["phash"]]
labels = audit["class"].tolist()
splits = audit["split_v3"].tolist()

cross_split_pairs = []
all_candidate_pairs = []

for i in range(len(audit)):
    for j in range(i + 1, len(audit)):

        if labels[i] != labels[j]:
            continue

        distance = (hash_values[i] ^ hash_values[j]).bit_count()

        if distance <= 4:
            pair = {
                "index_1": i,
                "index_2": j,
                "distance": distance,
                "split_1": splits[i],
                "split_2": splits[j]
            }

            all_candidate_pairs.append(pair)

            if splits[i] != splits[j]:
                cross_split_pairs.append(pair)

all_candidates_df = pd.DataFrame(all_candidate_pairs)
cross_split_v3_df = pd.DataFrame(cross_split_pairs)

print("Total same-class pHash <=4 pairs:", len(all_candidates_df))
print("Cross-split pHash <=4 pairs:", len(cross_split_v3_df))

if not cross_split_v3_df.empty:
    print("\nCross-split distance distribution:")
    print(
        cross_split_v3_df["distance"]
        .value_counts()
        .sort_index()
    )

# Roboflow filename lineage
roboflow = audit[audit["source"] == "cataract_v01"]

lineage_overlap = (
    roboflow.groupby("lineage_key")["split_v3"].nunique()
)

print(
    "\nRoboflow lineages crossing splits:",
    int((lineage_overlap > 1).sum())
)

# V3 family overlap
family_overlap = (
    audit.groupby("visual_family_id_v3")["split_v3"].nunique()
)

print(
    "V3 families crossing splits:",
    int((family_overlap > 1).sum())
)

# Missing images
missing_paths = sum(
    not Path(path).is_file()
    for path in audit["image_path"]
)

print("Missing image paths:", missing_paths)

print("\nAudit complete.")

### 23. Global Union-Find grouping

In [ ]:
# STEP 23 — Global Union-Find grouping
#Cell 22 — Global Union-Find grouping

# GLOBAL VISUAL-FAMILY GROUPING
# No files will be modified or overwritten.

global_df = v3_df.reset_index(drop=True).copy()

uf_global = UnionFind(len(global_df))

# Preserve existing V3 families
for _, group in global_df.groupby("visual_family_id_v3"):
    indices = group.index.tolist()

    for idx in indices[1:]:
        uf_global.union(indices[0], idx)

# Merge all same-class pHash <=4 candidate relationships
global_pairs = 0

for _, pair in all_candidates_df.iterrows():
    i = int(pair["index_1"])
    j = int(pair["index_2"])

    # Verify indices and labels
    assert global_df.at[i, "class"] == global_df.at[j, "class"]

    uf_global.union(i, j)
    global_pairs += 1

# Assign global family IDs
roots = [uf_global.find(i) for i in range(len(global_df))]

root_to_global_id = {
    root: f"VFG_{n + 1:05d}"
    for n, root in enumerate(sorted(set(roots)))
}

global_df["visual_family_id_v4"] = [
    root_to_global_id[root]
    for root in roots
]

family_sizes = global_df["visual_family_id_v4"].value_counts()

global_df["visual_family_size_v4"] = (
    global_df["visual_family_id_v4"].map(family_sizes)
)

# Validate class consistency
class_conflicts = (
    global_df.groupby("visual_family_id_v4")["class"]
    .nunique()
)

print("Total images:", len(global_df))
print("V3 families:", global_df["visual_family_id_v3"].nunique())
print("V4 families:", global_df["visual_family_id_v4"].nunique())
print("Candidate pairs processed:", global_pairs)
print("Largest V4 family:", family_sizes.max())
print("Families with conflicting labels:", int((class_conflicts > 1).sum()))

print("\nV4 family size distribution:")
print(family_sizes.value_counts().sort_index().tail(15))

assert (class_conflicts > 1).sum() == 0

print("\nPASS — Global candidate-family grouping complete.")

### 24. V4 stratified splitting and verification

In [ ]:
# STEP 24 — V4 stratified splitting and verification
#Cell 23 — V4 stratified splitting and verification

from sklearn.model_selection import train_test_split
from pathlib import Path
import pandas as pd

RANDOM_SEED = 42

# One representative row per V4 family
v4_family_df = (
    global_df.groupby("visual_family_id_v4")
    .agg(
        class_label=("class", "first"),
        family_size=("visual_family_size_v4", "first")
    )
    .reset_index()
)

# 70% train / 30% temporary
train_groups, temp_groups = train_test_split(
    v4_family_df,
    test_size=0.30,
    stratify=v4_family_df["class_label"],
    random_state=RANDOM_SEED
)

# 15% validation / 15% test
val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.50,
    stratify=temp_groups["class_label"],
    random_state=RANDOM_SEED
)

split_map = {}

for group_id in train_groups["visual_family_id_v4"]:
    split_map[group_id] = "train"

for group_id in val_groups["visual_family_id_v4"]:
    split_map[group_id] = "val"

for group_id in test_groups["visual_family_id_v4"]:
    split_map[group_id] = "test"

v4_df = global_df.copy()

v4_df["split_v4"] = (
    v4_df["visual_family_id_v4"].map(split_map)
)

print("V4 split distribution:")
print(v4_df["split_v4"].value_counts())

print("\nV4 class distribution:")
print(pd.crosstab(v4_df["split_v4"], v4_df["class"]))

# 1. Visual-family overlap
family_overlap = (
    v4_df.groupby("visual_family_id_v4")["split_v4"]
    .nunique()
)

# 2. Cross-split pHash <=4 candidates
# all_candidates_df indices correspond to global_df/v4_df
cross_split_candidates = sum(
    v4_df.iloc[int(row["index_1"])]["split_v4"]
    != v4_df.iloc[int(row["index_2"])]["split_v4"]
    for _, row in all_candidates_df.iterrows()
)

# 3. Roboflow augmentation lineage
roboflow_v4 = v4_df[
    v4_df["source"] == "cataract_v01"
]

lineage_overlap = (
    roboflow_v4.groupby("lineage_key")["split_v4"]
    .nunique()
)

# 4. Missing image paths
missing_images = sum(
    not Path(path).is_file()
    for path in v4_df["image_path"]
)

print("\nINTEGRITY CHECKS")
print("-" * 45)

print("Families crossing splits:",
      int((family_overlap > 1).sum()))

print("pHash <=4 pairs crossing splits:",
      cross_split_candidates)

print("Roboflow lineages crossing splits:",
      int((lineage_overlap > 1).sum()))

print("Missing image paths:", missing_images)

assert (family_overlap > 1).sum() == 0
assert cross_split_candidates == 0
assert (lineage_overlap > 1).sum() == 0
assert missing_images == 0

print("\nPASS — V4 split integrity checks successful.")

### 25. Export V4 manifests

In [ ]:
# STEP 25 — Export V4 manifests
from pathlib import Path
import pandas as pd
import json

V4_DIR = ROOT / "NetraScan_Metadata" / "v4"
V4_DIR.mkdir(parents=True, exist_ok=True)

# Preserve the original metadata, plus V4 grouping and split assignments.
export_columns = [
    "source",
    "original_split",
    "filename",
    "class",
    "image_path",
    "sha256",
    "phash",
    "visual_family_id_v4",
    "visual_family_size_v4",
    "split_v4"
]

master_v4 = v4_df[export_columns].copy()

# Standardize column names for reuse in the training pipeline.
master_v4 = master_v4.rename(columns={
    "visual_family_id_v4": "visual_family_id",
    "visual_family_size_v4": "visual_family_size",
    "split_v4": "split"
})

master_v4.to_csv(
    V4_DIR / "netrascan_3class_master_v4.csv",
    index=False
)

for split_name in ["train", "val", "test"]:
    subset = master_v4[
        master_v4["split"] == split_name
    ].copy()

    subset.to_csv(
        V4_DIR / f"netrascan_{split_name}_v4.csv",
        index=False
    )

# Record the methodology for reproducibility.
split_config = {
    "version": "v4",
    "seed": 42,
    "target_split_ratio": {
        "train": 0.70,
        "val": 0.15,
        "test": 0.15
    },
    "grouping": [
        "Original visual-family relationships",
        "Roboflow filename augmentation lineage",
        "Global same-class pHash Hamming distance <= 4",
        "Union-Find connected components"
    ],
    "grouping_note": (
        "Similarity-based groups for split isolation; "
        "not verified patient identities."
    ),
    "total_images": len(master_v4),
    "visual_families": master_v4["visual_family_id"].nunique(),
    "split_counts": {
        k: int(v)
        for k, v in master_v4["split"].value_counts().items()
    }
}

with open(
    V4_DIR / "split_config_v4.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(split_config, f, indent=4)

print("V4 manifests saved successfully!\n")

for file in sorted(V4_DIR.iterdir()):
    print(file.name)

print("\nTotal images:", len(master_v4))
print("Visual families:", master_v4["visual_family_id"].nunique())
print("Split counts:", master_v4["split"].value_counts().to_dict())

### Recorded V4 dataset composition
| Partition | Normal | Immature | Mature | Total |
|---|---:|---:|---:|---:|
| Training | 799 | 837 | 818 | 2454 |
| Validation | 178 | 164 | 175 | 517 |
| Test | 192 | 195 | 184 | 571 |
| Total | 1169 | 1196 | 1177 | 3542 |

Recorded integrity checks: zero cross-split visual families, pHash distance ≤4 pairs, detected Roboflow lineages and missing image paths.

## 3. ViT-B/16: final V4 training and evaluation
Fine-tune on the corrected partitions and evaluate the selected checkpoint on the held-out test set.

### 26. Load corrected V4 dataset

In [ ]:
# STEP 26 — Load corrected V4 dataset
# STEP 27 — Load corrected V4 dataset

import pandas as pd

V4_DIR = ROOT / "NetraScan_Metadata" / "v4"

train_df = pd.read_csv(V4_DIR / "netrascan_train_v4.csv")
val_df = pd.read_csv(V4_DIR / "netrascan_val_v4.csv")
test_df = pd.read_csv(V4_DIR / "netrascan_test_v4.csv")

print("V4 TRAIN:", train_df.shape)
print("V4 VALIDATION:", val_df.shape)
print("V4 TEST:", test_df.shape)

print("\nClass distribution:")
for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}")
    print(df["class"].value_counts())

print("\nMissing image paths:")
for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    missing = (~df["image_path"].map(
        lambda p: Path(p).is_file()
    )).sum()
    print(f"{name}: {missing}")

### 27. #Define the image transformations

In [ ]:
# STEP 27 — #Define the image transformations
#Define the image transformations

from torchvision import transforms

# Training: mild augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(degrees=8),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

# Validation and testing: deterministic preprocessing
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

test_transform = val_transform

print("Training transforms: READY")
print("Validation transforms: READY")
print("Testing transforms: READY")

### 28. Build V4 datasets and DataLoaders

In [ ]:
# STEP 28 — Build V4 datasets and DataLoaders
# STEP 28 — Build V4 datasets and DataLoaders

from torch.utils.data import DataLoader

BATCH_SIZE = 16
NUM_WORKERS = 0

# Reuse existing CataractDataset class and transforms
train_dataset = CataractDataset(
    train_df,
    transform=train_transform
)

val_dataset = CataractDataset(
    val_df,
    transform=val_transform
)

test_dataset = CataractDataset(
    test_df,
    transform=test_transform
)

# Recreate DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("V4 DATASET CONFIGURATION")
print("-" * 40)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

print("\nTraining batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

# Verify one training batch
images, labels = next(iter(train_loader))

print("\nBatch image shape:", images.shape)
print("Batch label shape:", labels.shape)

assert images.shape == (16, 3, 224, 224)
assert labels.shape == (16,)

print("\nPASS — V4 DataLoaders ready.")

### 29. Model and training configuration

In [ ]:
# STEP 29 — Model and training configuration
#Cell 29 — Model and training configuration

import torch
import torch.nn as nn
import timm
from pathlib import Path

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Hardware
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Model configuration
MODEL_NAME = "vit_base_patch16_224"
NUM_CLASSES = 3

# Fresh ImageNet-pretrained ViT-B/16
model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=NUM_CLASSES
)

model = model.to(DEVICE)

# Training hyperparameters
EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 3

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Mixed precision for RTX GPU
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

# Separate V4 output directories
MODEL_DIR = ROOT / "models" / "vit_v4"
RESULTS_DIR = ROOT / "results" / "vit_v4"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "vit_v4_best.pth"

print("V4 TRAINING CONFIGURATION")
print("-" * 45)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("Model:", MODEL_NAME)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Classes:", NUM_CLASSES)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Mixed precision:", scaler.is_enabled())
print("Checkpoint path:", BEST_MODEL_PATH)

print("\nPASS — Fresh ViT-B/16 initialized for V4.")

### 30. V4 ViT-B/16 training

In [ ]:
# STEP 30 — V4 ViT-B/16 training
# STEP 30 — V4 ViT-B/16 training

import json
import time
import pandas as pd
import torch

# Verify training functions are available
assert callable(train_one_epoch)
assert callable(validate_one_epoch)

# Save experiment configuration
experiment_config = {
    "experiment": "vit_v4",
    "model": MODEL_NAME,
    "dataset_version": "v4",
    "num_classes": NUM_CLASSES,
    "class_names": ["normal", "immature", "mature"],
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "patience": PATIENCE,
    "seed": SEED,
    "optimizer": "AdamW",
    "loss": "CrossEntropyLoss",
    "mixed_precision": scaler.is_enabled(),
    "selection_metric": "validation_macro_f1"
}

with open(
    RESULTS_DIR / "experiment_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(experiment_config, f, indent=4)

# Training state
history = []
best_val_f1 = -1.0
epochs_without_improvement = 0

print("Starting ViT-B/16 V4 training...")
print("Device:", DEVICE)
print("Checkpoint:", BEST_MODEL_PATH)

for epoch in range(1, EPOCHS + 1):

    start_time = time.time()

    # Training
    train_loss, train_acc, train_f1 = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        scaler,
        DEVICE
    )

    # Validation
    val_loss, val_acc, val_f1 = validate_one_epoch(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    epoch_time = time.time() - start_time

    history.append({
        "epoch": epoch,
        "train_loss": float(train_loss),
        "train_accuracy": float(train_acc),
        "train_macro_f1": float(train_f1),
        "val_loss": float(val_loss),
        "val_accuracy": float(val_acc),
        "val_macro_f1": float(val_f1),
        "epoch_time_seconds": round(epoch_time, 2)
    })

    # Save training history after every epoch
    pd.DataFrame(history).to_csv(
        RESULTS_DIR / "training_history.csv",
        index=False
    )

    print(
        f"\nEpoch {epoch}/{EPOCHS} | "
        f"Time: {epoch_time:.1f}s\n"
        f"Train: Loss={train_loss:.4f}, "
        f"Acc={train_acc:.4f}, F1={train_f1:.4f}\n"
        f"Val:   Loss={val_loss:.4f}, "
        f"Acc={val_acc:.4f}, F1={val_f1:.4f}"
    )

    # Select best checkpoint using validation macro-F1
    if val_f1 > best_val_f1 + 1e-6:

        best_val_f1 = float(val_f1)
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print("Best V4 checkpoint saved.")

    else:
        epochs_without_improvement += 1

        print(
            "No validation F1 improvement:",
            f"{epochs_without_improvement}/{PATIENCE}"
        )

    if epochs_without_improvement >= PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\nV4 TRAINING COMPLETE")
print("-" * 45)
print("Epochs completed:", len(history))
print("Best validation macro-F1:", round(best_val_f1, 6))
print("Best checkpoint:", BEST_MODEL_PATH)
print("History:", RESULTS_DIR / "training_history.csv")

### 31. Independently verify V4 validation predictions

In [ ]:
# STEP 31 — Independently verify V4 validation predictions
# STEP 31 — Independently verify V4 validation predictions

import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Load the best V4 checkpoint, not the final epoch weights
model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE,
        weights_only=True
    )
)

model.eval()

all_true = []
all_pred = []
all_confidences = []

with torch.inference_mode():
    for images, labels in val_loader:

        images = images.to(DEVICE)

        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)

        predictions = probabilities.argmax(dim=1)
        confidences = probabilities.max(dim=1).values

        all_true.extend(labels.cpu().numpy())
        all_pred.extend(predictions.cpu().numpy())
        all_confidences.extend(confidences.cpu().numpy())

all_true = np.array(all_true)
all_pred = np.array(all_pred)
all_confidences = np.array(all_confidences)

print("V4 VALIDATION VERIFICATION")
print("-" * 45)

print("Total predictions:", len(all_pred))
print("Correct predictions:", int((all_true == all_pred).sum()))
print("Incorrect predictions:", int((all_true != all_pred).sum()))

print("\nAccuracy:", accuracy_score(all_true, all_pred))
print("Macro-F1:", f1_score(all_true, all_pred, average="macro"))

print("\nConfusion matrix:")
print(confusion_matrix(all_true, all_pred, labels=[0, 1, 2]))

print("\nClassification report:")
print(
    classification_report(
        all_true,
        all_pred,
        target_names=["normal", "immature", "mature"],
        labels=[0, 1, 2],
        zero_division=0
    )
)

print("Mean confidence:", round(float(all_confidences.mean()), 6))
print("Minimum confidence:", round(float(all_confidences.min()), 6))

assert len(all_pred) == len(val_dataset)
assert len(all_true) == len(val_df)

print("\nPASS — Independent validation inference completed.")

### 32. Audit class distribution across dataset sources

In [ ]:
# STEP 32 — Audit class distribution across dataset sources
# STEP 32 — Audit class distribution across dataset sources

import pandas as pd

# Load the full V4 manifest
audit_df = pd.read_csv(
    V4_DIR / "netrascan_3class_master_v4.csv"
)

print("SOURCE VS CLASS DISTRIBUTION")
print("-" * 55)

source_class_counts = pd.crosstab(
    audit_df["source"],
    audit_df["class"]
)

print("\nImage counts:")
print(source_class_counts)

print("\nClass percentages within each source:")
print(
    pd.crosstab(
        audit_df["source"],
        audit_df["class"],
        normalize="index"
    ).mul(100).round(2)
)

print("\nSOURCE VS SPLIT")
print("-" * 55)

print(
    pd.crosstab(
        audit_df["source"],
        audit_df["split"]
    )
)

print("\nSOURCE + CLASS VS SPLIT")
print("-" * 55)

print(
    pd.crosstab(
        [audit_df["source"], audit_df["class"]],
        audit_df["split"]
    )
)

print("\nPASS — Source distribution audit completed.")

### 33. Investigate Dataset Katarak augmentation filenames

In [ ]:
# STEP 33 — Investigate Dataset Katarak augmentation filenames
# STEP 33 — Investigate Dataset Katarak augmentation filenames

import pandas as pd
import re

katarak_df = audit_df[
    audit_df["source"] == "dataset_katarak"
].copy()

# Remove the Roboflow export hash suffix
katarak_df["base_filename"] = (
    katarak_df["filename"]
    .str.replace(
        r"\.rf\.[0-9a-fA-F]+(?=\.[^.]+$)",
        "",
        regex=True
    )
    .str.lower()
)

# Extract common filename patterns
def get_filename_pattern(filename):
    pattern = re.sub(r"\d+", "<N>", filename)
    return pattern

katarak_df["filename_pattern"] = (
    katarak_df["base_filename"]
    .apply(get_filename_pattern)
)

print("DATASET KATARAK FILENAME AUDIT")
print("-" * 55)

print("Total images:", len(katarak_df))
print("Unique base filenames:", katarak_df["base_filename"].nunique())

print("\nMost common filename patterns:")
print(
    katarak_df["filename_pattern"]
    .value_counts()
    .head(15)
)

print("\nSample filenames:")
print(
    katarak_df[
        ["filename", "base_filename", "class", "split"]
    ].head(20).to_string(index=False)
)

# Check whether identical base filenames span splits
base_overlap = (
    katarak_df.groupby(
        ["class", "base_filename"]
    )["split"].nunique()
)

print(
    "\nBase filenames crossing splits:",
    int((base_overlap > 1).sum())
)

# Inspect largest V4 families from this source
print("\nLargest Katarak visual families:")
print(
    katarak_df.groupby("visual_family_id")
    .agg(
        images=("filename", "size"),
        classes=("class", "nunique"),
        splits=("split", "nunique")
    )
    .sort_values("images", ascending=False)
    .head(10)
)

print("\nPASS — Katarak filename audit completed.")

### 34. Evaluate V4 ViT-B/16 on the held-out test set

In [ ]:
# STEP 34 — Evaluate V4 ViT-B/16 on the held-out test set
# STEP 34 — Evaluate V4 ViT-B/16 on the held-out test set

import torch
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Load the best checkpoint selected using validation macro-F1
model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE,
        weights_only=True
    )
)

model.eval()

test_true = []
test_pred = []
test_confidence = []

# Run inference on the held-out test dataset
with torch.inference_mode():
    for images, labels in test_loader:

        images = images.to(DEVICE)

        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)

        predictions = probabilities.argmax(dim=1)
        confidence = probabilities.max(dim=1).values

        test_true.extend(labels.cpu().numpy())
        test_pred.extend(predictions.cpu().numpy())
        test_confidence.extend(confidence.cpu().numpy())

test_true = np.array(test_true)
test_pred = np.array(test_pred)
test_confidence = np.array(test_confidence)

CLASS_NAMES = ["normal", "immature", "mature"]

# Calculate overall metrics
metrics = {
    "model": "ViT-B/16",
    "dataset": "V4",
    "test_images": len(test_true),
    "accuracy": accuracy_score(test_true, test_pred),
    "macro_precision": precision_score(
        test_true, test_pred, average="macro", zero_division=0
    ),
    "macro_recall": recall_score(
        test_true, test_pred, average="macro", zero_division=0
    ),
    "macro_f1": f1_score(
        test_true, test_pred, average="macro", zero_division=0
    ),
    "weighted_f1": f1_score(
        test_true, test_pred, average="weighted", zero_division=0
    )
}

print("V4 ViT-B/16 TEST EVALUATION")
print("-" * 50)

for key, value in metrics.items():
    print(f"{key}: {value}")

print("\nConfusion matrix:")
cm = confusion_matrix(test_true, test_pred, labels=[0, 1, 2])
print(cm)

print("\nClassification report:")
print(
    classification_report(
        test_true,
        test_pred,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES,
        zero_division=0
    )
)

print("Mean confidence:", round(float(test_confidence.mean()), 6))
print("Minimum confidence:", round(float(test_confidence.min()), 6))

# Save metrics and predictions
pd.DataFrame([metrics]).to_csv(
    RESULTS_DIR / "test_metrics.csv",
    index=False
)

prediction_df = test_df.copy()
prediction_df["true_label"] = test_true
prediction_df["predicted_label"] = test_pred
prediction_df["confidence"] = test_confidence
prediction_df["correct"] = test_true == test_pred

prediction_df.to_csv(
    RESULTS_DIR / "test_predictions.csv",
    index=False
)

assert len(test_true) == 571

print("\nPASS — Test evaluation completed and results saved.")

### 35. Inspect ViT-B/16 V4 test misclassifications

In [ ]:
# STEP 35 — Inspect ViT-B/16 V4 test misclassifications
# STEP 35 — Inspect ViT-B/16 V4 test misclassifications

import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import math

# Load the saved test predictions
predictions_df = pd.read_csv(
    RESULTS_DIR / "test_predictions.csv"
)

errors_df = predictions_df[
    predictions_df["correct"] == False
].copy()

print("Total misclassified images:", len(errors_df))

print("\nErrors by source:")
print(errors_df["source"].value_counts())

print("\nErrors by actual class:")
print(errors_df["class"].value_counts())

# Display all misclassified images
n = len(errors_df)
cols = 5
rows = math.ceil(n / cols)

if n > 0:
    fig, axes = plt.subplots(
        rows, cols,
        figsize=(16, rows * 3.5)
    )

    axes = axes.flatten()

    for ax, (_, row) in zip(axes, errors_df.iterrows()):
        image = Image.open(row["image_path"]).convert("RGB")

        ax.imshow(image)
        ax.set_title(
            f"Actual: {CLASS_NAMES[int(row['true_label'])]}\n"
            f"Pred: {CLASS_NAMES[int(row['predicted_label'])]}\n"
            f"Confidence: {row['confidence']:.3f}",
            fontsize=9
        )
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    plt.suptitle(
        "ViT-B/16 — V4 Test Misclassifications",
        fontsize=15
    )

    plt.tight_layout()
    plt.show()
else:
    print("No misclassifications found.")

print("\nPASS — Misclassification inspection complete.")

### 36. Analyze ViT-B/16 errors by V4 visual family

In [ ]:
# STEP 36 — Analyze ViT-B/16 errors by V4 visual family
# STEP 36 — Analyze ViT-B/16 errors by V4 visual family

import pandas as pd

errors_df = predictions_df[
    predictions_df["correct"] == False
].copy()

print("V4 MISCLASSIFICATION FAMILY ANALYSIS")
print("-" * 55)

print("Total misclassified images:", len(errors_df))

print(
    "Distinct misclassified visual families:",
    errors_df["visual_family_id"].nunique()
)

print("\nErrors grouped by visual family:")

family_errors = (
    errors_df.groupby(
        ["visual_family_id", "source", "class"]
    )
    .agg(
        error_count=("filename", "size"),
        mean_confidence=("confidence", "mean"),
        min_confidence=("confidence", "min")
    )
    .sort_values("error_count", ascending=False)
)

print(family_errors.to_string())

print("\nSource-specific test performance:")

source_performance = (
    predictions_df.groupby("source")
    .agg(
        total_images=("correct", "size"),
        correct_predictions=("correct", "sum")
    )
)

source_performance["errors"] = (
    source_performance["total_images"]
    - source_performance["correct_predictions"]
)

source_performance["accuracy"] = (
    source_performance["correct_predictions"]
    / source_performance["total_images"]
)

print(source_performance.to_string())

print("\nPASS — Error-family analysis completed.")

### 37. Generate and save ViT-B/16 V4 performance figures

In [ ]:
# STEP 37 — Generate and save ViT-B/16 V4 performance figures
# STEP 37 — Generate and save ViT-B/16 V4 performance figures

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

history_df = pd.read_csv(
    RESULTS_DIR / "training_history.csv"
)

# Figure 1 — Training and validation accuracy
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    history_df["epoch"],
    history_df["train_accuracy"],
    marker="o",
    label="Training accuracy"
)

ax.plot(
    history_df["epoch"],
    history_df["val_accuracy"],
    marker="s",
    label="Validation accuracy"
)

ax.set_title("ViT-B/16 — V4 Accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.05)
ax.set_xticks(history_df["epoch"])
ax.legend()
ax.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(
    RESULTS_DIR / "vit_v4_accuracy_curve.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# Figure 2 — Held-out test confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Normal", "Immature", "Mature"]
)

display.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

ax.set_title("ViT-B/16 — V4 Test Confusion Matrix")

fig.tight_layout()
fig.savefig(
    RESULTS_DIR / "vit_v4_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("PASS — ViT-B/16 figures saved.")
print("Location:", RESULTS_DIR.resolve())

## 4. Swin-Tiny: training and evaluation
Restore V4 data and use architecture-specific preprocessing and training configuration.

### 38. Restore V4 dataset manifests

In [ ]:
# STEP 38 — Restore V4 dataset manifests
# STEP 51A — Restore V4 dataset manifests

from pathlib import Path
import pandas as pd
import torch
import timm

ROOT = Path(r"C:\NetraScanWorkspace\MY PROJECT E PAARVAI")

V4_DIR = ROOT / "NetraScan_Metadata" / "v4"

train_df = pd.read_csv(V4_DIR / "netrascan_train_v4.csv")
val_df = pd.read_csv(V4_DIR / "netrascan_val_v4.csv")
test_df = pd.read_csv(V4_DIR / "netrascan_test_v4.csv")

CLASS_NAMES = ["normal", "immature", "mature"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("V4 DATASET RESTORED")
print("-" * 45)
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Testing images:", len(test_df))
print("Classes:", CLASS_TO_IDX)
print("Device:", DEVICE)

assert len(train_df) == 2454
assert len(val_df) == 517
assert len(test_df) == 571

print("\nPASS — V4 manifests loaded successfully.")

### 39. B — Restore V4 transforms and DataLoaders

In [ ]:
# STEP 39 — B — Restore V4 transforms and DataLoaders
# STEP 51 B — Restore V4 transforms and DataLoaders

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMAGE_SIZE = 224
BATCH_SIZE = 16

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(8),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

test_transform = val_transform

class CataractDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        label = CLASS_TO_IDX[row["class"]]

        if self.transform:
            image = self.transform(image)

        return image, label

train_dataset = CataractDataset(train_df, train_transform)
val_dataset = CataractDataset(val_df, val_transform)
test_dataset = CataractDataset(test_df, test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

images, labels = next(iter(train_loader))

print("V4 DATALOADERS RESTORED")
print("-" * 45)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

print("\nPASS — V4 DataLoaders ready.")

### 40. Swin-Tiny initialization

In [ ]:
# STEP 40 — Swin-Tiny initialization
# STEP 52 — Swin-Tiny initialization

import timm
import torch

SWIN_MODEL_NAME = "swin_tiny_patch4_window7_224"

swin_model = timm.create_model(
    SWIN_MODEL_NAME,
    pretrained=True,
    num_classes=3
).to(DEVICE)

swin_parameters = sum(
    p.numel() for p in swin_model.parameters()
)

swin_config = timm.data.resolve_model_data_config(swin_model)

print("SWIN-TINY INITIALIZATION")
print("-" * 45)
print("Architecture:", SWIN_MODEL_NAME)
print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")
print("Parameters:", f"{swin_parameters:,}")
print("Output classes:", swin_model.num_classes)
print("Input size:", swin_config["input_size"])
print("Pretrained mean:", swin_config["mean"])
print("Pretrained std:", swin_config["std"])

swin_model.eval()

with torch.inference_mode():
    sample_images, _ = next(iter(val_loader))
    outputs = swin_model(sample_images[:2].to(DEVICE))

print("Output shape:", tuple(outputs.shape))

assert outputs.shape == (2, 3)

print("\nPASS — Swin-Tiny initialized successfully.")

### 41. Swin-Tiny preprocessing and DataLoaders

In [ ]:
# STEP 41 — Swin-Tiny preprocessing and DataLoaders
# STEP 53 — Swin-Tiny preprocessing and DataLoaders

from torchvision import transforms
from torch.utils.data import DataLoader

SWIN_MEAN = swin_config["mean"]
SWIN_STD = swin_config["std"]

swin_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(8),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(SWIN_MEAN, SWIN_STD)
])

swin_eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(SWIN_MEAN, SWIN_STD)
])

swin_train_dataset = CataractDataset(
    train_df, swin_train_transform
)

swin_val_dataset = CataractDataset(
    val_df, swin_eval_transform
)

swin_test_dataset = CataractDataset(
    test_df, swin_eval_transform
)

swin_train_loader = DataLoader(
    swin_train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

swin_val_loader = DataLoader(
    swin_val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

swin_test_loader = DataLoader(
    swin_test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

images, labels = next(iter(swin_train_loader))

print("SWIN-TINY V4 DATALOADERS")
print("-" * 45)
print("Training images:", len(swin_train_dataset))
print("Validation images:", len(swin_val_dataset))
print("Testing images:", len(swin_test_dataset))
print("Training batches:", len(swin_train_loader))
print("Validation batches:", len(swin_val_loader))
print("Testing batches:", len(swin_test_loader))
print("Batch shape:", images.shape)
print("Normalization mean:", SWIN_MEAN)
print("Normalization std:", SWIN_STD)

assert images.shape == (16, 3, 224, 224)

print("\nPASS — Swin-Tiny preprocessing ready.")

### 42. Swin-Tiny V4 training configuration

In [ ]:
# STEP 42 — Swin-Tiny V4 training configuration
# STEP 54 — Swin-Tiny V4 training configuration

import json
import torch
import torch.nn as nn
from pathlib import Path
from torch.amp import GradScaler

SWIN_EPOCHS = 10
SWIN_LR = 2e-5
SWIN_WEIGHT_DECAY = 0.01
SWIN_PATIENCE = 3

SWIN_MODEL_DIR = ROOT / "models" / "swin_v4"
SWIN_RESULTS_DIR = ROOT / "results" / "swin_v4"

SWIN_MODEL_DIR.mkdir(parents=True, exist_ok=True)
SWIN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SWIN_CHECKPOINT = SWIN_MODEL_DIR / "swin_v4_best.pth"

swin_criterion = nn.CrossEntropyLoss()

swin_optimizer = torch.optim.AdamW(
    swin_model.parameters(),
    lr=SWIN_LR,
    weight_decay=SWIN_WEIGHT_DECAY
)

swin_scaler = GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

swin_experiment_config = {
    "model": "Swin-Tiny",
    "architecture": SWIN_MODEL_NAME,
    "dataset": "V4",
    "train_images": len(swin_train_dataset),
    "val_images": len(swin_val_dataset),
    "test_images": len(swin_test_dataset),
    "num_classes": 3,
    "epochs": SWIN_EPOCHS,
    "batch_size": 16,
    "learning_rate": SWIN_LR,
    "weight_decay": SWIN_WEIGHT_DECAY,
    "early_stopping_patience": SWIN_PATIENCE,
    "optimizer": "AdamW",
    "loss": "CrossEntropyLoss",
    "mixed_precision": DEVICE.type == "cuda",
    "normalization_mean": list(SWIN_MEAN),
    "normalization_std": list(SWIN_STD),
    "parameters": swin_parameters
}

with open(
    SWIN_RESULTS_DIR / "experiment_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(swin_experiment_config, f, indent=4)

print("SWIN-TINY V4 TRAINING CONFIGURATION")
print("-" * 50)
print("Device:", DEVICE)
print("Architecture:", SWIN_MODEL_NAME)
print("Parameters:", f"{swin_parameters:,}")
print("Epochs:", SWIN_EPOCHS)
print("Learning rate:", SWIN_LR)
print("Batch size:", 16)
print("Mixed precision:", DEVICE.type == "cuda")
print("Checkpoint:", SWIN_CHECKPOINT)
print("Results directory:", SWIN_RESULTS_DIR)

print("\nPASS — Swin-Tiny training configuration ready.")

### 43. Swin-Tiny V4 training

In [ ]:
# STEP 43 — Swin-Tiny V4 training
# STEP 55 — Swin-Tiny V4 training

import time
import pandas as pd
import numpy as np
import torch
from torch.amp import autocast
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

def swin_run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    all_labels = []
    all_predictions = []

    for images, labels in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda")
            ):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * images.size(0)
        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(outputs.argmax(dim=1).detach().cpu().numpy())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_predictions),
        "macro_f1": f1_score(
            all_labels,
            all_predictions,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0
        )
    }

swin_history = []
swin_best_f1 = -1.0
swin_no_improvement = 0

print("Starting Swin-Tiny V4 training...")
print("Device:", DEVICE)
print("Checkpoint:", SWIN_CHECKPOINT)

for epoch in range(1, SWIN_EPOCHS + 1):
    start_time = time.time()

    train_metrics = swin_run_epoch(
        swin_model,
        swin_train_loader,
        swin_criterion,
        swin_optimizer,
        swin_scaler
    )

    val_metrics = swin_run_epoch(
        swin_model,
        swin_val_loader,
        swin_criterion
    )

    elapsed = time.time() - start_time

    swin_history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "epoch_seconds": elapsed
    })

    print(f"\nEpoch {epoch}/{SWIN_EPOCHS} | Time: {elapsed:.1f}s")
    print(
        f"Train: Loss={train_metrics['loss']:.4f}, "
        f"Acc={train_metrics['accuracy']:.4f}, "
        f"F1={train_metrics['macro_f1']:.4f}"
    )
    print(
        f"Val:   Loss={val_metrics['loss']:.4f}, "
        f"Acc={val_metrics['accuracy']:.4f}, "
        f"F1={val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["macro_f1"] > swin_best_f1 + 1e-8:
        swin_best_f1 = val_metrics["macro_f1"]
        swin_no_improvement = 0

        torch.save(
            swin_model.state_dict(),
            SWIN_CHECKPOINT
        )

        print("Best Swin-Tiny checkpoint saved.")
    else:
        swin_no_improvement += 1
        print(
            f"No validation F1 improvement: "
            f"{swin_no_improvement}/{SWIN_PATIENCE}"
        )

    pd.DataFrame(swin_history).to_csv(
        SWIN_RESULTS_DIR / "training_history.csv",
        index=False
    )

    if swin_no_improvement >= SWIN_PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\nSWIN-TINY V4 TRAINING COMPLETE")
print("-" * 45)
print("Epochs completed:", len(swin_history))
print("Best validation macro-F1:", round(swin_best_f1, 6))
print("Best checkpoint:", SWIN_CHECKPOINT)
print("History:", SWIN_RESULTS_DIR / "training_history.csv")

### 44. Swin-Tiny V4 test evaluation

In [ ]:
# STEP 44 — Swin-Tiny V4 test evaluation
# STEP 56 — Swin-Tiny V4 test evaluation

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

swin_model.load_state_dict(
    torch.load(SWIN_CHECKPOINT, map_location=DEVICE, weights_only=True)
)

swin_model.eval()

swin_true = []
swin_pred = []
swin_confidence = []

with torch.inference_mode():
    for images, labels in swin_test_loader:
        images = images.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            outputs = swin_model(images)

        probabilities = torch.softmax(outputs.float(), dim=1)
        confidence, predictions = probabilities.max(dim=1)

        swin_true.extend(labels.numpy().tolist())
        swin_pred.extend(predictions.cpu().numpy().tolist())
        swin_confidence.extend(confidence.cpu().numpy().tolist())

swin_accuracy = accuracy_score(swin_true, swin_pred)
swin_precision = precision_score(
    swin_true, swin_pred, average="macro", zero_division=0
)
swin_recall = recall_score(
    swin_true, swin_pred, average="macro", zero_division=0
)
swin_f1 = f1_score(
    swin_true, swin_pred, average="macro", zero_division=0
)

swin_cm = confusion_matrix(swin_true, swin_pred, labels=[0, 1, 2])

swin_metrics = pd.DataFrame([{
    "model": "Swin-Tiny",
    "dataset": "V4",
    "test_images": len(swin_true),
    "accuracy": swin_accuracy,
    "macro_precision": swin_precision,
    "macro_recall": swin_recall,
    "macro_f1": swin_f1,
    "parameters": swin_parameters
}])

swin_predictions = test_df.copy()
swin_predictions["true_label"] = [
    CLASS_NAMES[i] for i in swin_true
]
swin_predictions["predicted_label"] = [
    CLASS_NAMES[i] for i in swin_pred
]
swin_predictions["confidence"] = swin_confidence
swin_predictions["correct"] = np.array(swin_true) == np.array(swin_pred)

swin_metrics.to_csv(
    SWIN_RESULTS_DIR / "test_metrics.csv", index=False
)

swin_predictions.to_csv(
    SWIN_RESULTS_DIR / "test_predictions.csv", index=False
)

print("SWIN-TINY V4 TEST EVALUATION")
print("-" * 50)
print(swin_metrics.to_string(index=False))
print("\nConfusion matrix:")
print(swin_cm)
print("\nClassification report:")
print(
    classification_report(
        swin_true,
        swin_pred,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES,
        zero_division=0
    )
)
print("Mean confidence:", round(float(np.mean(swin_confidence)), 6))
print("Minimum confidence:", round(float(np.min(swin_confidence)), 6))
print("\nPASS — Swin-Tiny test evaluation completed.")

### 45. Swin-Tiny V4 misclassification analysis

In [ ]:
# STEP 45 — Swin-Tiny V4 misclassification analysis
# STEP 57 — Swin-Tiny V4 misclassification analysis

swin_errors = swin_predictions[
    ~swin_predictions["correct"]
].copy()

print("SWIN-TINY V4 ERROR ANALYSIS")
print("-" * 50)

print("Total misclassified images:", len(swin_errors))
print("Distinct visual families:", swin_errors["visual_family_id"].nunique())

print("\nErrors by source:")
print(swin_errors["source"].value_counts())

print("\nErrors by actual class:")
print(swin_errors["true_label"].value_counts())

print("\nErrors grouped by visual family:")
print(
    swin_errors.groupby(
        ["visual_family_id", "source", "true_label"]
    ).agg(
        error_count=("correct", "size"),
        mean_confidence=("confidence", "mean"),
        min_confidence=("confidence", "min")
    ).sort_values("error_count", ascending=False)
)

print("\nSource-specific test performance:")
print(
    swin_predictions.groupby("source").agg(
        total_images=("correct", "size"),
        correct_predictions=("correct", "sum"),
        accuracy=("correct", "mean")
    )
)

print("\nPASS — Swin-Tiny error analysis completed.")

### 46. Swin-Tiny V4 performance visualizations

In [ ]:
# STEP 46 — Swin-Tiny V4 performance visualizations
# STEP 58 — Swin-Tiny V4 performance visualizations

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay

swin_history_df = pd.read_csv(
    SWIN_RESULTS_DIR / "training_history.csv"
)

plt.figure(figsize=(9, 5))

plt.plot(
    swin_history_df["epoch"],
    swin_history_df["train_accuracy"],
    marker="o",
    label="Training accuracy"
)

plt.plot(
    swin_history_df["epoch"],
    swin_history_df["val_accuracy"],
    marker="s",
    label="Validation accuracy"
)

plt.title("Swin-Tiny — V4 Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

plt.savefig(
    SWIN_RESULTS_DIR / "swin_v4_accuracy_curve.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

fig, ax = plt.subplots(figsize=(7, 6))

display = ConfusionMatrixDisplay(
    confusion_matrix=swin_cm,
    display_labels=["Normal", "Immature", "Mature"]
)

display.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

plt.title("Swin-Tiny — V4 Test Confusion Matrix")
plt.tight_layout()

plt.savefig(
    SWIN_RESULTS_DIR / "swin_v4_confusion_matrix.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("PASS — Swin-Tiny figures saved.")
print("Location:", SWIN_RESULTS_DIR)

## 5. MobileViT-S: training and evaluation
Evaluate a compact hybrid convolutional-transformer architecture on the same V4 partitions.

### 47. MobileViT-S V4 model initialization

In [ ]:
# STEP 47 — MobileViT-S V4 model initialization
# STEP 59 — MobileViT-S V4 model initialization

import timm
import torch

MOBILEVIT_MODEL_NAME = "mobilevit_s"

mobilevit_model = timm.create_model(
    MOBILEVIT_MODEL_NAME,
    pretrained=True,
    num_classes=3
).to(DEVICE)

mobilevit_config = timm.data.resolve_model_data_config(
    mobilevit_model
)

mobilevit_parameters = sum(
    p.numel() for p in mobilevit_model.parameters()
)

mobilevit_model.eval()

with torch.inference_mode():
    sample_input = torch.randn(
        2, 3, 256, 256,
        device=DEVICE
    )
    sample_output = mobilevit_model(sample_input)

print("MOBILEVIT-S INITIALIZATION")
print("-" * 50)
print("Architecture:", MOBILEVIT_MODEL_NAME)
print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))
print("Parameters:", f"{mobilevit_parameters:,}")
print("Output classes:", 3)
print("Pretrained input size:", mobilevit_config["input_size"])
print("Pretrained mean:", mobilevit_config["mean"])
print("Pretrained std:", mobilevit_config["std"])
print("Output shape:", tuple(sample_output.shape))

assert sample_output.shape == (2, 3)

print("\nPASS — MobileViT-S initialized successfully.")

### 48. MobileViT-S V4 preprocessing and DataLoaders

In [ ]:
# STEP 48 — MobileViT-S V4 preprocessing and DataLoaders
# STEP 60 — MobileViT-S V4 preprocessing and DataLoaders

from torchvision import transforms
from torch.utils.data import DataLoader

MOBILEVIT_SIZE = mobilevit_config["input_size"][-2:]
MOBILEVIT_MEAN = mobilevit_config["mean"]
MOBILEVIT_STD = mobilevit_config["std"]

mobilevit_train_transform = transforms.Compose([
    transforms.Resize(MOBILEVIT_SIZE),
    transforms.RandomRotation(8),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(MOBILEVIT_MEAN, MOBILEVIT_STD)
])

mobilevit_eval_transform = transforms.Compose([
    transforms.Resize(MOBILEVIT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MOBILEVIT_MEAN, MOBILEVIT_STD)
])

mobilevit_train_dataset = CataractDataset(
    train_df, mobilevit_train_transform
)
mobilevit_val_dataset = CataractDataset(
    val_df, mobilevit_eval_transform
)
mobilevit_test_dataset = CataractDataset(
    test_df, mobilevit_eval_transform
)

mobilevit_train_loader = DataLoader(
    mobilevit_train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

mobilevit_val_loader = DataLoader(
    mobilevit_val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

mobilevit_test_loader = DataLoader(
    mobilevit_test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

images, labels = next(iter(mobilevit_train_loader))

print("MOBILEVIT-S V4 DATALOADERS")
print("-" * 45)
print("Training images:", len(mobilevit_train_dataset))
print("Validation images:", len(mobilevit_val_dataset))
print("Testing images:", len(mobilevit_test_dataset))
print("Training batches:", len(mobilevit_train_loader))
print("Validation batches:", len(mobilevit_val_loader))
print("Testing batches:", len(mobilevit_test_loader))
print("Batch shape:", images.shape)
print("Normalization mean:", MOBILEVIT_MEAN)
print("Normalization std:", MOBILEVIT_STD)

assert images.shape[1:] == (3, 256, 256)
assert len(mobilevit_train_dataset) == 2454
assert len(mobilevit_val_dataset) == 517
assert len(mobilevit_test_dataset) == 571

print("\nPASS — MobileViT-S V4 DataLoaders ready.")

### 49. MobileViT-S V4 training configuration

In [ ]:
# STEP 49 — MobileViT-S V4 training configuration
# STEP 61 — MobileViT-S V4 training configuration

import torch
import torch.nn as nn
from pathlib import Path

MOBILEVIT_EPOCHS = 10
MOBILEVIT_LR = 2e-5
MOBILEVIT_WEIGHT_DECAY = 0.01
MOBILEVIT_PATIENCE = 3

MOBILEVIT_MODEL_DIR = ROOT / "models" / "mobilevit_v4"
MOBILEVIT_RESULTS_DIR = ROOT / "results" / "mobilevit_v4"

MOBILEVIT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
MOBILEVIT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MOBILEVIT_CHECKPOINT = (
    MOBILEVIT_MODEL_DIR / "mobilevit_v4_best.pth"
)

mobilevit_model.train()

mobilevit_criterion = nn.CrossEntropyLoss()

mobilevit_optimizer = torch.optim.AdamW(
    mobilevit_model.parameters(),
    lr=MOBILEVIT_LR,
    weight_decay=MOBILEVIT_WEIGHT_DECAY
)

mobilevit_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

print("MOBILEVIT-S V4 TRAINING CONFIGURATION")
print("-" * 50)
print("Device:", DEVICE)
print("Architecture:", MOBILEVIT_MODEL_NAME)
print("Parameters:", f"{mobilevit_parameters:,}")
print("Epochs:", MOBILEVIT_EPOCHS)
print("Learning rate:", MOBILEVIT_LR)
print("Batch size:", mobilevit_train_loader.batch_size)
print("Weight decay:", MOBILEVIT_WEIGHT_DECAY)
print("Mixed precision:", mobilevit_scaler.is_enabled())
print("Early stopping patience:", MOBILEVIT_PATIENCE)
print("Checkpoint:", MOBILEVIT_CHECKPOINT)
print("Results directory:", MOBILEVIT_RESULTS_DIR)

print("\nPASS — MobileViT-S training configuration ready.")

### 50. MobileViT-S V4 training

In [ ]:
# STEP 50 — MobileViT-S V4 training
# STEP 62 — MobileViT-S V4 training

import time
import json
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

def mobilevit_run_epoch(model, loader, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    all_true = []
    all_pred = []

    for images, labels in tqdm(
        loader,
        desc="Train" if training else "Validation",
        leave=False
    ):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda")
            ):
                outputs = model(images)
                loss = mobilevit_criterion(outputs, labels)

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * labels.size(0)
        all_true.extend(labels.cpu().tolist())
        all_pred.extend(outputs.argmax(dim=1).cpu().tolist())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_true, all_pred),
        "macro_f1": f1_score(
            all_true,
            all_pred,
            average="macro",
            zero_division=0
        )
    }


mobilevit_history = []
mobilevit_best_f1 = -1.0
mobilevit_no_improvement = 0

mobilevit_experiment_config = {
    "model": "MobileViT-S",
    "architecture": MOBILEVIT_MODEL_NAME,
    "dataset": "V4",
    "train_images": len(mobilevit_train_dataset),
    "val_images": len(mobilevit_val_dataset),
    "test_images": len(mobilevit_test_dataset),
    "parameters": mobilevit_parameters,
    "epochs_max": MOBILEVIT_EPOCHS,
    "learning_rate": MOBILEVIT_LR,
    "weight_decay": MOBILEVIT_WEIGHT_DECAY,
    "batch_size": mobilevit_train_loader.batch_size,
    "patience": MOBILEVIT_PATIENCE,
    "checkpoint_selection": "validation_macro_f1",
    "input_size": list(MOBILEVIT_SIZE),
    "mean": list(MOBILEVIT_MEAN),
    "std": list(MOBILEVIT_STD)
}

with open(
    MOBILEVIT_RESULTS_DIR / "experiment_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(mobilevit_experiment_config, f, indent=2)

print("Starting MobileViT-S V4 training...")
print("Device:", DEVICE)
print("Checkpoint:", MOBILEVIT_CHECKPOINT)

for epoch in range(1, MOBILEVIT_EPOCHS + 1):
    start_time = time.time()

    train_metrics = mobilevit_run_epoch(
        mobilevit_model,
        mobilevit_train_loader,
        optimizer=mobilevit_optimizer,
        scaler=mobilevit_scaler
    )

    val_metrics = mobilevit_run_epoch(
        mobilevit_model,
        mobilevit_val_loader
    )

    elapsed = time.time() - start_time

    mobilevit_history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "epoch_seconds": elapsed
    })

    pd.DataFrame(mobilevit_history).to_csv(
        MOBILEVIT_RESULTS_DIR / "training_history.csv",
        index=False
    )

    print(f"\nEpoch {epoch}/{MOBILEVIT_EPOCHS} | Time: {elapsed:.1f}s")
    print(
        f"Train: Loss={train_metrics['loss']:.4f}, "
        f"Acc={train_metrics['accuracy']:.4f}, "
        f"F1={train_metrics['macro_f1']:.4f}"
    )
    print(
        f"Val:   Loss={val_metrics['loss']:.4f}, "
        f"Acc={val_metrics['accuracy']:.4f}, "
        f"F1={val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["macro_f1"] > mobilevit_best_f1 + 1e-6:
        mobilevit_best_f1 = val_metrics["macro_f1"]
        mobilevit_no_improvement = 0

        torch.save(
            mobilevit_model.state_dict(),
            MOBILEVIT_CHECKPOINT
        )

        print("Best MobileViT-S checkpoint saved.")
    else:
        mobilevit_no_improvement += 1
        print(
            "No validation F1 improvement:",
            f"{mobilevit_no_improvement}/{MOBILEVIT_PATIENCE}"
        )

    if mobilevit_no_improvement >= MOBILEVIT_PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\nMOBILEVIT-S V4 TRAINING COMPLETE")
print("-" * 45)
print("Epochs completed:", len(mobilevit_history))
print("Best validation macro-F1:", round(mobilevit_best_f1, 6))
print("Best checkpoint:", MOBILEVIT_CHECKPOINT)
print(
    "History:",
    MOBILEVIT_RESULTS_DIR / "training_history.csv"
)

### 51. MobileViT-S V4 test evaluation

In [ ]:
# STEP 51 — MobileViT-S V4 test evaluation
# STEP 63 — MobileViT-S V4 test evaluation

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

mobilevit_model.load_state_dict(
    torch.load(
        MOBILEVIT_CHECKPOINT,
        map_location=DEVICE,
        weights_only=True
    )
)

mobilevit_model.eval()

mobilevit_true = []
mobilevit_pred = []
mobilevit_confidence = []

with torch.inference_mode():
    for images, labels in mobilevit_test_loader:
        images = images.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            outputs = mobilevit_model(images)

        probabilities = torch.softmax(outputs.float(), dim=1)
        confidence, predictions = probabilities.max(dim=1)

        mobilevit_true.extend(labels.tolist())
        mobilevit_pred.extend(predictions.cpu().tolist())
        mobilevit_confidence.extend(confidence.cpu().tolist())

mobilevit_accuracy = accuracy_score(
    mobilevit_true, mobilevit_pred
)

precision, recall, macro_f1, _ = (
    precision_recall_fscore_support(
        mobilevit_true,
        mobilevit_pred,
        average="macro",
        zero_division=0
    )
)

mobilevit_cm = confusion_matrix(
    mobilevit_true,
    mobilevit_pred,
    labels=[0, 1, 2]
)

mobilevit_metrics = pd.DataFrame([{
    "model": "MobileViT-S",
    "dataset": "V4",
    "test_images": len(mobilevit_true),
    "accuracy": mobilevit_accuracy,
    "macro_precision": precision,
    "macro_recall": recall,
    "macro_f1": macro_f1,
    "parameters": mobilevit_parameters
}])

mobilevit_metrics.to_csv(
    MOBILEVIT_RESULTS_DIR / "test_metrics.csv",
    index=False
)

mobilevit_predictions = test_df.reset_index(drop=True).copy()

mobilevit_predictions["true_label"] = [
    CLASS_NAMES[i] for i in mobilevit_true
]

mobilevit_predictions["predicted_label"] = [
    CLASS_NAMES[i] for i in mobilevit_pred
]

mobilevit_predictions["confidence"] = mobilevit_confidence

mobilevit_predictions["correct"] = (
    np.array(mobilevit_true) == np.array(mobilevit_pred)
)

mobilevit_predictions.to_csv(
    MOBILEVIT_RESULTS_DIR / "test_predictions.csv",
    index=False
)

print("MOBILEVIT-S V4 TEST EVALUATION")
print("-" * 50)

print(mobilevit_metrics.to_string(index=False))

print("\nConfusion matrix:")
print(mobilevit_cm)

print("\nClassification report:")
print(
    classification_report(
        mobilevit_true,
        mobilevit_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )
)

print("Mean confidence:", round(np.mean(mobilevit_confidence), 6))
print("Minimum confidence:", round(np.min(mobilevit_confidence), 6))

print("\nPASS — MobileViT-S test evaluation completed.")

### 52. MobileViT-S V4 misclassification analysis

In [ ]:
# STEP 52 — MobileViT-S V4 misclassification analysis
# STEP 64 — MobileViT-S V4 misclassification analysis

mobilevit_errors = mobilevit_predictions[
    ~mobilevit_predictions["correct"]
].copy()

print("MOBILEVIT-S V4 ERROR ANALYSIS")
print("-" * 50)

print("Total misclassified images:", len(mobilevit_errors))
print(
    "Distinct visual families:",
    mobilevit_errors["visual_family_id"].nunique()
)

print("\nErrors by source:")
print(mobilevit_errors["source"].value_counts())

print("\nErrors by actual class:")
print(mobilevit_errors["true_label"].value_counts())

print("\nErrors grouped by visual family:")
print(
    mobilevit_errors.groupby(
        ["visual_family_id", "source", "true_label"]
    ).agg(
        error_count=("correct", "size"),
        mean_confidence=("confidence", "mean"),
        min_confidence=("confidence", "min")
    ).sort_values("error_count", ascending=False)
)

print("\nSource-specific test performance:")
print(
    mobilevit_predictions.groupby("source").agg(
        total_images=("correct", "size"),
        correct_predictions=("correct", "sum"),
        accuracy=("correct", "mean")
    )
)

print("\nPASS — MobileViT-S error analysis completed.")

### 53. MobileViT-S V4 performance visualizations

In [ ]:
# STEP 53 — MobileViT-S V4 performance visualizations
# STEP 65 — MobileViT-S V4 performance visualizations

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

mobilevit_history_df = pd.read_csv(
    MOBILEVIT_RESULTS_DIR / "training_history.csv"
)

plt.figure(figsize=(9, 5))

plt.plot(
    mobilevit_history_df["epoch"],
    mobilevit_history_df["train_accuracy"],
    marker="o",
    label="Training accuracy"
)

plt.plot(
    mobilevit_history_df["epoch"],
    mobilevit_history_df["val_accuracy"],
    marker="s",
    label="Validation accuracy"
)

plt.title("MobileViT-S — V4 Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

plt.savefig(
    MOBILEVIT_RESULTS_DIR / "mobilevit_v4_accuracy_curve.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

fig, ax = plt.subplots(figsize=(7, 6))

display = ConfusionMatrixDisplay(
    confusion_matrix=mobilevit_cm,
    display_labels=["Normal", "Immature", "Mature"]
)

display.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

plt.title("MobileViT-S — V4 Test Confusion Matrix")
plt.tight_layout()

plt.savefig(
    MOBILEVIT_RESULTS_DIR / "mobilevit_v4_confusion_matrix.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("PASS — MobileViT-S figures saved.")
print("Location:", MOBILEVIT_RESULTS_DIR)

## 6. Comparative evaluation and ensemble
Normalize prediction labels before joining per-image predictions. Compare individual models and majority voting on the common test set.

### 54. V4 three-model comparison

In [ ]:
# STEP 54 — V4 three-model comparison
# STEP 66 — V4 three-model comparison 

import pandas as pd

model_configs = {
    "ViT-B/16": {
        "path": ROOT / "results" / "vit_v4" / "test_metrics.csv",
        "parameters": 85800963
    },
    "Swin-Tiny": {
        "path": ROOT / "results" / "swin_v4" / "test_metrics.csv",
        "parameters": 27521661
    },
    "MobileViT-S": {
        "path": ROOT / "results" / "mobilevit_v4" / "test_metrics.csv",
        "parameters": 4939555
    }
}

comparison_rows = []

for model_name, config in model_configs.items():
    metrics = pd.read_csv(config["path"]).iloc[0]

    comparison_rows.append({
        "model": model_name,
        "accuracy": float(metrics["accuracy"]),
        "macro_precision": float(metrics["macro_precision"]),
        "macro_recall": float(metrics["macro_recall"]),
        "macro_f1": float(metrics["macro_f1"]),
        "parameters": config["parameters"],
        "test_images": int(metrics["test_images"])
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df["correct_predictions"] = (
    comparison_df["accuracy"] * comparison_df["test_images"]
).round().astype(int)

comparison_df["errors"] = (
    comparison_df["test_images"] - comparison_df["correct_predictions"]
)

comparison_df = comparison_df.sort_values(
    "macro_f1", ascending=False
).reset_index(drop=True)

comparison_dir = ROOT / "results" / "comparison_v4"
comparison_dir.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(
    comparison_dir / "three_model_comparison.csv",
    index=False
)

print("NETRASCAN.AI — V4 MODEL COMPARISON")
print("-" * 75)
print(comparison_df.to_string(index=False))

print("\nBest macro-F1:", comparison_df.iloc[0]["model"])

print("Fewest parameters:",
      comparison_df.loc[
          comparison_df["parameters"].idxmin(), "model"
      ])

print("\nPASS — Three-model comparison saved.")

### 55. V4 Model Comparison Dashboard

In [ ]:
# STEP 55 — V4 Model Comparison Dashboard
# STEP 67 — V4 Model Comparison Dashboard

import matplotlib.pyplot as plt
import numpy as np

plot_df = comparison_df.copy()

names = plot_df["model"].tolist()
accuracy = plot_df["accuracy"].to_numpy() * 100
macro_f1 = plot_df["macro_f1"].to_numpy() * 100
parameters = plot_df["parameters"].to_numpy() / 1_000_000
errors = plot_df["errors"].to_numpy()

colors = ["#4ECDC4", "#5B8DEF", "#F4A261"]

plt.style.use("dark_background")

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.patch.set_facecolor("#0B1220")

for ax in axes.flat:
    ax.set_facecolor("#151E2E")
    ax.grid(axis="y", alpha=0.15)
    ax.set_axisbelow(True)

bars = axes[0, 0].bar(names, accuracy, color=colors, width=0.55)
axes[0, 0].set_title("Test Accuracy", fontsize=15, weight="bold")
axes[0, 0].set_ylabel("Accuracy (%)")
axes[0, 0].set_ylim(0, 105)

for bar, value in zip(bars, accuracy):
    axes[0, 0].text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}%",
        ha="center",
        fontsize=11,
        weight="bold"
    )

bars = axes[0, 1].bar(names, macro_f1, color=colors, width=0.55)
axes[0, 1].set_title("Macro-F1 Score", fontsize=15, weight="bold")
axes[0, 1].set_ylabel("Macro-F1 (%)")
axes[0, 1].set_ylim(0, 105)

for bar, value in zip(bars, macro_f1):
    axes[0, 1].text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}%",
        ha="center",
        fontsize=11,
        weight="bold"
    )

bars = axes[1, 0].bar(names, parameters, color=colors, width=0.55)
axes[1, 0].set_title("Model Complexity", fontsize=15, weight="bold")
axes[1, 0].set_ylabel("Parameters (Millions)")
axes[1, 0].set_ylim(0, max(parameters) * 1.18)

for bar, value in zip(bars, parameters):
    axes[1, 0].text(
        bar.get_x() + bar.get_width() / 2,
        value + 1.5,
        f"{value:.2f}M",
        ha="center",
        fontsize=11,
        weight="bold"
    )

for i, name in enumerate(names):
    axes[1, 1].scatter(
        parameters[i],
        accuracy[i],
        s=220,
        color=colors[i],
        edgecolor="white",
        linewidth=1.2,
        label=name,
        zorder=3
    )

    axes[1, 1].annotate(
        name,
        (parameters[i], accuracy[i]),
        xytext=(8, 9),
        textcoords="offset points",
        fontsize=10
    )

axes[1, 1].set_title(
    "Accuracy vs Model Complexity",
    fontsize=15,
    weight="bold"
)
axes[1, 1].set_xlabel("Parameters (Millions)")
axes[1, 1].set_ylabel("Test Accuracy (%)")
axes[1, 1].set_xlim(0, max(parameters) * 1.2)
axes[1, 1].set_ylim(95, 100)
axes[1, 1].grid(alpha=0.2)
axes[1, 1].legend(loc="lower right")

fig.suptitle(
    "NetraScan.AI | Transformer Architecture Comparison",
    fontsize=22,
    weight="bold",
    y=0.98
)

fig.text(
    0.5,
    0.935,
    "V4 Dataset | 571 Test Images | Normal / Immature / Mature Cataract",
    ha="center",
    fontsize=12,
    color="#AAB7C4"
)

plt.tight_layout(rect=[0, 0.02, 1, 0.91])

dashboard_path = comparison_dir / "model_comparison_dashboard.png"

plt.savefig(
    dashboard_path,
    dpi=300,
    bbox_inches="tight",
    facecolor=fig.get_facecolor()
)

plt.show()

print("PASS — Model comparison dashboard saved.")
print("Location:", dashboard_path)

### 56. Corrected V4 Cross-Model Error Analysis

In [ ]:
# STEP 56 — Corrected V4 Cross-Model Error Analysis
# STEP 68B — Corrected V4 Cross-Model Error Analysis

import pandas as pd
import numpy as np

LABEL_MAP = {
    "0": "normal",
    "1": "immature",
    "2": "mature",
    "normal": "normal",
    "immature": "immature",
    "mature": "mature"
}

prediction_files = {
    "ViT-B/16": ROOT / "results" / "vit_v4" / "test_predictions.csv",
    "Swin-Tiny": ROOT / "results" / "swin_v4" / "test_predictions.csv",
    "MobileViT-S": ROOT / "results" / "mobilevit_v4" / "test_predictions.csv"
}

def normalize_labels(series):
    normalized = series.astype(str).str.strip().str.lower().map(LABEL_MAP)
    if normalized.isna().any():
        raise ValueError(
            f"Unknown labels: {series[normalized.isna()].unique()}"
        )
    return normalized

model_predictions = {}

for model_name, path in prediction_files.items():
    df = pd.read_csv(path)

    df["image_key"] = (
        df["source"].astype(str) + "::" +
        df["filename"].astype(str)
    )

    if df["image_key"].duplicated().any():
        raise ValueError(f"Duplicate image keys: {model_name}")

    df["true_label"] = normalize_labels(df["true_label"])
    df["predicted_label"] = normalize_labels(df["predicted_label"])

    model_predictions[model_name] = df.set_index("image_key")

reference = model_predictions["ViT-B/16"]

comparison = reference[
    ["source", "filename", "true_label", "visual_family_id"]
].copy()

for model_name, df in model_predictions.items():
    if set(df.index) != set(reference.index):
        raise ValueError(f"Test images differ for {model_name}")

    aligned = df.reindex(reference.index)

    if not aligned["true_label"].equals(reference["true_label"]):
        raise ValueError(f"Ground-truth mismatch: {model_name}")

    comparison[f"{model_name}_prediction"] = aligned["predicted_label"]
    comparison[f"{model_name}_correct"] = (
        aligned["predicted_label"] == comparison["true_label"]
    )

correct_cols = [
    f"{name}_correct" for name in prediction_files
]

comparison["models_correct"] = comparison[correct_cols].sum(axis=1)
comparison["models_wrong"] = 3 - comparison["models_correct"]

print("NETRASCAN.AI — CROSS-MODEL ERROR ANALYSIS")
print("-" * 65)
print("Total test images:", len(comparison))

print("\nError agreement:")
for wrong_count in range(4):
    count = (comparison["models_wrong"] == wrong_count).sum()
    print(f"{wrong_count} models wrong: {count}")

print("\nIndividual errors:")
for model_name in prediction_files:
    count = (~comparison[f"{model_name}_correct"]).sum()
    print(f"{model_name}: {count}")

print("\nUnique errors:")
for model_name in prediction_files:
    others = [m for m in prediction_files if m != model_name]

    mask = (
        (~comparison[f"{model_name}_correct"])
        & comparison[f"{others[0]}_correct"]
        & comparison[f"{others[1]}_correct"]
    )

    print(f"{model_name}: {mask.sum()}")

prediction_cols = [
    f"{name}_prediction" for name in prediction_files
]

def majority_vote(row):
    counts = row[prediction_cols].value_counts()
    return counts.index[0] if counts.iloc[0] >= 2 else "NO_MAJORITY"

comparison["ensemble_prediction"] = comparison.apply(
    majority_vote, axis=1
)

comparison["ensemble_correct"] = (
    comparison["ensemble_prediction"] == comparison["true_label"]
)

ensemble_correct = int(comparison["ensemble_correct"].sum())
ensemble_errors = len(comparison) - ensemble_correct

print("\nMAJORITY-VOTING ENSEMBLE")
print("-" * 65)
print(f"Accuracy: {ensemble_correct / len(comparison):.4%}")
print(f"Correct predictions: {ensemble_correct}")
print(f"Misclassifications: {ensemble_errors}")
print(
    "No-majority cases:",
    (comparison["ensemble_prediction"] == "NO_MAJORITY").sum()
)

comparison_dir = ROOT / "results" / "comparison_v4"
comparison_dir.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    comparison_dir / "cross_model_error_analysis.csv",
    index=False
)

print("\nPASS — Cross-model error analysis saved.")

### 57. Cross-Model Error Analysis Dashboard

In [ ]:
# STEP 57 — Cross-Model Error Analysis Dashboard
# STEP 69 — Cross-Model Error Analysis Dashboard

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

models = list(prediction_files.keys())
colors = ["#F4A261", "#4ECDC4", "#5B8DEF"]

individual_errors = [
    int((~comparison[f"{m}_correct"]).sum())
    for m in models
]

unique_errors = []
for model in models:
    others = [m for m in models if m != model]
    mask = (
        (~comparison[f"{model}_correct"])
        & comparison[f"{others[0]}_correct"]
        & comparison[f"{others[1]}_correct"]
    )
    unique_errors.append(int(mask.sum()))

agreement_counts = [
    int((comparison["models_wrong"] == i).sum())
    for i in range(4)
]

accuracy_values = [
    comparison[f"{m}_correct"].mean() * 100
    for m in models
]
accuracy_values.append(comparison["ensemble_correct"].mean() * 100)

plt.style.use("dark_background")

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.patch.set_facecolor("#0B1220")

for ax in axes.flat:
    ax.set_facecolor("#151E2E")
    ax.grid(axis="y", alpha=0.15)
    ax.set_axisbelow(True)

bars = axes[0, 0].bar(
    models, individual_errors, color=colors, width=0.55
)
axes[0, 0].set_title("Misclassifications by Architecture", fontsize=15, weight="bold")
axes[0, 0].set_ylabel("Incorrect Predictions")
axes[0, 0].set_ylim(0, max(individual_errors) * 1.25)

for bar, value in zip(bars, individual_errors):
    axes[0, 0].text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.35,
        str(value),
        ha="center", fontsize=12, weight="bold"
    )

agreement_labels = [
    "All Correct", "1 Wrong", "2 Wrong", "All Wrong"
]
agreement_colors = ["#4ECDC4", "#5B8DEF", "#F4A261", "#EF476F"]

bars = axes[0, 1].bar(
    agreement_labels, agreement_counts,
    color=agreement_colors, width=0.6
)
axes[0, 1].set_title("Cross-Model Error Agreement", fontsize=15, weight="bold")
axes[0, 1].set_ylabel("Number of Test Images")

for bar, value in zip(bars, agreement_counts):
    axes[0, 1].text(
        bar.get_x() + bar.get_width() / 2,
        value + 5,
        str(value),
        ha="center", fontsize=11, weight="bold"
    )

bars = axes[1, 0].bar(
    models, unique_errors, color=colors, width=0.55
)
axes[1, 0].set_title("Unique Misclassifications", fontsize=15, weight="bold")
axes[1, 0].set_ylabel("Errors Not Shared with Other Models")
axes[1, 0].set_ylim(0, max(unique_errors) + 2)

for bar, value in zip(bars, unique_errors):
    axes[1, 0].text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.15,
        str(value),
        ha="center", fontsize=12, weight="bold"
    )

accuracy_names = models + ["Majority Vote"]
accuracy_colors = colors + ["#B388EB"]

bars = axes[1, 1].bar(
    accuracy_names, accuracy_values,
    color=accuracy_colors, width=0.6
)
axes[1, 1].set_title("Single Models vs Majority Voting", fontsize=15, weight="bold")
axes[1, 1].set_ylabel("Test Accuracy (%)")
axes[1, 1].set_ylim(0, 105)
axes[1, 1].tick_params(axis="x", labelrotation=12)

for bar, value in zip(bars, accuracy_values):
    axes[1, 1].text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}%",
        ha="center", fontsize=10, weight="bold"
    )

fig.suptitle(
    "NetraScan.AI | Cross-Model Error Analysis",
    fontsize=22, weight="bold", y=0.98
)

fig.text(
    0.5, 0.935,
    "V4 Dataset | 571 Test Images | ViT-B/16 vs Swin-Tiny vs MobileViT-S",
    ha="center", fontsize=12, color="#AAB7C4"
)

plt.tight_layout(rect=[0, 0.02, 1, 0.91])

dashboard_path = comparison_dir / "cross_model_error_dashboard.png"

plt.savefig(
    dashboard_path,
    dpi=300,
    bbox_inches="tight",
    facecolor=fig.get_facecolor()
)

plt.show()

print("PASS — Cross-model error dashboard saved.")
print("Location:", dashboard_path)

### Recorded held-out results (571 images)
| Model | Accuracy | Correct | Macro-F1 |
|---|---:|---:|---:|
| ViT-B/16 | 97.37% | 556 | 0.97349 |
| Swin-Tiny | 98.95% | 565 | 0.98943 |
| MobileViT-S | 98.25% | 561 | 0.98266 |
| Majority vote | 98.25% | 561 | — |

Swin-Tiny achieved the highest individual accuracy. Majority voting did not outperform it on this test set. Results are transcribed from the original evaluation outputs, not recomputed.

## 7. V2 application workspace
V2 extends the existing standalone V1 application. The superseded notebook UI prototypes and V1 installation history are omitted; a working V1 source checkout and portable runtime are prerequisites for the migration cells below.

### 58. Environment and Project Discovery

In [ ]:
# STEP 58 — Environment and Project Discovery
# Cell 01: NetraScan.AI V2 - Environment and Project Discovery

from pathlib import Path
import sys
import shutil
import torch
import importlib.metadata as metadata

print("=" * 70)
print("NETRASCAN.AI V2 — DEVELOPMENT INITIALIZATION")
print("=" * 70)

CWD = Path.cwd().resolve()

print("\n1. SYSTEM ENVIRONMENT")
print("-" * 70)

print("Python:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Working directory:", CWD)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("Total VRAM:", round(props.total_memory / 1024**3, 2), "GB")
    print("CUDA version:", torch.version.cuda)

print("\n2. INSTALLED PACKAGES")
print("-" * 70)

for package in [
    "torch", "torchvision", "timm", "gradio",
    "transformers", "accelerate", "llama-cpp-python",
    "reportlab", "pillow", "grad-cam"
]:
    try:
        print(f"{package:22} {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package:22} NOT INSTALLED")

print("\n3. PROJECT ROOT DISCOVERY")
print("-" * 70)

PROJECT_ROOT = next(
    (
        p for p in [CWD, *CWD.parents]
        if (p / "models").is_dir()
        and (p / "notebooks").is_dir()
    ),
    None
)

if PROJECT_ROOT is not None:
    print("Project root:", PROJECT_ROOT)

    for item in sorted(PROJECT_ROOT.iterdir()):
        print("[DIR]" if item.is_dir() else "[FILE]", item.name)

    print("\n4. MODEL CHECKPOINTS")
    print("-" * 70)

    for p in (PROJECT_ROOT / "models").rglob("*"):
        if p.is_file() and p.suffix.lower() in {
            ".pt", ".pth", ".safetensors", ".ckpt", ".bin"
        }:
            print(p.relative_to(PROJECT_ROOT))
            print(f"  Size: {p.stat().st_size / 1024**2:.2f} MB")
else:
    print("Project root not automatically detected.")

print("\n5. DESKTOP STORAGE")
print("-" * 70)

total, used, free = shutil.disk_usage(CWD)

print(f"Free storage: {free / 1024**3:.2f} GB")

print("\n6. APPLICATION SOURCE DISCOVERY")
print("-" * 70)

SEARCH_ROOT = PROJECT_ROOT if PROJECT_ROOT else CWD

for filename in [
    "app.py",
    "inference.py",
    "explainability.py",
    "report_generator.py"
]:
    matches = list(SEARCH_ROOT.rglob(filename))

    if matches:
        for p in matches:
            print(p)
    else:
        print(f"{filename}: NOT FOUND IN PROJECT ROOT")

print("\n" + "=" * 70)
print("V2 ENVIRONMENT DISCOVERY COMPLETE")
print("No files created or modified.")
print("=" * 70)

### 59. Application Source Discovery

In [ ]:
# STEP 59 — Application Source Discovery
# Cell 02: NetraScan.AI V2 - Application Source Discovery

from pathlib import Path
import os

print("=" * 70)
print("NETRASCAN.AI V2 — APPLICATION SOURCE DISCOVERY")
print("=" * 70)

SEARCH_ROOTS = [
    Path(r"C:\NetraScanWorkspace"),
    Path.home() / "Desktop",
    Path.home() / "Documents",
    Path.home() / "PycharmProjects",
    Path.home() / "source" / "repos"
]

REQUIRED_FILES = [
    "app.py",
    "src/inference.py",
    "src/explainability.py",
    "src/report_generator.py"
]

SKIP_DIRS = {
    ".git", ".idea", ".venv", "venv",
    "__pycache__", "site-packages", "node_modules",
    "runtime", "Cataract v01.v7i.folder",
    "Dataset Katarak", "SLID"
}

CANDIDATES = []

for root in SEARCH_ROOTS:
    if not root.is_dir():
        continue

    print(f"\nSearching: {root}")

    for current, dirs, files in os.walk(root):
        dirs[:] = [
            d for d in dirs
            if d not in SKIP_DIRS
        ]

        if "app.py" not in files:
            continue

        candidate = Path(current)

        checks = {
            name: (candidate / name).is_file()
            for name in REQUIRED_FILES
        }

        if sum(checks.values()) >= 2:
            if candidate not in CANDIDATES:
                CANDIDATES.append(candidate)

                print("\nAPPLICATION CANDIDATE FOUND")
                print("-" * 70)
                print("Path:", candidate)

                for name, exists in checks.items():
                    print(
                        "PASS" if exists else "MISSING",
                        name
                    )

                print(
                    "Git repository:",
                    (candidate / ".git").exists()
                )

print("\n" + "=" * 70)
print("DISCOVERY SUMMARY")
print("=" * 70)

if CANDIDATES:
    for i, candidate in enumerate(CANDIDATES, 1):
        print(f"{i}. {candidate}")
else:
    print("No existing application source located.")
    print("We can recover the source from GitHub.")

print("\nNo files created or modified.")
print("SSD access: NONE")

### 60. Isolated Workspace Creation

In [ ]:
# STEP 60 — Isolated Workspace Creation
# Cell 03: NetraScan.AI V2 - Isolated Workspace Creation

from pathlib import Path
import shutil
import hashlib

BASE = Path(r"C:\NetraScanWorkspace")

SOURCE = BASE / "NetraScan_GitHub"
TRAINING_ROOT = BASE / "MY PROJECT E PAARVAI"
V2_ROOT = BASE / "NetraScan_V2"

print("=" * 70)
print("NETRASCAN.AI V2 — ISOLATED WORKSPACE CREATION")
print("=" * 70)

assert SOURCE.is_dir(), "Application source not found."
assert TRAINING_ROOT.is_dir(), "Training project not found."
assert not V2_ROOT.exists(), "V2 workspace already exists. Stopping to prevent overwrite."

CHECKPOINTS = [
    "vit_v4/vit_v4_best.pth",
    "swin_v4/swin_v4_best.pth",
    "mobilevit_v4/mobilevit_v4_best.pth"
]

REQUIRED_FILES = [
    "app.py",
    "src/inference.py",
    "src/explainability.py",
    "src/report_generator.py"
]

for filename in REQUIRED_FILES:
    assert (SOURCE / filename).is_file(), f"Missing: {filename}"

for filename in CHECKPOINTS:
    assert (TRAINING_ROOT / "models" / filename).is_file(), f"Missing: {filename}"

print("\n1. CREATING V2 WORKSPACE")
print("-" * 70)

shutil.copytree(
    SOURCE,
    V2_ROOT,
    ignore=shutil.ignore_patterns(
        ".git",
        ".idea",
        ".venv",
        "venv",
        "__pycache__",
        ".ipynb_checkpoints",
        "runtime",
        "models",
        "datasets",
        "reports"
    )
)

print("Workspace:", V2_ROOT)

for folder in [
    "models", "assets", "config",
    "llm", "reports", "tests"
]:
    (V2_ROOT / folder).mkdir(parents=True, exist_ok=True)
    print("READY:", folder)

print("\n2. COPYING FINAL V4 CHECKPOINTS")
print("-" * 70)

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

total_size = 0

for filename in CHECKPOINTS:

    source_file = TRAINING_ROOT / "models" / filename
    target_file = V2_ROOT / "models" / filename

    target_file.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(source_file, target_file)

    verified = sha256_file(source_file) == sha256_file(target_file)

    assert verified, f"Checkpoint integrity failed: {filename}"

    size = target_file.stat().st_size / 1024**2
    total_size += size

    print(f"\n{filename}")
    print(f"Size: {size:.2f} MB")
    print("SHA-256 integrity: PASS")

print(f"\nTotal checkpoint size: {total_size:.2f} MB")

print("\n3. VERIFYING APPLICATION SOURCE")
print("-" * 70)

for filename in REQUIRED_FILES:

    original = SOURCE / filename
    copied = V2_ROOT / filename

    verified = (
        copied.is_file()
        and sha256_file(original) == sha256_file(copied)
    )

    print("PASS" if verified else "FAIL", filename)

    assert verified, f"Source verification failed: {filename}"

print("\n4. WORKSPACE ISOLATION")
print("-" * 70)

assert not (V2_ROOT / ".git").exists()

print("Independent workspace: PASS")
print("Original source files unchanged: PASS")
print("Original checkpoints unchanged: PASS")
print("Git repository metadata excluded: PASS")
print("SSD access: NONE")

print("\n5. V2 ROOT CONTENTS")
print("-" * 70)

for item in sorted(V2_ROOT.iterdir()):
    print("[DIR]" if item.is_dir() else "[FILE]", item.name)

print("\n" + "=" * 70)
print("PASS — NETRASCAN.AI V2 WORKSPACE CREATED")
print("=" * 70)

### 61. Application Architecture Inspection

In [ ]:
# STEP 61 — Application Architecture Inspection
# Cell 04: NetraScan.AI V2 - Application Architecture Inspection

from pathlib import Path
import ast

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

print("=" * 70)
print("NETRASCAN.AI V2 — APPLICATION ARCHITECTURE INSPECTION")
print("=" * 70)

FILES = [
    "app.py",
    "src/inference.py",
    "src/explainability.py",
    "src/report_generator.py",
    "requirements.txt"
]

for filename in FILES:

    path = V2_ROOT / filename

    print("\n" + "=" * 70)
    print("FILE:", filename)
    print("=" * 70)

    assert path.is_file(), f"Missing file: {filename}"

    content = path.read_text(encoding="utf-8-sig")

    print("Size:", path.stat().st_size, "bytes")
    print("Lines:", len(content.splitlines()))

    if path.suffix == ".py":

        tree = ast.parse(content)

        print("\nFUNCTIONS AND CLASSES")
        print("-" * 70)

        for node in ast.walk(tree):

            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                print(f"Function: {node.name} | Line: {node.lineno}")

            elif isinstance(node, ast.ClassDef):
                print(f"Class: {node.name} | Line: {node.lineno}")

    print("\nCOMPLETE SOURCE")
    print("-" * 70)

    for number, line in enumerate(content.splitlines(), 1):
        print(f"{number:04d} | {line}")

print("\n" + "=" * 70)
print("ADDITIONAL PROJECT ASSETS")
print("=" * 70)

for folder in ["assets", "config", "data", "src"]:

    directory = V2_ROOT / folder

    print(f"\n[{folder}]")

    if directory.exists():

        files = sorted(p for p in directory.rglob("*") if p.is_file())

        if not files:
            print("EMPTY")

        for path in files:
            print(
                path.relative_to(V2_ROOT),
                f"| {path.stat().st_size / 1024:.2f} KB"
            )

    else:
        print("NOT FOUND")

print("\n" + "=" * 70)
print("PASS — APPLICATION ARCHITECTURE INSPECTED")
print("No files modified.")
print("SSD access: NONE")
print("=" * 70)

### 62. Checkpoint Correction and Baseline Test

In [ ]:
# STEP 62 — Checkpoint Correction and Baseline Test
# Cell 05: NetraScan.AI V2 - Checkpoint Correction and Baseline Test

from pathlib import Path
from PIL import Image
import subprocess
import sys
import ast
import shutil

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

INFERENCE_FILE = V2_ROOT / "src" / "inference.py"

print("=" * 70)
print("NETRASCAN.AI V2 — BASELINE VERIFICATION")
print("=" * 70)

print("\n1. CORRECTING CHECKPOINT PATHS")
print("-" * 70)

source = INFERENCE_FILE.read_text(encoding="utf-8-sig")

corrections = {
    "vit_v4_best.pth": "vit_v4/vit_v4_best.pth",
    "swin_v4_best.pth": "swin_v4/swin_v4_best.pth",
    "mobilevit_v4_best.pth": "mobilevit_v4/mobilevit_v4_best.pth"
}

for original, corrected in corrections.items():

    checkpoint = V2_ROOT / "models" / corrected

    assert checkpoint.is_file(), f"Missing checkpoint: {checkpoint}"

    old = f'"checkpoint": "{original}"'
    new = f'"checkpoint": "{corrected}"'

    if old in source:
        source = source.replace(old, new)
        print("UPDATED:", corrected)

    elif new in source:
        print("ALREADY CORRECT:", corrected)

    else:
        raise RuntimeError(f"Unexpected configuration: {original}")

ast.parse(source)

backup = V2_ROOT / "tests" / "inference_v1_backup.py"

if not backup.exists():
    shutil.copy2(INFERENCE_FILE, backup)

INFERENCE_FILE.write_text(source, encoding="utf-8")

print("PASS — Checkpoint paths corrected.")
print("PASS — Python syntax verified.")

print("\n2. RUNNING ISOLATED INFERENCE TEST")
print("-" * 70)

test_code = r'''
import torch
import json
import numpy as np
from PIL import Image

from src.inference import NetraScanInference
from src.explainability import NetraScanXAI

print("CUDA available:", torch.cuda.is_available())

engine = NetraScanInference()

assert len(engine.models) == 3

print("\nPASS — All three models loaded.")

rng = np.random.default_rng(42)

pixels = rng.integers(
    0, 256,
    size=(256, 256, 3),
    dtype=np.uint8
)

image = Image.fromarray(pixels)

result = engine.predict(image)

print("\nBASELINE PREDICTION OUTPUT")
print(json.dumps(result, indent=4))

for name, details in result["predictions"].items():

    probabilities = details["probabilities"]

    assert abs(sum(probabilities.values()) - 1.0) < 1e-5

    assert 0 <= details["confidence"] <= 1

    print("PASS — Probability validation:", name)

print("\nGRAD-CAM VERIFICATION")

xai = NetraScanXAI(engine)

explanation = xai.generate(image)

assert explanation["overlay"].size == (224, 224)
assert explanation["heatmap"].size == (224, 224)
assert np.isfinite(explanation["cam"]).all()

print("Overlay size:", explanation["overlay"].size)
print("Heatmap size:", explanation["heatmap"].size)
print("Target class index:", explanation["class_index"])

print("PASS — Grad-CAM generated successfully.")

if torch.cuda.is_available():

    print("\nGPU MEMORY")

    print(
        "Allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 3),
        "GB"
    )

    print(
        "Peak allocated:",
        round(torch.cuda.max_memory_allocated() / 1024**3, 3),
        "GB"
    )

print("\nPASS — V2 BASELINE INFERENCE VERIFIED")
'''

process = subprocess.run(
    [sys.executable, "-u", "-c", test_code],
    cwd=str(V2_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(process.stdout)

if process.stderr:
    print("\nDIAGNOSTIC OUTPUT")
    print(process.stderr)

assert process.returncode == 0, "Baseline test failed."

print("\n" + "=" * 70)
print("PASS — NETRASCAN.AI V2 BASELINE VERIFIED")
print("Original project untouched.")
print("SSD access: NONE")
print("=" * 70)

### 63. Local LLM Environment Preparation

In [ ]:
# STEP 63 — Local LLM Environment Preparation
# Cell 06: NetraScan.AI V2 - Local LLM Environment Preparation

from pathlib import Path
import sys
import subprocess
import shutil
import platform
import importlib.metadata as metadata
import torch

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

print("=" * 70)
print("NETRASCAN.AI V2 — LOCAL LLM ENVIRONMENT")
print("=" * 70)

print("\n1. SYSTEM INFORMATION")
print("-" * 70)

print("Operating system:", platform.platform())
print("Architecture:", platform.machine())
print("Processor:", platform.processor())
print("Python:", sys.version.split()[0])
print("Python executable:", sys.executable)

print("\n2. GPU RESOURCES")
print("-" * 70)

if torch.cuda.is_available():

    free_vram, total_vram = torch.cuda.mem_get_info()

    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Total VRAM: {total_vram / 1024**3:.2f} GB")
    print(f"Free VRAM: {free_vram / 1024**3:.2f} GB")
    print("Compute capability:", torch.cuda.get_device_capability(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

else:
    print("CUDA unavailable.")

print("\n3. SYSTEM RAM")
print("-" * 70)

try:
    import psutil

    memory = psutil.virtual_memory()

    print(f"Total RAM: {memory.total / 1024**3:.2f} GB")
    print(f"Available RAM: {memory.available / 1024**3:.2f} GB")

except ImportError:
    print("psutil not installed.")

print("\n4. AVAILABLE LLM DEPENDENCIES")
print("-" * 70)

PACKAGES = [
    "transformers",
    "accelerate",
    "bitsandbytes",
    "llama-cpp-python",
    "huggingface-hub",
    "safetensors",
    "tokenizers",
    "sentencepiece",
    "reportlab",
    "requests",
    "psutil"
]

for package in PACKAGES:

    try:
        version = metadata.version(package)
        print(f"{package:24} {version}")

    except metadata.PackageNotFoundError:
        print(f"{package:24} NOT INSTALLED")

print("\n5. AVAILABLE INFERENCE BACKENDS")
print("-" * 70)

for executable in [
    "ollama",
    "llama-cli",
    "llama-server",
    "nvcc",
    "cmake",
    "nvidia-smi"
]:

    location = shutil.which(executable)

    print(
        f"{executable:18}",
        location if location else "NOT FOUND"
    )

print("\n6. NVIDIA DRIVER INFORMATION")
print("-" * 70)

if shutil.which("nvidia-smi"):

    process = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,driver_version,memory.total,memory.free",
            "--format=csv"
        ],
        capture_output=True,
        text=True
    )

    print(process.stdout)

print("\n7. LLM WORKSPACE")
print("-" * 70)

LLM_ROOT = V2_ROOT / "llm"

assert LLM_ROOT.is_dir(), "LLM directory missing."

print("LLM directory:", LLM_ROOT)

files = list(LLM_ROOT.iterdir())

if files:
    for file in files:
        print(file.name)
else:
    print("LLM directory is empty.")

total, used, free = shutil.disk_usage(V2_ROOT)

print(f"Available storage: {free / 1024**3:.2f} GB")

print("\n" + "=" * 70)
print("PASS — LOCAL LLM ENVIRONMENT INSPECTED")
print("Selected candidate: Qwen2.5-3B-Instruct")
print("Target quantization: Q4_K_M GGUF")
print("No packages installed.")
print("No model downloaded.")
print("SSD access: NONE")
print("=" * 70)

## 8. Local GenAI and narrative validation
Qwen2.5-3B-Instruct (GGUF Q4_K_M) runs through a local llama.cpp server. The initial free-form and structured generation experiments are retained as evidence motivating grounded generation and deterministic fallback. Regression tests assess selected failure modes; they do not guarantee hallucination-free output.

### 64. Local LLM and Backend Download

In [ ]:
# STEP 64 — Local LLM and Backend Download
# Cell 07: NetraScan.AI V2 - Local LLM and Backend Download

from pathlib import Path
import requests
import zipfile
import hashlib
import json
import time
import subprocess
import re

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

LLM_ROOT = V2_ROOT / "llm"
BACKEND_DIR = LLM_ROOT / "backend"
WEIGHTS_DIR = LLM_ROOT / "weights"
DOWNLOAD_DIR = LLM_ROOT / "downloads"

for folder in [BACKEND_DIR, WEIGHTS_DIR, DOWNLOAD_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("NETRASCAN.AI V2 — LOCAL LLM DOWNLOAD")
print("=" * 70)

SESSION = requests.Session()

SESSION.headers.update({
    "User-Agent": "NetraScan-AI-V2"
})


def download_file(url, destination, expected_size=None):

    if destination.exists():

        if expected_size and destination.stat().st_size == expected_size:
            print("ALREADY DOWNLOADED:", destination.name)
            return destination

    temporary = destination.with_suffix(
        destination.suffix + ".part"
    )

    start = time.perf_counter()

    with SESSION.get(
        url,
        stream=True,
        timeout=(30, 180)
    ) as response:

        response.raise_for_status()

        total = int(
            response.headers.get("content-length", 0)
        ) or expected_size or 0

        downloaded = 0
        last_update = 0

        with temporary.open("wb") as file:

            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):

                if not chunk:
                    continue

                file.write(chunk)
                downloaded += len(chunk)

                elapsed = time.perf_counter() - start

                if elapsed - last_update >= 2:

                    speed = (
                        downloaded
                        / max(elapsed, 0.001)
                        / 1024**2
                    )

                    percentage = (
                        downloaded / total * 100
                        if total else 0
                    )

                    print(
                        f"\r{destination.name} | "
                        f"{percentage:.1f}% | "
                        f"{downloaded / 1024**2:.1f} MB | "
                        f"{speed:.2f} MB/s",
                        end="",
                        flush=True
                    )

                    last_update = elapsed

    print()

    if total:
        assert downloaded == total, "Incomplete download."

    if expected_size:
        assert downloaded == expected_size, "File size mismatch."

    temporary.replace(destination)

    print("PASS:", destination.name)

    return destination


def sha256_file(path):

    digest = hashlib.sha256()

    with path.open("rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


print("\n1. DISCOVERING LLAMA.CPP RELEASE")
print("-" * 70)

release_url = (
    "https://api.github.com/repos/"
    "ggml-org/llama.cpp/releases"
)

release = None
backend_asset = None
runtime_asset = None

for page in range(1, 4):

    print(f"Searching release page {page}...")

    response = SESSION.get(
        release_url,
        params={
            "per_page": 30,
            "page": page
        },
        timeout=60
    )

    response.raise_for_status()

    releases = response.json()

    if not releases:
        break

    for candidate in releases:

        tag = candidate["tag_name"]

        if not re.fullmatch(r"b\d+", tag):
            print("SKIPPING non-build release:", tag)
            continue

        if candidate.get("draft"):
            continue

        backend_candidates = []
        runtime_candidates = []

        for asset in candidate.get("assets", []):

            name = asset["name"].lower()

            compatible = (
                "win" in name
                and "cuda" in name
                and "x64" in name
                and re.search(
                    r"cuda[-_]?12(?:[._-]|$)",
                    name
                )
                and name.endswith(".zip")
            )

            if not compatible:
                continue

            if "cudart" in name:
                runtime_candidates.append(asset)

            elif name.startswith("llama-"):
                backend_candidates.append(asset)

        for backend in backend_candidates:

            backend_version = re.search(
                r"cuda[-_]?(12(?:\.\d+)*)",
                backend["name"].lower()
            )

            for runtime in runtime_candidates:

                runtime_version = re.search(
                    r"cuda[-_]?(12(?:\.\d+)*)",
                    runtime["name"].lower()
                )

                if (
                    backend_version
                    and runtime_version
                    and backend_version.group(1)
                    == runtime_version.group(1)
                ):

                    release = candidate
                    backend_asset = backend
                    runtime_asset = runtime
                    break

            if release is not None:
                break

        if release is not None:
            break

        print("SKIPPING incomplete build:", tag)

    if release is not None:
        break

assert release is not None, (
    "No compatible Windows x64 CUDA 12 build pair found."
)

print("\nRelease selected:", release["tag_name"])

selected_assets = [
    backend_asset,
    runtime_asset
]

for asset in selected_assets:

    print("SELECTED:", asset["name"])

    print(
        "Size:",
        round(asset["size"] / 1024**2, 2),
        "MB"
    )

print("PASS — Compatible CUDA backend discovered.")


print("\n2. DOWNLOADING PORTABLE BACKEND")
print("-" * 70)

archive_records = []

for asset in selected_assets:

    archive = download_file(
        asset["browser_download_url"],
        DOWNLOAD_DIR / asset["name"],
        expected_size=asset["size"]
    )

    checksum = sha256_file(archive)

    published_digest = asset.get("digest")

    if (
        published_digest
        and published_digest.startswith("sha256:")
    ):

        assert checksum == published_digest.split(":", 1)[1], (
            f"SHA-256 mismatch: {archive.name}"
        )

        print("Published SHA-256 verification: PASS")

    else:

        print("Published checksum unavailable.")
        print("Local SHA-256 recorded.")

    with zipfile.ZipFile(archive, "r") as zip_file:

        for member in zip_file.infolist():

            target = (
                BACKEND_DIR / member.filename
            ).resolve()

            assert target.is_relative_to(
                BACKEND_DIR.resolve()
            ), "Unsafe archive path."

        assert zip_file.testzip() is None, (
            f"Corrupted archive: {archive.name}"
        )

        zip_file.extractall(BACKEND_DIR)

    archive_records.append({
        "filename": archive.name,
        "sha256": checksum,
        "size_bytes": archive.stat().st_size
    })

    print("Extraction complete:", archive.name)


print("\n3. VERIFYING BACKEND")
print("-" * 70)

servers = list(
    BACKEND_DIR.rglob("llama-server.exe")
)

assert len(servers) == 1, (
    "llama-server.exe missing or ambiguous."
)

SERVER_EXE = servers[0]

print("Server:", SERVER_EXE)

process = subprocess.run(
    [str(SERVER_EXE), "--version"],
    cwd=str(SERVER_EXE.parent),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    timeout=60
)

print(process.stdout)
print(process.stderr)

assert process.returncode == 0, (
    f"Backend executable failed: {process.returncode}"
)

print("PASS — Backend executable verified.")


print("\n4. DOWNLOADING QWEN2.5-3B-INSTRUCT")
print("-" * 70)

MODEL_REPO = "Qwen/Qwen2.5-3B-Instruct-GGUF"

MODEL_NAME = "qwen2.5-3b-instruct-q4_k_m.gguf"

model_info = SESSION.get(
    f"https://huggingface.co/api/models/{MODEL_REPO}",
    params={"blobs": "true"},
    timeout=60
)

model_info.raise_for_status()

model_metadata = model_info.json()

revision = model_metadata["sha"]

model_entry = next(
    (
        entry
        for entry in model_metadata["siblings"]
        if entry["rfilename"] == MODEL_NAME
    ),
    None
)

assert model_entry is not None, (
    "Requested GGUF model not found in repository."
)

print("Model repository:", MODEL_REPO)
print("Model revision:", revision)
print("Selected model:", MODEL_NAME)

expected_size = model_entry.get("size")

if expected_size:

    print(
        "Download size:",
        round(expected_size / 1024**3, 3),
        "GB"
    )

MODEL_URL = (
    f"https://huggingface.co/{MODEL_REPO}/"
    f"resolve/{revision}/{MODEL_NAME}"
)

MODEL_PATH = download_file(
    MODEL_URL,
    WEIGHTS_DIR / MODEL_NAME,
    expected_size=expected_size
)


print("\n5. VERIFYING MODEL INTEGRITY")
print("-" * 70)

with MODEL_PATH.open("rb") as file:
    magic = file.read(4)

assert magic == b"GGUF", "Invalid GGUF file."

model_checksum = sha256_file(MODEL_PATH)

expected_checksum = (
    model_entry.get("lfs") or {}
).get("sha256")

if expected_checksum:

    assert model_checksum == expected_checksum, (
        "Model SHA-256 verification failed."
    )

    print("Published SHA-256 verification: PASS")

else:

    print("Remote checksum unavailable.")
    print("Local SHA-256 recorded.")

print("GGUF header: PASS")

print(
    "Model size:",
    round(MODEL_PATH.stat().st_size / 1024**3, 3),
    "GB"
)

print("SHA-256:", model_checksum)


print("\n6. SAVING REPRODUCIBILITY MANIFEST")
print("-" * 70)

manifest = {
    "model": "Qwen2.5-3B-Instruct",
    "repository": MODEL_REPO,
    "revision": revision,
    "quantization": "Q4_K_M",
    "format": "GGUF",
    "model_path": MODEL_PATH.relative_to(V2_ROOT).as_posix(),
    "model_sha256": model_checksum,
    "model_size_bytes": MODEL_PATH.stat().st_size,
    "backend": "llama.cpp",
    "backend_release": release["tag_name"],
    "backend_executable": SERVER_EXE.relative_to(V2_ROOT).as_posix(),
    "backend_archives": archive_records,
    "target_context_size": 4096
}

manifest_path = LLM_ROOT / "manifest.json"

manifest_path.write_text(
    json.dumps(manifest, indent=4),
    encoding="utf-8"
)

print("Manifest:", manifest_path)

print(json.dumps(manifest, indent=4))


print("\n7. FINAL WORKSPACE VERIFICATION")
print("-" * 70)

assert SERVER_EXE.is_file()
assert MODEL_PATH.is_file()
assert manifest_path.is_file()

print("PASS — Backend executable")
print("PASS — GGUF model checkpoint")
print("PASS — Reproducibility manifest")

total_size = sum(
    file.stat().st_size
    for file in LLM_ROOT.rglob("*")
    if file.is_file()
)

print(
    f"Total LLM workspace size: "
    f"{total_size / 1024**3:.3f} GB"
)

print("\n" + "=" * 70)
print("PASS — LOCAL LLM AND BACKEND DOWNLOADED")
print("No Python packages installed.")
print("Original project untouched.")
print("SSD access: NONE")
print("=" * 70)

### 65. First GenAI Inference and GPU Benchmark

In [ ]:
# STEP 65 — First GenAI Inference and GPU Benchmark
# Cell 08: NetraScan.AI V2 - First GenAI Inference and GPU Benchmark

from pathlib import Path
import subprocess
import requests
import json
import time
import socket
import os

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

LLM_ROOT = V2_ROOT / "llm"

MANIFEST = json.loads(
    (LLM_ROOT / "manifest.json").read_text(encoding="utf-8")
)

SERVER_EXE = V2_ROOT / MANIFEST["backend_executable"]
MODEL_PATH = V2_ROOT / MANIFEST["model_path"]

LOG_PATH = V2_ROOT / "tests" / "llm_benchmark.log"
RESULT_PATH = V2_ROOT / "tests" / "llm_benchmark.json"

HOST = "127.0.0.1"
PORT = 8089

BASE_URL = f"http://{HOST}:{PORT}"

SESSION = requests.Session()
SESSION.trust_env = False

print("=" * 70)
print("NETRASCAN.AI V2 — FIRST GENERATIVE AI INFERENCE")
print("=" * 70)

assert SERVER_EXE.is_file(), "Backend executable missing."
assert MODEL_PATH.is_file(), "GGUF checkpoint missing."


def gpu_memory():

    process = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=memory.used,memory.free,memory.total",
            "--format=csv,noheader,nounits"
        ],
        capture_output=True,
        text=True,
        timeout=15
    )

    process.check_returncode()

    used, free, total = [
        int(value.strip())
        for value in process.stdout.splitlines()[0].split(",")
    ]

    return {
        "used_mib": used,
        "free_mib": free,
        "total_mib": total
    }


print("\n1. PRE-INFERENCE GPU RESOURCES")
print("-" * 70)

memory_before = gpu_memory()

print(json.dumps(memory_before, indent=4))

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:

    assert sock.connect_ex((HOST, PORT)) != 0, (
        f"Port {PORT} is already occupied."
    )


print("\n2. CONFIGURING LOCAL LLM SERVER")
print("-" * 70)

command = [
    str(SERVER_EXE),
    "--model", str(MODEL_PATH),
    "--host", HOST,
    "--port", str(PORT),
    "--ctx-size", "4096",
    "--n-gpu-layers", "99",
    "--parallel", "1",
    "--threads", "8"
]

environment = os.environ.copy()

environment["PATH"] = (
    str(SERVER_EXE.parent)
    + os.pathsep
    + environment.get("PATH", "")
)

print("Model:", MANIFEST["model"])
print("Quantization:", MANIFEST["quantization"])
print("Backend:", MANIFEST["backend_release"])
print("Context size: 4096")
print("GPU layers requested: 99 (full offload)")
print("Local endpoint:", BASE_URL)


server = None
benchmark = None

with LOG_PATH.open("w", encoding="utf-8") as log_file:

    try:

        print("\n3. LOADING QWEN ON GPU")
        print("-" * 70)

        startup_start = time.perf_counter()

        server = subprocess.Popen(
            command,
            cwd=str(SERVER_EXE.parent),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=environment
        )

        ready = False

        for attempt in range(180):

            if server.poll() is not None:

                raise RuntimeError(
                    f"LLM server exited with code "
                    f"{server.returncode}. See {LOG_PATH}"
                )

            try:

                response = SESSION.get(
                    f"{BASE_URL}/health",
                    timeout=2
                )

                if response.status_code == 200:

                    if response.json().get("status") == "ok":
                        ready = True
                        break

            except (
                requests.RequestException,
                ValueError
            ):
                pass

            if attempt % 5 == 0:
                print("Waiting for model initialization...")

            time.sleep(1)

        assert ready, "LLM server initialization timed out."

        startup_time = time.perf_counter() - startup_start

        print("PASS — Local LLM server ready.")
        print(f"Initialization time: {startup_time:.2f} seconds")

        memory_loaded = gpu_memory()

        print("\nGPU MEMORY AFTER MODEL LOADING")
        print(json.dumps(memory_loaded, indent=4))

        print(
            "Observed VRAM increase:",
            memory_loaded["used_mib"]
            - memory_before["used_mib"],
            "MiB"
        )


        print("\n4. PREPARING STRUCTURED INFERENCE EVIDENCE")
        print("-" * 70)

        evidence = {
            "input_type": "Synthetic random image",
            "test_purpose": "Software functionality verification",
            "ensemble_prediction": "Mature Cataract",
            "agreement": "2/3",
            "models": [
                {
                    "architecture": "ViT-B/16",
                    "prediction": "Normal",
                    "confidence_percent": 44.81
                },
                {
                    "architecture": "Swin-Tiny",
                    "prediction": "Mature Cataract",
                    "confidence_percent": 46.46
                },
                {
                    "architecture": "MobileViT-S",
                    "prediction": "Mature Cataract",
                    "confidence_percent": 49.31
                }
            ],
            "explainability": {
                "method": "Grad-CAM",
                "architecture": "Swin-Tiny",
                "target_class": "Mature Cataract",
                "generated": True,
                "clinical_localization_validated": False
            },
            "clinical_use": False
        }

        print(json.dumps(evidence, indent=4))


        print("\n5. GENERATING FIRST AI INTERPRETATION")
        print("-" * 70)

        system_prompt = """
You are the research reporting assistant for NetraScan.AI,
an academic multi-transformer cataract classification prototype.

Interpret only the supplied structured evidence.

Rules:
1. Preserve all model names, predictions and numerical values.
2. Explain majority voting and model disagreement.
3. Explain that model confidence is not clinical certainty.
4. Explain Grad-CAM without inventing anatomical findings.
5. Explicitly acknowledge that the input is a synthetic random image.
6. Do not diagnose disease or recommend treatment.
7. Do not invent patient information.
8. Produce a concise research interpretation in 120-180 words.
9. Use professional academic English.
10. Do not include unsupported clinical conclusions.
""".strip()

        user_prompt = (
            "Generate a research interpretation of this evidence:\n\n"
            + json.dumps(evidence, indent=2)
        )

        payload = {
            "model": MANIFEST["model"],
            "messages": [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],
            "temperature": 0.2,
            "top_p": 0.9,
            "max_tokens": 384,
            "seed": 42,
            "stream": False
        }

        generation_start = time.perf_counter()

        response = SESSION.post(
            f"{BASE_URL}/v1/chat/completions",
            json=payload,
            timeout=180
        )

        response.raise_for_status()

        generation_time = (
            time.perf_counter() - generation_start
        )

        output = response.json()

        generated_text = (
            output["choices"][0]["message"]["content"]
        ).strip()

        assert generated_text, "Empty LLM response."

        print("\n" + "=" * 70)
        print("GENERATED RESEARCH INTERPRETATION")
        print("=" * 70)

        print(generated_text)

        print("=" * 70)


        print("\n6. PERFORMANCE BENCHMARK")
        print("-" * 70)

        usage = output.get("usage", {})
        timings = output.get("timings", {})

        prompt_tokens = usage.get("prompt_tokens", 0)
        completion_tokens = usage.get("completion_tokens", 0)

        generation_tps = timings.get("predicted_per_second")
        prompt_tps = timings.get("prompt_per_second")

        end_to_end_tps = (
            completion_tokens / generation_time
            if generation_time > 0
            else 0
        )

        memory_after = gpu_memory()

        print(f"Initialization: {startup_time:.2f} seconds")
        print(f"Response latency: {generation_time:.2f} seconds")
        print("Prompt tokens:", prompt_tokens)
        print("Generated tokens:", completion_tokens)

        if prompt_tps is not None:
            print(f"Prompt processing: {prompt_tps:.2f} tokens/sec")

        if generation_tps is not None:
            print(f"Generation speed: {generation_tps:.2f} tokens/sec")

        print(
            f"End-to-end throughput: "
            f"{end_to_end_tps:.2f} tokens/sec"
        )

        print(
            "Finish reason:",
            output["choices"][0].get("finish_reason")
        )

        print("\nGPU MEMORY AFTER GENERATION")
        print(json.dumps(memory_after, indent=4))


        print("\n7. BASIC OUTPUT CONSISTENCY CHECK")
        print("-" * 70)

        normalized = generated_text.lower()

        checks = {
            "Synthetic input acknowledged": (
                "synthetic" in normalized
                or "random image" in normalized
            ),
            "Majority prediction mentioned": (
                "mature cataract" in normalized
            ),
            "Model disagreement acknowledged": (
                "disagree" in normalized
                or "disagreement" in normalized
                or "two of" in normalized
                or "2/3" in normalized
                or "two out of" in normalized
            ),
            "Grad-CAM mentioned": (
                "grad-cam" in normalized
            ),
            "Clinical limitation mentioned": (
                "clinical" in normalized
                or "diagnos" in normalized
            )
        }

        for name, passed in checks.items():

            print(
                "PASS" if passed else "REVIEW",
                "—",
                name
            )

        print(
            f"\nBasic checks passed: "
            f"{sum(checks.values())}/{len(checks)}"
        )

        benchmark = {
            "model": MANIFEST["model"],
            "quantization": MANIFEST["quantization"],
            "backend": MANIFEST["backend_release"],
            "startup_seconds": round(startup_time, 3),
            "response_seconds": round(generation_time, 3),
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "prompt_tokens_per_second": prompt_tps,
            "generation_tokens_per_second": generation_tps,
            "end_to_end_tokens_per_second": round(
                end_to_end_tps, 3
            ),
            "memory_before": memory_before,
            "memory_loaded": memory_loaded,
            "memory_after_generation": memory_after,
            "consistency_checks": checks,
            "finish_reason": output["choices"][0].get(
                "finish_reason"
            ),
            "evidence": evidence,
            "generated_text": generated_text,
            "raw_timings": timings
        }

    finally:

        print("\n8. SHUTTING DOWN TEMPORARY LLM SERVER")
        print("-" * 70)

        if server is not None and server.poll() is None:

            server.terminate()

            try:
                server.wait(timeout=15)

            except subprocess.TimeoutExpired:
                server.kill()
                server.wait()

        print("Temporary LLM server stopped.")


print("\n9. GPU OFFLOADING VERIFICATION")
print("-" * 70)

log_content = LOG_PATH.read_text(
    encoding="utf-8",
    errors="replace"
)

keywords = [
    "offload",
    "cuda0",
    "cuda device",
    "cuda buffer",
    "cuda backend",
    "model buffer size",
    "kv buffer size",
    "compute buffer size"
]

matched_lines = []

for line in log_content.splitlines():

    if any(
        keyword in line.lower()
        for keyword in keywords
    ):
        matched_lines.append(line)
        print(line)

if not matched_lines:
    print("GPU offloading details require manual log inspection.")

memory_released = gpu_memory()

print("\nGPU MEMORY AFTER SERVER SHUTDOWN")
print(json.dumps(memory_released, indent=4))

assert benchmark is not None, "Benchmark did not complete."

benchmark["memory_after_shutdown"] = memory_released
benchmark["gpu_offloading_logs"] = matched_lines

RESULT_PATH.write_text(
    json.dumps(benchmark, indent=4),
    encoding="utf-8"
)

print("\n10. SAVING BENCHMARK RESULTS")
print("-" * 70)

print("Benchmark:", RESULT_PATH)
print("Backend logs:", LOG_PATH)

print("\n" + "=" * 70)
print("PASS — FIRST LOCAL GENAI INFERENCE COMPLETED")
print("No cloud API used.")
print("No Python packages installed.")
print("Original project untouched.")
print("SSD access: NONE")
print("=" * 70)

### 66. Structured Generation and Hallucination Mitigation

In [ ]:
# STEP 66 — Structured Generation and Hallucination Mitigation
# Cell 09: NetraScan.AI V2 - Structured Generation and Hallucination Mitigation

from pathlib import Path
from collections import Counter
import subprocess
import requests
import json
import time
import socket
import os
import re

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

MANIFEST = json.loads(
    (V2_ROOT / "llm" / "manifest.json").read_text(
        encoding="utf-8"
    )
)

SERVER_EXE = V2_ROOT / MANIFEST["backend_executable"]
MODEL_PATH = V2_ROOT / MANIFEST["model_path"]

LOG_PATH = V2_ROOT / "tests" / "structured_generation.log"
RESULT_PATH = V2_ROOT / "tests" / "structured_generation.json"

HOST = "127.0.0.1"
PORT = 8089
BASE_URL = f"http://{HOST}:{PORT}"

SESSION = requests.Session()
SESSION.trust_env = False

MODEL_NAMES = [
    "ViT-B/16",
    "Swin-Tiny",
    "MobileViT-S"
]

CLASSES = [
    "Normal",
    "Immature Cataract",
    "Mature Cataract"
]

print("=" * 70)
print("NETRASCAN.AI V2 — STRUCTURED GENERATION")
print("=" * 70)


def prepare_evidence(predictions, confidences):

    models = [
        {
            "model": name,
            "prediction": prediction,
            "confidence_percent": confidence
        }
        for name, prediction, confidence in zip(
            MODEL_NAMES,
            predictions,
            confidences
        )
    ]

    votes = Counter(predictions)

    majority_class, majority_count = votes.most_common(1)[0]

    ensemble = (
        majority_class
        if majority_count >= 2
        else "Inconclusive"
    )

    highest = max(
        models,
        key=lambda item: item["confidence_percent"]
    )

    return {
        "input_type": "Synthetic test evidence",
        "ensemble_prediction": ensemble,
        "agreement": f"{majority_count}/3",
        "highest_confidence_model": highest["model"],
        "highest_confidence_percent": highest["confidence_percent"],
        "models": models,
        "gradcam_model": "Swin-Tiny",
        "gradcam_target": predictions[1],
        "clinical_use": False
    }


TEST_CASES = {
    "majority_agreement": prepare_evidence(
        [
            "Normal",
            "Mature Cataract",
            "Mature Cataract"
        ],
        [44.81, 46.46, 49.31]
    ),

    "unanimous_agreement": prepare_evidence(
        [
            "Immature Cataract",
            "Immature Cataract",
            "Immature Cataract"
        ],
        [96.20, 98.40, 94.70]
    ),

    "inconclusive": prepare_evidence(
        [
            "Normal",
            "Immature Cataract",
            "Mature Cataract"
        ],
        [85.20, 91.30, 88.60]
    )
}


SCHEMA = {
    "type": "object",
    "properties": {
        "ensemble_prediction": {
            "type": "string",
            "enum": CLASSES + ["Inconclusive"]
        },
        "agreement": {
            "type": "string",
            "enum": ["1/3", "2/3", "3/3"]
        },
        "highest_confidence_model": {
            "type": "string",
            "enum": MODEL_NAMES
        },
        "gradcam_target": {
            "type": "string",
            "enum": CLASSES
        },
        "interpretation": {
            "type": "string"
        },
        "limitations": {
            "type": "string"
        }
    },
    "required": [
        "ensemble_prediction",
        "agreement",
        "highest_confidence_model",
        "gradcam_target",
        "interpretation",
        "limitations"
    ],
    "additionalProperties": False
}


SYSTEM_PROMPT = """
You are a research reporting assistant for NetraScan.AI,
an academic cataract image classification prototype.

Return a JSON object matching the requested schema.

Copy these fields exactly from the supplied evidence:
- ensemble_prediction
- agreement
- highest_confidence_model
- gradcam_target

The interpretation field must contain two or three concise
sentences explaining the qualitative significance of model
agreement or disagreement.

Do not repeat individual model names, class labels,
confidence scores or numerical values in the interpretation.
These facts are displayed separately by the application.

For unanimous agreement, explain that all models agree,
but agreement does not establish correctness.

For majority agreement, explain that the models are not
unanimous and the result requires cautious interpretation.

For inconclusive results, explain that no majority exists
and no ensemble classification can be established.

The limitations field must explain that:
- confidence is not clinical certainty;
- Grad-CAM visualizes model activation patterns;
- the supplied evidence is synthetic;
- the prototype is not clinically validated.

Never claim that disease has been confirmed.
Never claim that Grad-CAM identifies actual pathology.
Never provide treatment recommendations.
Never invent patient information.

Use concise professional academic English.
""".strip()


def validate_output(output, evidence):

    required = set(SCHEMA["required"])

    schema_ok = (
        isinstance(output, dict)
        and set(output.keys()) == required
        and all(
            isinstance(output[key], str)
            for key in required
        )
    )

    checks = {
        "Required JSON fields": schema_ok
    }

    if not schema_ok:
        return checks

    for field in [
        "ensemble_prediction",
        "agreement",
        "highest_confidence_model",
        "gradcam_target"
    ]:

        checks[f"Exact {field}"] = (
            output[field] == evidence[field]
        )

    interpretation = output["interpretation"].strip()
    limitations = output["limitations"].strip()

    combined = (
        interpretation + " " + limitations
    ).lower()

    checks["Non-empty interpretation"] = bool(
        interpretation
    )

    checks["Non-empty limitations"] = bool(
        limitations
    )

    checks["No numerical claims in narrative"] = (
        re.search(r"\d", combined) is None
    )

    checks["No repeated architecture names"] = not any(
        name.lower() in combined
        for name in MODEL_NAMES
    )

    checks["No repeated class labels in interpretation"] = (
        re.search(
            r"\b(?:normal|immature cataract|mature cataract)\b",
            interpretation.lower()
        ) is None
    )

    prohibited = [
        "supports a diagnosis",
        "confirms a diagnosis",
        "confirmed cataract",
        "disease confirmed",
        "patient has",
        "patient suffers",
        "localized the cataract",
        "identified pathology",
        "requires surgery",
        "recommended treatment"
    ]

    checks["No prohibited claim patterns"] = not any(
        phrase in combined
        for phrase in prohibited
    )

    checks["Synthetic evidence acknowledged"] = (
        "synthetic" in combined
    )

    checks["Grad-CAM acknowledged"] = (
        "grad-cam" in combined
    )

    checks["Clinical validation limitation"] = (
        "not clinically validated" in combined
        or "not been clinically validated" in combined
        or "lacks clinical validation" in combined
        or "no clinical validation" in combined
    )

    checks["Confidence limitation"] = (
        "confidence" in combined
        and (
            "certainty" in combined
            or "guarantee" in combined
            or "clinical accuracy" in combined
        )
    )

    return checks


def fallback_interpretation(evidence):

    agreement = evidence["agreement"]

    if agreement == "3/3":

        interpretation = (
            "The models demonstrate unanimous agreement. "
            "However, agreement alone does not establish "
            "classification correctness."
        )

    elif agreement == "2/3":

        interpretation = (
            "The majority vote establishes an ensemble "
            "classification, but the models are not unanimous. "
            "The disagreement warrants cautious interpretation."
        )

    else:

        interpretation = (
            "The models disagree and no majority exists. "
            "An ensemble classification cannot be established."
        )

    return {
        "ensemble_prediction": evidence["ensemble_prediction"],
        "agreement": evidence["agreement"],
        "highest_confidence_model": evidence[
            "highest_confidence_model"
        ],
        "gradcam_target": evidence["gradcam_target"],
        "interpretation": interpretation,
        "limitations": (
            "Model confidence is not clinical certainty. "
            "Grad-CAM visualizes model activation patterns "
            "rather than validated pathology. "
            "The supplied evidence is synthetic. "
            "The prototype is not clinically validated."
        )
    }


print("\n1. STARTING LOCAL LLM SERVER")
print("-" * 70)

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:

    assert sock.connect_ex((HOST, PORT)) != 0, (
        f"Port {PORT} already occupied."
    )

command = [
    str(SERVER_EXE),
    "--model", str(MODEL_PATH),
    "--host", HOST,
    "--port", str(PORT),
    "--ctx-size", "4096",
    "--n-gpu-layers", "99",
    "--parallel", "1",
    "--threads", "8"
]

environment = os.environ.copy()

environment["PATH"] = (
    str(SERVER_EXE.parent)
    + os.pathsep
    + environment.get("PATH", "")
)

server = None
results = {}

with LOG_PATH.open("w", encoding="utf-8") as log_file:

    try:

        server = subprocess.Popen(
            command,
            cwd=str(SERVER_EXE.parent),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=environment
        )

        startup = time.perf_counter()

        while True:

            if server.poll() is not None:

                raise RuntimeError(
                    f"Server exited: {server.returncode}. "
                    f"See {LOG_PATH}"
                )

            try:

                response = SESSION.get(
                    f"{BASE_URL}/health",
                    timeout=2
                )

                if (
                    response.status_code == 200
                    and response.json().get("status") == "ok"
                ):
                    break

            except (requests.RequestException, ValueError):
                pass

            if time.perf_counter() - startup > 180:
                raise TimeoutError("LLM initialization timed out.")

            time.sleep(1)

        print("PASS — Local LLM ready.")

        print(
            f"Startup time: "
            f"{time.perf_counter() - startup:.2f} seconds"
        )


        print("\n2. RUNNING STRUCTURED GENERATION TESTS")
        print("-" * 70)

        for test_name, evidence in TEST_CASES.items():

            print("\n" + "=" * 70)
            print("TEST:", test_name.upper())
            print("=" * 70)

            print("EXPECTED FACTS")

            for field in [
                "ensemble_prediction",
                "agreement",
                "highest_confidence_model",
                "gradcam_target"
            ]:

                print(f"{field}: {evidence[field]}")

            payload = {
                "model": MANIFEST["model"],
                "messages": [
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": json.dumps(
                            evidence,
                            indent=2
                        )
                    }
                ],
                "response_format": {
                    "type": "json_schema",
                    "json_schema": {
                        "name": "netrascan_interpretation",
                        "strict": True,
                        "schema": SCHEMA
                    }
                },
                "temperature": 0.0,
                "seed": 42,
                "max_tokens": 512,
                "stream": False
            }

            started = time.perf_counter()

            response = SESSION.post(
                f"{BASE_URL}/v1/chat/completions",
                json=payload,
                timeout=180
            )

            if not response.ok:

                print("HTTP error:", response.status_code)
                print(response.text[:2000])

                response.raise_for_status()

            latency = time.perf_counter() - started

            raw_response = response.json()

            generated_text = raw_response[
                "choices"
            ][0]["message"]["content"]

            finish_reason = raw_response[
                "choices"
            ][0].get("finish_reason")

            print("\nRAW GENERATED OUTPUT")

            print(generated_text)

            try:

                parsed = json.loads(generated_text)

                checks = validate_output(
                    parsed,
                    evidence
                )

            except (json.JSONDecodeError, TypeError):

                parsed = None

                checks = {
                    "Valid JSON": False
                }

            checks["Generation completed"] = (
                finish_reason == "stop"
            )

            accepted = all(checks.values())

            print("\nVALIDATION RESULTS")

            for name, passed in checks.items():

                print(
                    "PASS" if passed else "FAIL",
                    "—",
                    name
                )

            if accepted:

                final_output = parsed
                source = "LLM"

                print("\nPASS — LLM output accepted by checks.")

            else:

                final_output = fallback_interpretation(
                    evidence
                )

                source = "DETERMINISTIC_FALLBACK"

                print("\nREJECTED — Deterministic fallback activated.")

            print("\nFINAL INTERPRETATION")

            print(final_output["interpretation"])

            print("\nFINAL LIMITATIONS")

            print(final_output["limitations"])

            timings = raw_response.get("timings", {})

            print(f"\nLatency: {latency:.3f} seconds")

            print(
                "Generation tokens/sec:",
                timings.get(
                    "predicted_per_second",
                    "Unavailable"
                )
            )

            print("Report source:", source)

            results[test_name] = {
                "evidence": evidence,
                "raw_generated_text": generated_text,
                "parsed_output": parsed,
                "validation": checks,
                "llm_accepted": accepted,
                "final_output": final_output,
                "source": source,
                "latency_seconds": round(latency, 3),
                "timings": timings
            }

    finally:

        print("\n3. SHUTTING DOWN LOCAL SERVER")
        print("-" * 70)

        if server is not None and server.poll() is None:

            server.terminate()

            try:
                server.wait(timeout=15)

            except subprocess.TimeoutExpired:
                server.kill()
                server.wait()

        print("PASS — Temporary server stopped.")


print("\n4. FINAL VALIDATION SUMMARY")
print("-" * 70)

accepted_count = sum(
    result["llm_accepted"]
    for result in results.values()
)

for name, result in results.items():

    print(
        name,
        "=>",
        result["source"]
    )

print(
    f"\nLLM outputs accepted: "
    f"{accepted_count}/{len(TEST_CASES)}"
)

print(
    f"Fallbacks activated: "
    f"{len(results) - accepted_count}"
)

RESULT_PATH.write_text(
    json.dumps(results, indent=4),
    encoding="utf-8"
)

print("\nResults:", RESULT_PATH)
print("Server logs:", LOG_PATH)

print("\n" + "=" * 70)

if accepted_count == len(TEST_CASES):

    print("PASS — ALL STRUCTURED GENERATION TESTS ACCEPTED")

else:

    print("REVIEW — SOME LLM OUTPUTS REQUIRED FALLBACK")

print("Original project untouched.")
print("No Python packages installed.")
print("SSD access: NONE")
print("=" * 70)

### 67. Semantic Grounding and Controlled Generation

In [ ]:
# STEP 67 — Semantic Grounding and Controlled Generation
# Cell 10: NetraScan.AI V2 - Semantic Grounding and Controlled Generation

from pathlib import Path
import subprocess
import requests
import json
import time
import socket
import os
import re

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

MANIFEST = json.loads(
    (V2_ROOT / "llm" / "manifest.json").read_text(
        encoding="utf-8"
    )
)

PREVIOUS_RESULTS = json.loads(
    (V2_ROOT / "tests" / "structured_generation.json").read_text(
        encoding="utf-8"
    )
)

SERVER_EXE = V2_ROOT / MANIFEST["backend_executable"]
MODEL_PATH = V2_ROOT / MANIFEST["model_path"]

LOG_PATH = V2_ROOT / "tests" / "grounded_generation.log"
RESULT_PATH = V2_ROOT / "tests" / "grounded_generation.json"

HOST = "127.0.0.1"
PORT = 8089
BASE_URL = f"http://{HOST}:{PORT}"

SESSION = requests.Session()
SESSION.trust_env = False

print("=" * 70)
print("NETRASCAN.AI V2 — SEMANTIC GROUNDING")
print("=" * 70)


GROUNDING = {
    "3/3": {
        "facts": [
            "All models agree.",
            "An ensemble classification exists.",
            "Agreement alone does not establish correctness."
        ],
        "fallback": (
            "The models demonstrate unanimous agreement. "
            "Nevertheless, agreement alone does not establish "
            "classification correctness."
        ),
        "required_concepts": [
            r"\b(?:all models|unanimous|unanimity|complete agreement)\b",
            r"\b(?:correctness|accuracy|correct|accurate|validity)\b"
        ],
        "forbidden_concepts": [
            r"\bdisagree",
            r"\binconclusive\b",
            r"\bno majority\b",
            r"\bnot unanimous\b"
        ]
    },

    "2/3": {
        "facts": [
            "A majority agreement exists.",
            "An ensemble classification exists.",
            "The models are not unanimous.",
            "The disagreement warrants cautious interpretation."
        ],
        "fallback": (
            "A majority agreement establishes an ensemble "
            "classification, although the models are not unanimous. "
            "The disagreement warrants cautious interpretation."
        ),
        "required_concepts": [
            r"\bmajority\b",
            r"\b(?:disagreement|disagree|not unanimous|dissent)\b",
            r"\b(?:caution|cautious|careful|carefully)\b"
        ],
        "forbidden_concepts": [
            r"\binconclusive\b",
            r"\bno majority\b",
            r"\bno ensemble classification\b",
            r"\ball models agree\b",
            r"\bunanimous agreement\b",
            r"\bcannot establish an ensemble\b"
        ]
    },

    "1/3": {
        "facts": [
            "The models disagree.",
            "No majority agreement exists.",
            "No ensemble classification can be established.",
            "The ensemble result is inconclusive."
        ],
        "fallback": (
            "The models disagree and no majority agreement exists. "
            "Consequently, the ensemble result is inconclusive."
        ),
        "required_concepts": [
            r"\b(?:disagree|disagreement|differ)\b",
            r"\b(?:no majority|lack of majority|absence of a majority)\b",
            r"\binconclusive\b"
        ],
        "forbidden_concepts": [
            r"\bmajority supports\b",
            r"\bmajority predicts\b",
            r"\bmajority agreement exists\b",
            r"\bmajority vote establishes\b",
            r"\bunanimous agreement\b"
        ]
    }
}


FIXED_LIMITATIONS = (
    "Model confidence does not represent clinical certainty. "
    "Grad-CAM visualizes model activation patterns and does not "
    "establish clinically validated anatomical localization. "
    "NetraScan.AI is an academic research prototype and is not "
    "clinically validated for diagnosis. "
    "This test uses synthetic evidence."
)


SYSTEM_PROMPT = """
You are an academic scientific writing assistant.

Your only task is to rewrite supplied authoritative facts
into a concise professional research paragraph.

The supplied facts are already verified.

Do not calculate, infer, classify, diagnose or introduce
new information.

Preserve the meaning of every supplied fact.

Do not introduce model names, disease names, confidence
scores, percentages or numerical values.

Do not introduce medical recommendations.

Return only the requested JSON object.

The paragraph should contain two or three sentences
and approximately 25 to 60 words.
""".strip()


SCHEMA = {
    "type": "object",
    "properties": {
        "interpretation": {
            "type": "string"
        }
    },
    "required": ["interpretation"],
    "additionalProperties": False
}


def validate_narrative(text, agreement):

    normalized = text.lower().strip()

    config = GROUNDING[agreement]

    checks = {
        "Non-empty narrative": bool(normalized),

        "Appropriate word count": (
            15 <= len(normalized.split()) <= 85
        ),

        "No numerical values": (
            re.search(r"\d", normalized) is None
        ),

        "No disease or architecture names": (
            re.search(
                r"\b(?:cataract|vit|swin|mobilevit|normal eye)\b",
                normalized
            ) is None
        ),

        "No clinical assertions": (
            re.search(
                r"\b(?:diagnos\w*|patient|surgery|treatment|pathology)\b",
                normalized
            ) is None
        )
    }

    for index, pattern in enumerate(
        config["required_concepts"],
        start=1
    ):

        checks[f"Required concept {index}"] = bool(
            re.search(pattern, normalized)
        )

    for index, pattern in enumerate(
        config["forbidden_concepts"],
        start=1
    ):

        checks[f"No contradictory pattern {index}"] = (
            re.search(pattern, normalized) is None
        )

    return checks


def generate_narrative(agreement):

    config = GROUNDING[agreement]

    facts = "\n".join(
        f"- {fact}"
        for fact in config["facts"]
    )

    user_prompt = (
        "Rewrite these verified facts without changing "
        "their meaning:\n\n"
        + facts
    )

    payload = {
        "model": MANIFEST["model"],
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "response_format": {
            "type": "json_schema",
            "json_schema": {
                "name": "grounded_narrative",
                "strict": True,
                "schema": SCHEMA
            }
        },
        "temperature": 0.0,
        "seed": 42,
        "max_tokens": 192,
        "stream": False
    }

    started = time.perf_counter()

    response = SESSION.post(
        f"{BASE_URL}/v1/chat/completions",
        json=payload,
        timeout=120
    )

    response.raise_for_status()

    latency = time.perf_counter() - started

    output = response.json()

    choice = output["choices"][0]

    raw_text = choice["message"]["content"]

    try:

        parsed = json.loads(raw_text)

        assert isinstance(parsed, dict)
        assert set(parsed) == {"interpretation"}
        assert isinstance(parsed["interpretation"], str)

        narrative = parsed["interpretation"].strip()

        checks = validate_narrative(
            narrative,
            agreement
        )

    except (
        json.JSONDecodeError,
        AssertionError,
        TypeError,
        KeyError
    ):

        narrative = ""

        checks = {
            "Valid structured output": False
        }

    checks["Generation completed"] = (
        choice.get("finish_reason") == "stop"
    )

    accepted = all(checks.values())

    final_narrative = (
        narrative
        if accepted
        else config["fallback"]
    )

    return {
        "authoritative_facts": config["facts"],
        "raw_generated_output": raw_text,
        "generated_narrative": narrative,
        "validation": checks,
        "llm_accepted": accepted,
        "final_narrative": final_narrative,
        "limitations": FIXED_LIMITATIONS,
        "source": (
            "LLM"
            if accepted
            else "DETERMINISTIC_FALLBACK"
        ),
        "latency_seconds": round(latency, 3),
        "timings": output.get("timings", {})
    }


print("\n1. STARTING LOCAL LLM SERVER")
print("-" * 70)

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:

    assert sock.connect_ex((HOST, PORT)) != 0, (
        f"Port {PORT} is already occupied."
    )

command = [
    str(SERVER_EXE),
    "--model", str(MODEL_PATH),
    "--host", HOST,
    "--port", str(PORT),
    "--ctx-size", "4096",
    "--n-gpu-layers", "99",
    "--parallel", "1",
    "--threads", "8"
]

environment = os.environ.copy()

environment["PATH"] = (
    str(SERVER_EXE.parent)
    + os.pathsep
    + environment.get("PATH", "")
)

server = None
results = {}

with LOG_PATH.open("w", encoding="utf-8") as log_file:

    try:

        started = time.perf_counter()

        server = subprocess.Popen(
            command,
            cwd=str(SERVER_EXE.parent),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=environment
        )

        while True:

            if server.poll() is not None:

                raise RuntimeError(
                    f"Server exited with code {server.returncode}. "
                    f"See {LOG_PATH}"
                )

            try:

                response = SESSION.get(
                    f"{BASE_URL}/health",
                    timeout=2
                )

                if (
                    response.status_code == 200
                    and response.json().get("status") == "ok"
                ):
                    break

            except (
                requests.RequestException,
                ValueError
            ):
                pass

            if time.perf_counter() - started > 180:

                raise TimeoutError(
                    "LLM initialization timed out."
                )

            time.sleep(1)

        print("PASS — Local LLM ready.")

        print(
            f"Startup time: "
            f"{time.perf_counter() - started:.2f} seconds"
        )


        print("\n2. RUNNING GROUNDED GENERATION TESTS")
        print("-" * 70)

        for name, previous in PREVIOUS_RESULTS.items():

            evidence = previous["evidence"]

            agreement = evidence["agreement"]

            print("\n" + "=" * 70)
            print("TEST:", name.upper())
            print("=" * 70)

            print("\nDETERMINISTIC CLASSIFICATION FACTS")

            print(
                "Ensemble:",
                evidence["ensemble_prediction"]
            )

            print("Agreement:", agreement)

            print(
                "Highest confidence:",
                evidence["highest_confidence_model"],
                evidence["highest_confidence_percent"]
            )

            print("\nAUTHORITATIVE FACTS PROVIDED TO LLM")

            for fact in GROUNDING[agreement]["facts"]:
                print("-", fact)

            result = generate_narrative(agreement)

            result["evidence"] = evidence

            print("\nRAW GENERATED OUTPUT")

            print(result["raw_generated_output"])

            print("\nVALIDATION RESULTS")

            for check, passed in result["validation"].items():

                print(
                    "PASS" if passed else "FAIL",
                    "—",
                    check
                )

            print("\nFINAL REPORT INTERPRETATION")

            print(result["final_narrative"])

            print("\nFIXED RESEARCH LIMITATIONS")

            print(result["limitations"])

            print("\nReport source:", result["source"])

            print(
                "Generation latency:",
                result["latency_seconds"],
                "seconds"
            )

            print(
                "Generation tokens/sec:",
                result["timings"].get(
                    "predicted_per_second",
                    "Unavailable"
                )
            )

            results[name] = result

    finally:

        print("\n3. SHUTTING DOWN LOCAL SERVER")
        print("-" * 70)

        if server is not None and server.poll() is None:

            server.terminate()

            try:
                server.wait(timeout=15)

            except subprocess.TimeoutExpired:

                server.kill()
                server.wait()

        print("PASS — Temporary server stopped.")


print("\n4. COMPARATIVE VALIDATION")
print("-" * 70)

previous_accepted = sum(
    result["llm_accepted"]
    for result in PREVIOUS_RESULTS.values()
)

current_accepted = sum(
    result["llm_accepted"]
    for result in results.values()
)

for name, result in results.items():

    print(
        name,
        "=>",
        result["source"]
    )

print(
    f"\nCell 09 accepted: "
    f"{previous_accepted}/{len(PREVIOUS_RESULTS)}"
)

print(
    f"Cell 10 accepted: "
    f"{current_accepted}/{len(results)}"
)

print(
    f"Fallbacks activated: "
    f"{len(results) - current_accepted}"
)

average_latency = sum(
    result["latency_seconds"]
    for result in results.values()
) / len(results)

print(
    f"Average generation latency: "
    f"{average_latency:.3f} seconds"
)

RESULT_PATH.write_text(
    json.dumps(results, indent=4),
    encoding="utf-8"
)

print("\nResults:", RESULT_PATH)
print("Server logs:", LOG_PATH)

print("\n" + "=" * 70)

if current_accepted == len(results):

    print("PASS — ALL GROUNDED GENERATION TESTS ACCEPTED")

else:

    print("REVIEW — SOME OUTPUTS REQUIRED FALLBACK")

print("Original project untouched.")
print("No Python packages installed.")
print("SSD access: NONE")
print("=" * 70)

### 68. Validator Correction and Regression Testing

In [ ]:
# STEP 68 — Validator Correction and Regression Testing
# Cell 11: NetraScan.AI V2 - Validator Correction and Regression Testing

from pathlib import Path
from copy import deepcopy
import json
import re

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

INPUT_PATH = V2_ROOT / "tests" / "grounded_generation.json"
OUTPUT_PATH = V2_ROOT / "tests" / "validator_regression.json"

previous_results = json.loads(
    INPUT_PATH.read_text(encoding="utf-8")
)

print("=" * 70)
print("NETRASCAN.AI V2 — VALIDATOR REGRESSION TESTING")
print("=" * 70)


def validate_narrative_v2(text, agreement):

    normalized = re.sub(
        r"\s+",
        " ",
        text.lower().strip()
    )

    checks = {
        "Non-empty narrative": bool(normalized),

        "Appropriate word count": (
            10 <= len(normalized.split()) <= 85
        ),

        "No numerical values": (
            re.search(r"\d", normalized) is None
        ),

        "No disease or architecture names": (
            re.search(
                r"\b(?:cataract|vit|swin|mobilevit|normal eye)\b",
                normalized
            ) is None
        ),

        "No clinical assertions": (
            re.search(
                r"\b(?:diagnos\w*|patient|surgery|treatment|"
                r"pathology|disease|medication)\b",
                normalized
            ) is None
        )
    }

    if agreement == "3/3":

        checks["Unanimous agreement acknowledged"] = bool(
            re.search(
                r"\b(?:all models|unanimous|unanimity|"
                r"complete agreement)\b",
                normalized
            )
        )

        checks["Correctness limitation acknowledged"] = bool(
            re.search(
                r"\b(?:does not|do not|cannot|"
                r"not necessarily|no guarantee)\b"
                r".{0,100}"
                r"\b(?:correctness|accuracy|correct|accurate|validity)\b",
                normalized
            )
        )

        checks["No contradictory disagreement"] = (
            re.search(
                r"\b(?:disagreement|disagree|inconclusive|"
                r"no majority|not unanimous)\b",
                normalized
            ) is None
        )

    elif agreement == "2/3":

        checks["Majority acknowledged"] = bool(
            re.search(r"\bmajority\b", normalized)
        )

        checks["Disagreement acknowledged"] = bool(
            re.search(
                r"\b(?:disagreement|disagree|"
                r"not unanimous|dissent)\b",
                normalized
            )
        )

        checks["Cautious interpretation acknowledged"] = bool(
            re.search(
                r"\b(?:caution|cautious|careful|carefully)\b",
                normalized
            )
        )

        checks["No contradictory ensemble conclusion"] = (
            re.search(
                r"\b(?:inconclusive|no majority|"
                r"no ensemble classification|"
                r"all models agree|unanimous agreement|"
                r"cannot establish an ensemble)\b",
                normalized
            ) is None
        )

    elif agreement == "1/3":

        checks["Disagreement acknowledged"] = bool(
            re.search(
                r"\b(?:disagree|disagreement|differ)\b",
                normalized
            )
        )

        checks["Absence of majority acknowledged"] = bool(
            re.search(
                r"\b(?:no majority|lack of majority|"
                r"absence of a majority)\b",
                normalized
            )
        )

        checks["Inconclusive result acknowledged"] = bool(
            re.search(
                r"\binconclusive\b",
                normalized
            )
        )

        contradiction_text = re.sub(
            r"\bno majority agreement exists\b",
            "",
            normalized
        )

        checks["No false majority assertion"] = (
            re.search(
                r"\b(?:majority supports|majority predicts|"
                r"majority agreement exists|"
                r"majority vote establishes|"
                r"unanimous agreement|all models agree)\b",
                contradiction_text
            ) is None
        )

    else:

        checks["Valid agreement category"] = False

    checks["No absolute correctness claims"] = (
        re.search(
            r"\b(?:guarantees correctness|"
            r"guarantees accuracy|"
            r"always correct|"
            r"certainly correct|"
            r"proves correctness|"
            r"perfectly accurate)\b",
            normalized
        ) is None
    )

    return checks


print("\n1. REVALIDATING EXISTING LLM OUTPUTS")
print("-" * 70)

updated_results = deepcopy(previous_results)

for name, result in updated_results.items():

    agreement = result["evidence"]["agreement"]
    narrative = result["generated_narrative"]

    checks = validate_narrative_v2(
        narrative,
        agreement
    )

    checks["Generation completed"] = (
        result["validation"].get(
            "Generation completed",
            False
        )
    )

    accepted = all(checks.values())

    result["validation_v2"] = checks
    result["llm_accepted_v2"] = accepted

    if accepted:

        result["final_narrative"] = narrative
        result["source"] = "LLM"

    else:

        result["source"] = "DETERMINISTIC_FALLBACK"

    print("\nTEST:", name.upper())
    print("Narrative:", narrative)

    for check, passed in checks.items():

        print(
            "PASS" if passed else "FAIL",
            "—",
            check
        )

    print(
        "FINAL:",
        "ACCEPTED" if accepted else "REJECTED"
    )


print("\n2. ADVERSARIAL HALLUCINATION TESTS")
print("-" * 70)

ADVERSARIAL_CASES = [
    {
        "name": "False unanimous agreement",
        "agreement": "2/3",
        "text": (
            "All models agree unanimously and the "
            "ensemble classification is certainly correct."
        )
    },
    {
        "name": "False inconclusive majority",
        "agreement": "2/3",
        "text": (
            "A majority exists, but disagreement makes "
            "the ensemble classification inconclusive. "
            "Cautious interpretation is necessary."
        )
    },
    {
        "name": "Invented majority",
        "agreement": "1/3",
        "text": (
            "The models disagree and no majority agreement "
            "exists. Although the result is inconclusive, "
            "the majority supports a classification."
        )
    },
    {
        "name": "Unsupported medical diagnosis",
        "agreement": "3/3",
        "text": (
            "All models agree and the patient has "
            "a confirmed diagnosis requiring treatment."
        )
    },
    {
        "name": "Invented confidence",
        "agreement": "2/3",
        "text": (
            "The majority prediction has 99 percent "
            "confidence despite disagreement and warrants "
            "cautious interpretation."
        )
    },
    {
        "name": "Guaranteed correctness",
        "agreement": "3/3",
        "text": (
            "All models agree, and unanimous agreement "
            "guarantees correctness of the ensemble classification."
        )
    },
    {
        "name": "Invented architecture prediction",
        "agreement": "2/3",
        "text": (
            "The majority supports Swin-Tiny despite "
            "disagreement, warranting cautious interpretation."
        )
    },
    {
        "name": "Empty generated response",
        "agreement": "3/3",
        "text": ""
    },
    {
        "name": "Contradictory unanimous interpretation",
        "agreement": "3/3",
        "text": (
            "All models agree, but disagreement makes "
            "the classification inconclusive."
        )
    },
    {
        "name": "False majority after negated statement",
        "agreement": "1/3",
        "text": (
            "The models disagree and no majority agreement "
            "exists. The result is inconclusive, although "
            "majority agreement exists."
        )
    }
]

adversarial_results = []

for case in ADVERSARIAL_CASES:

    checks = validate_narrative_v2(
        case["text"],
        case["agreement"]
    )

    rejected = not all(checks.values())

    failed_checks = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    print("\nTEST:", case["name"])

    print(
        "PASS — Hallucination rejected"
        if rejected
        else "FAIL — Invalid narrative accepted"
    )

    print("Failed checks:", failed_checks)

    adversarial_results.append({
        **case,
        "validation": checks,
        "correctly_rejected": rejected
    })


print("\n3. FINAL REGRESSION SUMMARY")
print("-" * 70)

old_accepted = sum(
    result["llm_accepted"]
    for result in previous_results.values()
)

new_accepted = sum(
    result["llm_accepted_v2"]
    for result in updated_results.values()
)

hallucinations_rejected = sum(
    result["correctly_rejected"]
    for result in adversarial_results
)

print(
    f"Original validator accepted: "
    f"{old_accepted}/{len(previous_results)}"
)

print(
    f"Corrected validator accepted: "
    f"{new_accepted}/{len(updated_results)}"
)

print(
    f"Adversarial hallucinations rejected: "
    f"{hallucinations_rejected}/{len(ADVERSARIAL_CASES)}"
)

print(
    f"False rejections on reviewed samples: "
    f"{len(updated_results) - new_accepted}"
)

print(
    f"False acceptances on adversarial samples: "
    f"{len(ADVERSARIAL_CASES) - hallucinations_rejected}"
)

report = {
    "original_accepted": old_accepted,
    "corrected_accepted": new_accepted,
    "total_valid_samples": len(updated_results),
    "adversarial_rejected": hallucinations_rejected,
    "total_adversarial_samples": len(ADVERSARIAL_CASES),
    "revalidated_outputs": updated_results,
    "adversarial_results": adversarial_results
}

OUTPUT_PATH.write_text(
    json.dumps(report, indent=4),
    encoding="utf-8"
)

print("\nResults saved:", OUTPUT_PATH)

print("\n" + "=" * 70)

if (
    new_accepted == len(updated_results)
    and hallucinations_rejected == len(ADVERSARIAL_CASES)
):

    print("PASS — ALL VALIDATOR REGRESSION TESTS PASSED")

else:

    print("REVIEW — VALIDATOR REQUIRES FURTHER CORRECTION")

print("LLM server not started.")
print("GPU resources not required.")
print("Original project untouched.")
print("SSD access: NONE")
print("=" * 70)

### 69. Reusable Local GenAI Reporting Module

In [ ]:
# STEP 69 — Reusable Local GenAI Reporting Module
# Cell 12: NetraScan.AI V2 - Reusable Local GenAI Reporting Module

from pathlib import Path
import inspect
import textwrap
import pprint
import ast
import subprocess
import sys
import json

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

MODULE_PATH = V2_ROOT / "src" / "llm_reporter.py"

print("=" * 70)
print("NETRASCAN.AI V2 — GENAI REPORTING MODULE")
print("=" * 70)

print("\n1. VERIFYING PREVIOUS EXPERIMENTS")
print("-" * 70)

regression = json.loads(
    (V2_ROOT / "tests" / "validator_regression.json").read_text(
        encoding="utf-8"
    )
)

assert regression["corrected_accepted"] == regression["total_valid_samples"]

assert regression["adversarial_rejected"] == regression["total_adversarial_samples"]

assert callable(validate_narrative_v2)
assert isinstance(GROUNDING, dict)
assert isinstance(SYSTEM_PROMPT, str)
assert isinstance(SCHEMA, dict)

print("PASS — Validator regression results verified.")
print("PASS — Grounding configuration available.")
print("PASS — Structured generation configuration available.")

validator_source = textwrap.dedent(
    inspect.getsource(validate_narrative_v2)
)

header = '''
from pathlib import Path
from collections import Counter
import subprocess
import requests
import threading
import atexit
import socket
import secrets
import time
import json
import math
import os
import re

APP_ROOT = Path(__file__).resolve().parent.parent

MANIFEST_PATH = APP_ROOT / "llm" / "manifest.json"

MODEL_NAMES = (
    "ViT-B/16",
    "Swin-Tiny",
    "MobileViT-S"
)

CLASS_NAMES = (
    "normal",
    "immature",
    "mature"
)
'''

configuration = (
    "\nGROUNDING = "
    + pprint.pformat(GROUNDING, sort_dicts=False)
    + "\n\nSYSTEM_PROMPT = "
    + repr(SYSTEM_PROMPT)
    + "\n\nSCHEMA = "
    + pprint.pformat(SCHEMA, sort_dicts=False)
    + "\n"
)

module_body = '''

def verify_inference_result(result):

    if not isinstance(result, dict):
        raise ValueError("Invalid inference result.")

    predictions = result.get("predictions")

    if not isinstance(predictions, dict):
        raise ValueError("Missing model predictions.")

    if set(predictions) != set(MODEL_NAMES):
        raise ValueError("Unexpected vision model configuration.")

    votes = []

    for name in MODEL_NAMES:

        details = predictions[name]

        prediction = details["predicted_class"]
        confidence = float(details["confidence"])

        if prediction not in CLASS_NAMES:
            raise ValueError(f"Invalid class: {prediction}")

        if not math.isfinite(confidence) or not 0 <= confidence <= 1:
            raise ValueError(f"Invalid confidence: {name}")

        probabilities = details["probabilities"]

        if set(probabilities) != set(CLASS_NAMES):
            raise ValueError(f"Invalid probabilities: {name}")

        values = {
            key: float(value)
            for key, value in probabilities.items()
        }

        if not all(
            math.isfinite(value) and 0 <= value <= 1
            for value in values.values()
        ):
            raise ValueError(f"Invalid probability values: {name}")

        if abs(sum(values.values()) - 1.0) > 1e-4:
            raise ValueError(f"Probability normalization failed: {name}")

        if abs(values[prediction] - confidence) > 1e-4:
            raise ValueError(f"Confidence mismatch: {name}")

        if max(values.values()) - confidence > 1e-4:
            raise ValueError(f"Prediction is not an argmax: {name}")

        votes.append(prediction)

    majority_class, majority_count = Counter(
        votes
    ).most_common(1)[0]

    agreement = f"{majority_count}/3"

    ensemble = (
        majority_class
        if majority_count >= 2
        else "inconclusive"
    )

    if result.get("agreement") != agreement:
        raise ValueError("Ensemble agreement mismatch.")

    if result.get("ensemble_prediction") != ensemble:
        raise ValueError("Ensemble prediction mismatch.")

    return agreement


class LocalLLMReporter:

    def __init__(
        self,
        gpu_layers=99,
        context_size=4096,
        threads=8,
        startup_timeout=180,
        generation_timeout=120
    ):

        self.manifest = json.loads(
            MANIFEST_PATH.read_text(encoding="utf-8")
        )

        self.server_exe = (
            APP_ROOT / self.manifest["backend_executable"]
        )

        self.model_path = (
            APP_ROOT / self.manifest["model_path"]
        )

        self.gpu_layers = int(gpu_layers)
        self.context_size = int(context_size)
        self.threads = int(threads)

        self.startup_timeout = startup_timeout
        self.generation_timeout = generation_timeout

        self.process = None
        self.log_file = None
        self.base_url = None
        self.api_key = None

        self.lock = threading.RLock()

        self.session = requests.Session()
        self.session.trust_env = False

        self.log_path = (
            APP_ROOT / "llm" / "runtime" / "server.log"
        )

        self.log_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        atexit.register(self.close)


    def _available_port(self):

        with socket.socket(
            socket.AF_INET,
            socket.SOCK_STREAM
        ) as sock:

            sock.bind(("127.0.0.1", 0))

            return sock.getsockname()[1]


    def start(self):

        with self.lock:

            if (
                self.process is not None
                and self.process.poll() is None
                and self.base_url
            ):

                try:

                    response = self.session.get(
                        self.base_url + "/health",
                        timeout=3
                    )

                    if (
                        response.ok
                        and response.json().get("status") == "ok"
                    ):
                        return

                except (requests.RequestException, ValueError):
                    pass

            self.close()

            if not self.server_exe.is_file():
                raise FileNotFoundError(self.server_exe)

            if not self.model_path.is_file():
                raise FileNotFoundError(self.model_path)

            port = self._available_port()

            self.base_url = f"http://127.0.0.1:{port}"
            self.api_key = secrets.token_urlsafe(32)

            self.session.headers.update({
                "Authorization": f"Bearer {self.api_key}"
            })

            command = [
                str(self.server_exe),
                "--model", str(self.model_path),
                "--host", "127.0.0.1",
                "--port", str(port),
                "--ctx-size", str(self.context_size),
                "--n-gpu-layers", str(self.gpu_layers),
                "--parallel", "1",
                "--threads", str(self.threads),
                "--api-key", self.api_key
            ]

            environment = os.environ.copy()

            environment["PATH"] = (
                str(self.server_exe.parent)
                + os.pathsep
                + environment.get("PATH", "")
            )

            self.log_file = self.log_path.open(
                "w",
                encoding="utf-8"
            )

            try:

                self.process = subprocess.Popen(
                    command,
                    cwd=str(self.server_exe.parent),
                    stdout=self.log_file,
                    stderr=subprocess.STDOUT,
                    env=environment,
                    creationflags=getattr(
                        subprocess,
                        "CREATE_NO_WINDOW",
                        0
                    )
                )

                started = time.perf_counter()

                while True:

                    if self.process.poll() is not None:

                        raise RuntimeError(
                            "LLM server exited unexpectedly. "
                            f"See {self.log_path}"
                        )

                    try:

                        response = self.session.get(
                            self.base_url + "/health",
                            timeout=2
                        )

                        if (
                            response.ok
                            and response.json().get("status") == "ok"
                        ):
                            return

                    except (requests.RequestException, ValueError):
                        pass

                    if (
                        time.perf_counter() - started
                        > self.startup_timeout
                    ):
                        raise TimeoutError(
                            "LLM initialization timed out."
                        )

                    time.sleep(0.5)

            except BaseException:

                self.close()
                raise


    def close(self):

        with self.lock:

            if self.process is not None:

                if self.process.poll() is None:

                    self.process.terminate()

                    try:

                        self.process.wait(timeout=10)

                    except subprocess.TimeoutExpired:

                        self.process.kill()
                        self.process.wait()

                self.process = None

            if self.log_file is not None:

                self.log_file.close()
                self.log_file = None

            self.base_url = None
            self.api_key = None

            self.session.headers.pop(
                "Authorization",
                None
            )


    def generate(self, result):

        agreement = verify_inference_result(result)

        config = GROUNDING[agreement]

        started = time.perf_counter()

        output = {
            "interpretation": config["fallback"],
            "source": "DETERMINISTIC_FALLBACK",
            "llm_accepted": False,
            "model": self.manifest["model"],
            "quantization": self.manifest["quantization"],
            "agreement": agreement,
            "validation": {},
            "error": None,
            "generation_seconds": None,
            "total_seconds": None,
            "tokens_per_second": None
        }

        with self.lock:

            try:

                self.start()

                facts = "\\n".join(
                    "- " + fact
                    for fact in config["facts"]
                )

                payload = {
                    "model": self.manifest["model"],
                    "messages": [
                        {
                            "role": "system",
                            "content": SYSTEM_PROMPT
                        },
                        {
                            "role": "user",
                            "content": (
                                "Rewrite these verified facts "
                                "without changing their meaning:"
                                "\\n\\n" + facts
                            )
                        }
                    ],
                    "response_format": {
                        "type": "json_schema",
                        "json_schema": {
                            "name": "grounded_narrative",
                            "strict": True,
                            "schema": SCHEMA
                        }
                    },
                    "temperature": 0.0,
                    "seed": 42,
                    "max_tokens": 192,
                    "stream": False
                }

                generation_started = time.perf_counter()

                response = self.session.post(
                    self.base_url + "/v1/chat/completions",
                    json=payload,
                    timeout=self.generation_timeout
                )

                response.raise_for_status()

                output["generation_seconds"] = round(
                    time.perf_counter() - generation_started,
                    3
                )

                raw = response.json()
                choice = raw["choices"][0]

                parsed = json.loads(
                    choice["message"]["content"]
                )

                if (
                    not isinstance(parsed, dict)
                    or set(parsed) != {"interpretation"}
                    or not isinstance(
                        parsed["interpretation"],
                        str
                    )
                ):

                    raise ValueError(
                        "Invalid structured LLM response."
                    )

                narrative = parsed["interpretation"].strip()

                checks = validate_narrative_v2(
                    narrative,
                    agreement
                )

                checks["Generation completed"] = (
                    choice.get("finish_reason") == "stop"
                )

                output["validation"] = checks

                output["tokens_per_second"] = (
                    raw.get("timings", {}).get(
                        "predicted_per_second"
                    )
                )

                if all(checks.values()):

                    output["interpretation"] = narrative
                    output["source"] = "LLM"
                    output["llm_accepted"] = True

                else:

                    output["error"] = (
                        "Generated narrative failed validation."
                    )

            except Exception as error:

                output["error"] = (
                    f"{type(error).__name__}: {error}"
                )

                self.close()

            finally:

                output["total_seconds"] = round(
                    time.perf_counter() - started,
                    3
                )

        return output


    def __enter__(self):

        return self


    def __exit__(self, exc_type, exc_value, traceback):

        self.close()
'''

print("\n2. BUILDING REUSABLE PYTHON MODULE")
print("-" * 70)

source = (
    header
    + "\n"
    + configuration
    + "\n\n"
    + validator_source
    + "\n\n"
    + module_body
)

ast.parse(source)

if MODULE_PATH.exists():

    backup = V2_ROOT / "tests" / "llm_reporter_previous.py"

    if not backup.exists():

        backup.write_bytes(
            MODULE_PATH.read_bytes()
        )

MODULE_PATH.write_text(
    source,
    encoding="utf-8"
)

print("Module:", MODULE_PATH)
print("Lines:", len(source.splitlines()))
print("Size:", MODULE_PATH.stat().st_size, "bytes")
print("PASS — Python syntax verified.")

print("\n3. RUNNING ISOLATED MODULE TESTS")
print("-" * 70)

test_code = r'''
from pathlib import Path
from copy import deepcopy
from unittest.mock import patch
import json

from src.llm_reporter import (
    LocalLLMReporter,
    validate_narrative_v2,
    verify_inference_result
)

root = Path.cwd()

regression = json.loads(
    (root / "tests" / "validator_regression.json").read_text(
        encoding="utf-8"
    )
)

print("\nVALIDATOR REGRESSION")

for name, item in regression["revalidated_outputs"].items():

    checks = validate_narrative_v2(
        item["generated_narrative"],
        item["evidence"]["agreement"]
    )

    assert all(checks.values()), name

    print("PASS —", name)

for case in regression["adversarial_results"]:

    checks = validate_narrative_v2(
        case["text"],
        case["agreement"]
    )

    assert not all(checks.values()), case["name"]

print("PASS — All adversarial samples rejected.")

benchmark = json.loads(
    (root / "tests" / "llm_benchmark.json").read_text(
        encoding="utf-8"
    )
)

result = {
    "ensemble_prediction": "mature",
    "agreement": "2/3",
    "predictions": {
        "ViT-B/16": {
            "predicted_class": "normal",
            "confidence": 0.4480535387992859,
            "probabilities": {
                "normal": 0.4480535387992859,
                "immature": 0.12131041288375854,
                "mature": 0.43063607811927795
            }
        },
        "Swin-Tiny": {
            "predicted_class": "mature",
            "confidence": 0.4646258056163788,
            "probabilities": {
                "normal": 0.3041735589504242,
                "immature": 0.23120062053203583,
                "mature": 0.4646258056163788
            }
        },
        "MobileViT-S": {
            "predicted_class": "mature",
            "confidence": 0.49312832951545715,
            "probabilities": {
                "normal": 0.07715090364217758,
                "immature": 0.42972075939178467,
                "mature": 0.49312832951545715
            }
        }
    }
}

print("\nINFERENCE CONTRACT VERIFICATION")

assert verify_inference_result(result) == "2/3"

print("PASS — Valid vision inference accepted.")

invalid = deepcopy(result)
invalid["ensemble_prediction"] = "normal"

try:

    verify_inference_result(invalid)

except ValueError:

    print("PASS — Incorrect ensemble prediction rejected.")

else:

    raise AssertionError("Invalid ensemble accepted.")

print("\nLIVE GENAI GENERATION")

with LocalLLMReporter(gpu_layers=99) as reporter:

    generated = reporter.generate(result)

    print(json.dumps(generated, indent=4))

    assert generated["source"] == "LLM", generated["error"]

    assert generated["llm_accepted"]

    print("PASS — Live LLM narrative generated.")

    second = reporter.generate(result)

    assert second["llm_accepted"]

    print("PASS — Warm server reuse.")
    print("Warm response seconds:", second["total_seconds"])

assert reporter.process is None

print("PASS — Server shutdown verified.")

print("\nFORCED BACKEND FAILURE TEST")

with LocalLLMReporter() as reporter:

    with patch.object(
        reporter,
        "start",
        side_effect=RuntimeError("Simulated backend failure")
    ):

        fallback = reporter.generate(result)

    assert fallback["source"] == "DETERMINISTIC_FALLBACK"
    assert not fallback["llm_accepted"]
    assert fallback["interpretation"]

    print(json.dumps(fallback, indent=4))

    print("PASS — Backend failure handled safely.")

print("\nPASS — REUSABLE GENAI MODULE VERIFIED")
'''

process = subprocess.run(
    [
        sys.executable,
        "-u",
        "-c",
        test_code
    ],
    cwd=str(V2_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    timeout=600
)

print(process.stdout)

if process.stderr:

    print("\nDIAGNOSTIC OUTPUT")
    print(process.stderr)

assert process.returncode == 0, (
    "Reusable GenAI module test failed."
)

print("\n" + "=" * 70)
print("PASS — NETRASCAN.AI V2 GENAI MODULE CREATED")
print("Original application source unchanged.")
print("Original V1 project untouched.")
print("SSD access: NONE")
print("=" * 70)

## 9. PDF reporting and integrated application
ReportLab produces research reports using verified predictions, narrative provenance and matching Grad-CAM artifacts. The final UI patch is retained after the original integration because it modifies the generated application source.

### 70. PDF Reporting Environment Setup

In [ ]:
# STEP 70 — PDF Reporting Environment Setup
# Cell 13: NetraScan.AI V2 - PDF Reporting Environment Setup

from pathlib import Path
from importlib import metadata
import importlib.util
import subprocess
import sys
import json
import hashlib

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

REPORTS_DIR = V2_ROOT / "reports"
TESTS_DIR = V2_ROOT / "tests"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
TESTS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("NETRASCAN.AI V2 — PDF REPORTING ENVIRONMENT")
print("=" * 70)

print("\n1. RECORDING EXISTING ENVIRONMENT")
print("-" * 70)

protected_packages = [
    "torch",
    "torchvision",
    "timm",
    "transformers",
    "gradio",
    "numpy",
    "pandas",
    "Pillow",
    "grad-cam",
    "requests",
    "opencv-python",
    "plotly"
]


def package_version(name):

    try:
        return metadata.version(name)

    except metadata.PackageNotFoundError:
        return None


versions_before = {
    name: package_version(name)
    for name in protected_packages
}

for name, version in versions_before.items():
    print(f"{name}: {version}")

print("\nPython executable:", sys.executable)


print("\n2. CHECKING PDF DEPENDENCIES")
print("-" * 70)

pdf_dependencies = [
    "reportlab",
    "charset-normalizer",
    "Pillow"
]

for name in pdf_dependencies:

    version = package_version(name)

    print(
        name,
        "=>",
        version if version else "NOT INSTALLED"
    )

assert package_version("Pillow") is not None
assert package_version("charset-normalizer") is not None


print("\n3. INSTALLING REPORTLAB")
print("-" * 70)

if package_version("reportlab") is None:

    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "reportlab==4.4.4"
    ]

    process = subprocess.run(
        command,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=300
    )

    print(process.stdout)

    if process.stderr:
        print(process.stderr)

    assert process.returncode == 0, (
        "ReportLab installation failed."
    )

else:

    print(
        "ReportLab already installed:",
        package_version("reportlab")
    )

print("PASS — ReportLab available.")


print("\n4. VERIFYING PROTECTED PACKAGES")
print("-" * 70)

versions_after = {
    name: package_version(name)
    for name in protected_packages
}

for name in protected_packages:

    before = versions_before[name]
    after = versions_after[name]

    assert before == after, (
        f"Unexpected dependency change: {name}"
    )

    print(f"PASS — {name}: {after}")

print("\nPASS — All protected package versions unchanged.")


print("\n5. TESTING PDF GENERATION")
print("-" * 70)

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    Image as PDFImage
)

from reportlab.lib import colors
from reportlab.lib.styles import (
    getSampleStyleSheet,
    ParagraphStyle
)

from reportlab.lib.pagesizes import A4
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.units import mm

from PIL import Image, ImageDraw
from io import BytesIO

TEST_PDF = TESTS_DIR / "pdf_environment_test.pdf"

NAVY = colors.HexColor("#0B1729")
CYAN = colors.HexColor("#0891B2")
SLATE = colors.HexColor("#475569")
LIGHT = colors.HexColor("#F1F5F9")

styles = getSampleStyleSheet()

styles.add(
    ParagraphStyle(
        name="NetraScanTitle",
        fontName="Helvetica-Bold",
        fontSize=28,
        leading=34,
        textColor=NAVY,
        alignment=TA_CENTER,
        spaceAfter=10
    )
)

styles.add(
    ParagraphStyle(
        name="NetraScanSubtitle",
        fontName="Helvetica",
        fontSize=11,
        leading=15,
        textColor=SLATE,
        alignment=TA_CENTER,
        spaceAfter=20
    )
)

styles.add(
    ParagraphStyle(
        name="NetraScanHeading",
        fontName="Helvetica-Bold",
        fontSize=14,
        leading=18,
        textColor=CYAN,
        spaceBefore=14,
        spaceAfter=8
    )
)

styles.add(
    ParagraphStyle(
        name="NetraScanBody",
        fontName="Helvetica",
        fontSize=10,
        leading=15,
        textColor=NAVY,
        spaceAfter=8
    )
)

document = SimpleDocTemplate(
    str(TEST_PDF),
    pagesize=A4,
    rightMargin=22 * mm,
    leftMargin=22 * mm,
    topMargin=22 * mm,
    bottomMargin=22 * mm,
    title="NetraScan.AI V2 - PDF Environment Test",
    author="NetraScan.AI"
)

story = []

story.append(
    Paragraph(
        "NetraScan.AI",
        styles["NetraScanTitle"]
    )
)

story.append(
    Paragraph(
        "V2 | PDF Reporting Environment Test",
        styles["NetraScanSubtitle"]
    )
)

story.append(
    Paragraph(
        "Environment Verification",
        styles["NetraScanHeading"]
    )
)

table_data = [
    ["Component", "Status"],
    ["PDF generation", "Operational"],
    ["ReportLab", package_version("reportlab")],
    ["Vision dependencies", "Unchanged"],
    ["Local GenAI backend", "Previously verified"],
    ["Clinical use", "Research prototype only"]
]

table = Table(
    table_data,
    colWidths=[85 * mm, 75 * mm]
)

table.setStyle(
    TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), NAVY),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTNAME", (0, 1), (-1, -1), "Helvetica"),
        ("FONTSIZE", (0, 0), (-1, -1), 10),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 10),
        ("TOPPADDING", (0, 0), (-1, -1), 10),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1),
         [colors.white, LIGHT]),
        ("LINEBELOW", (0, -1), (-1, -1), 1, CYAN)
    ])
)

story.append(table)
story.append(Spacer(1, 12))

story.append(
    Paragraph(
        "Image Embedding Verification",
        styles["NetraScanHeading"]
    )
)

test_image = Image.new(
    "RGB",
    (224, 224),
    (11, 23, 41)
)

draw = ImageDraw.Draw(test_image)

draw.ellipse(
    (25, 25, 199, 199),
    outline=(8, 145, 178),
    width=5
)

draw.ellipse(
    (65, 65, 159, 159),
    fill=(8, 145, 178)
)

draw.ellipse(
    (95, 95, 129, 129),
    fill=(11, 23, 41)
)

image_buffer = BytesIO()

test_image.save(
    image_buffer,
    format="PNG"
)

image_buffer.seek(0)

story.append(
    PDFImage(
        image_buffer,
        width=45 * mm,
        height=45 * mm
    )
)

story.append(Spacer(1, 8))

story.append(
    Paragraph(
        "Synthetic graphic used exclusively to verify "
        "PDF image embedding. This is not a clinical image.",
        styles["NetraScanBody"]
    )
)

story.append(
    Paragraph(
        "Research Disclaimer",
        styles["NetraScanHeading"]
    )
)

story.append(
    Paragraph(
        "NetraScan.AI is an academic research prototype. "
        "Its outputs are not clinically validated and must "
        "not be used as a substitute for professional "
        "medical diagnosis.",
        styles["NetraScanBody"]
    )
)


def page_footer(canvas, doc):

    width, height = A4

    canvas.saveState()

    canvas.setStrokeColor(CYAN)

    canvas.line(
        22 * mm,
        17 * mm,
        width - 22 * mm,
        17 * mm
    )

    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(SLATE)

    canvas.drawString(
        22 * mm,
        12 * mm,
        "NetraScan.AI V2 | Research prototype"
    )

    canvas.drawRightString(
        width - 22 * mm,
        12 * mm,
        f"Page {doc.page}"
    )

    canvas.restoreState()


document.build(
    story,
    onFirstPage=page_footer,
    onLaterPages=page_footer
)

assert TEST_PDF.is_file()

pdf_bytes = TEST_PDF.read_bytes()

assert pdf_bytes.startswith(b"%PDF-")
assert pdf_bytes.rstrip().endswith(b"%%EOF")
assert len(pdf_bytes) > 1000

print("PASS — PDF file generated.")
print("PASS — PDF header verified.")
print("PASS — PDF EOF marker verified.")
print("PASS — Image embedding completed.")

print("PDF:", TEST_PDF)
print("Size:", len(pdf_bytes), "bytes")

pdf_checksum = hashlib.sha256(
    pdf_bytes
).hexdigest()

print("SHA-256:", pdf_checksum)


print("\n6. RECORDING PDF ENVIRONMENT")
print("-" * 70)

environment_record = {
    "python": sys.version,
    "python_executable": sys.executable,
    "reportlab": package_version("reportlab"),
    "protected_packages_before": versions_before,
    "protected_packages_after": versions_after,
    "protected_packages_unchanged": (
        versions_before == versions_after
    ),
    "test_pdf": TEST_PDF.relative_to(V2_ROOT).as_posix(),
    "test_pdf_size_bytes": len(pdf_bytes),
    "test_pdf_sha256": pdf_checksum
}

ENVIRONMENT_PATH = (
    TESTS_DIR / "pdf_environment.json"
)

ENVIRONMENT_PATH.write_text(
    json.dumps(environment_record, indent=4),
    encoding="utf-8"
)

print("Environment record:", ENVIRONMENT_PATH)

print("\n" + "=" * 70)
print("PASS — PDF REPORTING ENVIRONMENT READY")
print("Protected package versions unchanged.")
print("Vision models not loaded.")
print("LLM server not started.")
print("Original application source unchanged.")
print("Original V1 project untouched.")
print("SSD access: NONE")
print("=" * 70)

### 71. Complete PDF Report Generation Module

In [ ]:
# STEP 71 — Complete PDF Report Generation Module
# Cell 14: NetraScan.AI V2 - Complete PDF Report Generation Module

from pathlib import Path
import ast
import subprocess
import sys
import json

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

MODULE_PATH = V2_ROOT / "src" / "pdf_reporter.py"

print("=" * 70)
print("NETRASCAN.AI V2 — PDF REPORTING MODULE")
print("=" * 70)

print("\n1. BUILDING PDF REPORT GENERATOR")
print("-" * 70)

module_source = r'''
from pathlib import Path
from datetime import datetime, timezone
from io import BytesIO
from xml.sax.saxutils import escape
from uuid import uuid4
import math

from PIL import Image as PILImage

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    Image as PDFImage,
    PageBreak,
    KeepTogether
)

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.units import mm

from .llm_reporter import (
    verify_inference_result,
    validate_narrative_v2,
    GROUNDING
)


APP_ROOT = Path(__file__).resolve().parent.parent
REPORTS_DIR = APP_ROOT / "reports"

NAVY = colors.HexColor("#0B1729")
CYAN = colors.HexColor("#0891B2")
BRIGHT_CYAN = colors.HexColor("#22D3EE")
SLATE = colors.HexColor("#475569")
LIGHT = colors.HexColor("#F1F5F9")
BORDER = colors.HexColor("#CBD5E1")
WHITE = colors.white

CLASS_LABELS = {
    "normal": "Normal",
    "immature": "Immature Cataract",
    "mature": "Mature Cataract",
    "inconclusive": "Inconclusive"
}

MODEL_NAMES = [
    "ViT-B/16",
    "Swin-Tiny",
    "MobileViT-S"
]

CLASS_NAMES = [
    "normal",
    "immature",
    "mature"
]

CONTENT_WIDTH = 174 * mm


def create_styles():

    return {
        "brand": ParagraphStyle(
            "Brand",
            fontName="Helvetica-Bold",
            fontSize=27,
            leading=32,
            textColor=WHITE
        ),

        "header": ParagraphStyle(
            "Header",
            fontName="Helvetica",
            fontSize=11,
            leading=16,
            textColor=WHITE
        ),

        "heading": ParagraphStyle(
            "Heading",
            fontName="Helvetica-Bold",
            fontSize=12,
            leading=16,
            textColor=CYAN,
            spaceBefore=12,
            spaceAfter=7
        ),

        "body": ParagraphStyle(
            "Body",
            fontName="Helvetica",
            fontSize=10,
            leading=15,
            textColor=NAVY,
            spaceAfter=6
        ),

        "small": ParagraphStyle(
            "Small",
            fontName="Helvetica",
            fontSize=8.5,
            leading=12,
            textColor=SLATE,
            spaceAfter=4
        ),

        "caption": ParagraphStyle(
            "Caption",
            fontName="Helvetica-Bold",
            fontSize=9,
            leading=12,
            textColor=SLATE,
            alignment=TA_CENTER
        ),

        "metric_label": ParagraphStyle(
            "MetricLabel",
            fontName="Helvetica",
            fontSize=9,
            leading=12,
            textColor=SLATE
        ),

        "metric_value": ParagraphStyle(
            "MetricValue",
            fontName="Helvetica-Bold",
            fontSize=17,
            leading=22,
            textColor=NAVY
        ),

        "table_header": ParagraphStyle(
            "TableHeader",
            fontName="Helvetica-Bold",
            fontSize=8.5,
            leading=11,
            textColor=WHITE
        ),

        "table_body": ParagraphStyle(
            "TableBody",
            fontName="Helvetica",
            fontSize=9,
            leading=12,
            textColor=NAVY
        )
    }


def paragraph(text, style):

    return Paragraph(
        escape(str(text)),
        style
    )


def image_flowable(image, max_width, max_height):

    if not isinstance(image, PILImage.Image):
        raise TypeError("Expected a PIL image.")

    image = image.convert("RGB").copy()

    image.thumbnail((1200, 1200))

    buffer = BytesIO()

    image.save(
        buffer,
        format="PNG"
    )

    buffer.seek(0)

    width, height = image.size

    scale = min(
        max_width / width,
        max_height / height
    )

    return PDFImage(
        buffer,
        width=width * scale,
        height=height * scale
    )


def styled_table(rows, widths, styles):

    formatted = []

    for index, row in enumerate(rows):

        style = (
            styles["table_header"]
            if index == 0
            else styles["table_body"]
        )

        formatted.append([
            paragraph(value, style)
            for value in row
        ])

    table = Table(
        formatted,
        colWidths=widths,
        repeatRows=1,
        hAlign="LEFT"
    )

    table.setStyle(
        TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), NAVY),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1),
             [WHITE, LIGHT]),
            ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
            ("TOPPADDING", (0, 0), (-1, -1), 8),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
            ("LEFTPADDING", (0, 0), (-1, -1), 7),
            ("RIGHTPADDING", (0, 0), (-1, -1), 7),
            ("LINEBELOW", (0, -1), (-1, -1), 0.7, CYAN)
        ])
    )

    return table


def resolve_narrative(result, genai):

    agreement = verify_inference_result(result)

    fallback = GROUNDING[agreement]["fallback"]

    if not isinstance(genai, dict):

        return {
            "text": fallback,
            "source": "DETERMINISTIC_FALLBACK",
            "model": "Not used",
            "quantization": "Not applicable",
            "generation_seconds": None
        }

    narrative = genai.get("interpretation", "")

    accepted = (
        genai.get("source") == "LLM"
        and genai.get("llm_accepted") is True
        and genai.get("agreement") == agreement
        and isinstance(narrative, str)
        and all(
            validate_narrative_v2(
                narrative,
                agreement
            ).values()
        )
    )

    if accepted:

        return {
            "text": narrative,
            "source": "LLM",
            "model": str(
                genai.get("model", "Qwen2.5-3B-Instruct")
            ),
            "quantization": str(
                genai.get("quantization", "Q4_K_M")
            ),
            "generation_seconds": genai.get(
                "generation_seconds"
            )
        }

    return {
        "text": fallback,
        "source": "DETERMINISTIC_FALLBACK",
        "model": "Not used",
        "quantization": "Not applicable",
        "generation_seconds": None
    }


def page_decoration(canvas, doc):

    width, height = A4

    canvas.saveState()

    canvas.setFillColor(NAVY)

    canvas.rect(
        0,
        height - 7 * mm,
        width,
        7 * mm,
        fill=1,
        stroke=0
    )

    canvas.setFillColor(CYAN)

    canvas.rect(
        0,
        height - 7 * mm,
        48 * mm,
        7 * mm,
        fill=1,
        stroke=0
    )

    canvas.setStrokeColor(BORDER)

    canvas.line(
        18 * mm,
        16 * mm,
        width - 18 * mm,
        16 * mm
    )

    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(SLATE)

    canvas.drawString(
        18 * mm,
        11 * mm,
        "NetraScan.AI V2 | Research use only"
    )

    canvas.drawRightString(
        width - 18 * mm,
        11 * mm,
        f"Page {doc.page}"
    )

    canvas.restoreState()


def generate_pdf_report(
    result,
    original_image,
    xai_result=None,
    genai=None,
    image_name="Uploaded image",
    synthetic=False,
    output_dir=None
):

    agreement = verify_inference_result(result)

    if not isinstance(original_image, PILImage.Image):
        raise TypeError("Original image must be a PIL image.")

    narrative = resolve_narrative(
        result,
        genai
    )

    styles = create_styles()

    destination = (
        Path(output_dir)
        if output_dir is not None
        else REPORTS_DIR
    )

    destination.mkdir(
        parents=True,
        exist_ok=True
    )

    now = datetime.now(timezone.utc)

    report_id = (
        "NS-"
        + now.strftime("%Y%m%d")
        + "-"
        + uuid4().hex[:8].upper()
    )

    pdf_path = destination / f"{report_id}.pdf"

    temporary_path = destination / f"{report_id}.tmp"

    document = SimpleDocTemplate(
        str(temporary_path),
        pagesize=A4,
        leftMargin=18 * mm,
        rightMargin=18 * mm,
        topMargin=15 * mm,
        bottomMargin=22 * mm,
        title="NetraScan.AI V2 - AI-Assisted Research Report",
        author="NetraScan.AI",
        subject="Multi-transformer cataract classification research"
    )

    story = []

    header_content = [
        Paragraph(
            'NetraScan.<font color="#22D3EE">AI</font>',
            styles["brand"]
        ),

        paragraph(
            "AI-Assisted Research Report",
            styles["header"]
        ),

        paragraph(
            "MULTI-TRANSFORMER CLASSIFICATION | VERSION 2.0",
            styles["header"]
        )
    ]

    header = Table(
        [[header_content]],
        colWidths=[CONTENT_WIDTH]
    )

    header.setStyle(
        TableStyle([
            ("BACKGROUND", (0, 0), (-1, -1), NAVY),
            ("LEFTPADDING", (0, 0), (-1, -1), 15),
            ("RIGHTPADDING", (0, 0), (-1, -1), 15),
            ("TOPPADDING", (0, 0), (-1, -1), 13),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 13)
        ])
    )

    story.append(header)
    story.append(Spacer(1, 8))

    story.append(
        paragraph(
            f"Report ID: {report_id} | "
            f"Generated: {now.strftime('%Y-%m-%d %H:%M UTC')}",
            styles["small"]
        )
    )

    story.append(
        paragraph(
            f"Input: {str(image_name)[:150]}",
            styles["small"]
        )
    )

    if synthetic:

        story.append(
            paragraph(
                "SYNTHETIC SOFTWARE TEST — NOT A CLINICAL CASE",
                styles["heading"]
            )
        )

    story.append(
        paragraph(
            "01 / Classification Summary",
            styles["heading"]
        )
    )

    ensemble = CLASS_LABELS[
        result["ensemble_prediction"]
    ]

    summary = Table(
        [
            [
                paragraph(
                    "Ensemble classification",
                    styles["metric_label"]
                ),
                paragraph(
                    "Model agreement",
                    styles["metric_label"]
                )
            ],
            [
                paragraph(
                    ensemble,
                    styles["metric_value"]
                ),
                paragraph(
                    agreement,
                    styles["metric_value"]
                )
            ]
        ],
        colWidths=[
            112 * mm,
            62 * mm
        ]
    )

    summary.setStyle(
        TableStyle([
            ("BACKGROUND", (0, 0), (-1, -1), LIGHT),
            ("BOX", (0, 0), (-1, -1), 0.5, BORDER),
            ("LINEBEFORE", (1, 0), (1, -1), 0.5, BORDER),
            ("LEFTPADDING", (0, 0), (-1, -1), 10),
            ("TOPPADDING", (0, 0), (-1, -1), 6),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 6)
        ])
    )

    story.append(summary)

    story.append(
        paragraph(
            "02 / Individual Model Predictions",
            styles["heading"]
        )
    )

    prediction_rows = [
        [
            "Architecture",
            "Prediction",
            "Confidence",
            "Latency"
        ]
    ]

    for name in MODEL_NAMES:

        prediction = result["predictions"][name]

        latency = prediction.get("inference_ms")

        if latency is not None:

            latency = float(latency)

            if not math.isfinite(latency) or latency < 0:
                raise ValueError("Invalid inference latency.")

            latency_text = f"{latency:.2f} ms"

        else:

            latency_text = "Not recorded"

        prediction_rows.append([
            name,
            CLASS_LABELS[prediction["predicted_class"]],
            f'{prediction["confidence"] * 100:.2f}%',
            latency_text
        ])

    story.append(
        styled_table(
            prediction_rows,
            [44 * mm, 54 * mm, 37 * mm, 39 * mm],
            styles
        )
    )

    story.append(
        paragraph(
            "03 / Class Probability Distribution",
            styles["heading"]
        )
    )

    probability_rows = [
        [
            "Architecture",
            "Normal",
            "Immature",
            "Mature"
        ]
    ]

    for name in MODEL_NAMES:

        probabilities = result[
            "predictions"
        ][name]["probabilities"]

        probability_rows.append([
            name,
            f'{probabilities["normal"] * 100:.2f}%',
            f'{probabilities["immature"] * 100:.2f}%',
            f'{probabilities["mature"] * 100:.2f}%'
        ])

    story.append(
        styled_table(
            probability_rows,
            [48 * mm, 42 * mm, 42 * mm, 42 * mm],
            styles
        )
    )

    story.append(
        paragraph(
            "04 / Research Interpretation",
            styles["heading"]
        )
    )

    if narrative["source"] == "LLM":

        provenance = (
            f'Local GenAI: {narrative["model"]} | '
            f'{narrative["quantization"]}'
        )

    else:

        provenance = (
            "Deterministic fallback — "
            "no LLM narrative used"
        )

    story.append(
        paragraph(
            provenance,
            styles["small"]
        )
    )

    narrative_box = Table(
        [[
            paragraph(
                narrative["text"],
                styles["body"]
            )
        ]],
        colWidths=[CONTENT_WIDTH]
    )

    narrative_box.setStyle(
        TableStyle([
            ("BACKGROUND", (0, 0), (-1, -1), LIGHT),
            ("LINEBEFORE", (0, 0), (0, 0), 3, CYAN),
            ("LEFTPADDING", (0, 0), (-1, -1), 12),
            ("RIGHTPADDING", (0, 0), (-1, -1), 12),
            ("TOPPADDING", (0, 0), (-1, -1), 10),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 10)
        ])
    )

    story.append(narrative_box)

    generation_seconds = narrative["generation_seconds"]

    if isinstance(generation_seconds, (int, float)):

        if math.isfinite(generation_seconds) and generation_seconds >= 0:

            story.append(
                paragraph(
                    f"LLM generation latency: "
                    f"{generation_seconds:.3f} seconds",
                    styles["small"]
                )
            )

    story.append(
        paragraph(
            "Research prototype only. Model predictions and "
            "confidence scores do not establish a clinical diagnosis.",
            styles["small"]
        )
    )

    story.append(PageBreak())

    story.append(
        paragraph(
            "05 / Explainable AI",
            styles["heading"]
        )
    )

    story.append(
        paragraph(
            "Swin-Tiny | Gradient-weighted Class Activation Mapping",
            styles["body"]
        )
    )

    xai_available = False
    target_label = None

    if isinstance(xai_result, dict):

        overlay = xai_result.get("overlay")
        heatmap = xai_result.get("heatmap")
        class_index = xai_result.get("class_index")

        if (
            isinstance(overlay, PILImage.Image)
            and isinstance(heatmap, PILImage.Image)
            and isinstance(class_index, int)
            and 0 <= class_index < len(CLASS_NAMES)
        ):

            target_class = CLASS_NAMES[class_index]

            swin_prediction = result[
                "predictions"
            ]["Swin-Tiny"]["predicted_class"]

            if target_class == swin_prediction:

                xai_available = True

                target_label = CLASS_LABELS[
                    target_class
                ]

    if xai_available:

        story.append(
            paragraph(
                f"Grad-CAM target class: {target_label}",
                styles["body"]
            )
        )

        image_table = Table(
            [
                [
                    image_flowable(
                        original_image,
                        51 * mm,
                        51 * mm
                    ),
                    image_flowable(
                        overlay,
                        51 * mm,
                        51 * mm
                    ),
                    image_flowable(
                        heatmap,
                        51 * mm,
                        51 * mm
                    )
                ],
                [
                    paragraph(
                        "Original image",
                        styles["caption"]
                    ),
                    paragraph(
                        "Grad-CAM overlay",
                        styles["caption"]
                    ),
                    paragraph(
                        "Activation heatmap",
                        styles["caption"]
                    )
                ]
            ],
            colWidths=[58 * mm] * 3
        )

        image_table.setStyle(
            TableStyle([
                ("ALIGN", (0, 0), (-1, -1), "CENTER"),
                ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
                ("TOPPADDING", (0, 0), (-1, -1), 8),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
                ("BACKGROUND", (0, 0), (-1, -1), LIGHT)
            ])
        )

        story.append(image_table)

        if synthetic:

            story.append(
                paragraph(
                    "Synthetic test visualizations. These images "
                    "do not represent actual Grad-CAM inference.",
                    styles["small"]
                )
            )

        else:

            story.append(
                paragraph(
                    "The heatmap represents relative activation "
                    "patterns associated with the selected "
                    "Swin-Tiny class. Warmer colors generally "
                    "indicate stronger relative activation.",
                    styles["body"]
                )
            )

        story.append(
            paragraph(
                "Grad-CAM explains the Swin-Tiny prediction, "
                "not the complete ensemble decision. It does "
                "not establish clinically validated anatomical "
                "localization or disease severity.",
                styles["body"]
            )
        )

    else:

        story.append(
            image_flowable(
                original_image,
                75 * mm,
                75 * mm
            )
        )

        story.append(
            paragraph(
                "Grad-CAM is unavailable or its target class "
                "could not be verified. No explainability "
                "conclusions are presented.",
                styles["body"]
            )
        )

    story.append(
        paragraph(
            "06 / Methodology",
            styles["heading"]
        )
    )

    methodology_rows = [
        ["Component", "Method"],
        ["Vision Transformer", "ViT-B/16"],
        ["Hierarchical Transformer", "Swin-Tiny"],
        ["Lightweight Transformer", "MobileViT-S"],
        ["Decision fusion", "Majority voting"],
        ["Explainability", "Swin-Tiny Grad-CAM"],
        ["Narrative generation",
         "Deterministic semantic grounding with local LLM"]
    ]

    story.append(
        styled_table(
            methodology_rows,
            [62 * mm, 112 * mm],
            styles
        )
    )

    story.append(
        paragraph(
            "07 / Research Limitations",
            styles["heading"]
        )
    )

    limitations = [
        "Model confidence is not clinical certainty.",
        "Agreement between models does not guarantee correctness.",
        "Grad-CAM activation maps are not validated pathology maps.",
        "The language model does not independently analyze the image.",
        "Generated narratives undergo heuristic validation; "
        "hallucination resistance is not guaranteed.",
        "NetraScan.AI is not clinically validated for diagnosis."
    ]

    if synthetic:

        limitations.append(
            "This report uses synthetic software-test evidence."
        )

    for item in limitations:

        story.append(
            paragraph(
                "- " + item,
                styles["small"]
            )
        )

    story.append(
        paragraph(
            "This document is intended exclusively for academic "
            "research and software demonstration. It must not be "
            "used to make medical or treatment decisions.",
            styles["body"]
        )
    )

    try:

        document.build(
            story,
            onFirstPage=page_decoration,
            onLaterPages=page_decoration
        )

        pdf_bytes = temporary_path.read_bytes()

        if not pdf_bytes.startswith(b"%PDF-"):
            raise RuntimeError("Invalid PDF header.")

        if not pdf_bytes.rstrip().endswith(b"%%EOF"):
            raise RuntimeError("Invalid PDF EOF marker.")

        temporary_path.replace(pdf_path)

    finally:

        if temporary_path.exists():
            temporary_path.unlink()

    return {
        "path": str(pdf_path),
        "report_id": report_id,
        "ensemble_prediction": ensemble,
        "agreement": agreement,
        "interpretation": narrative["text"],
        "report_source": narrative["source"],
        "xai_available": xai_available,
        "xai_target": target_label,
        "synthetic": bool(synthetic),
        "size_bytes": pdf_path.stat().st_size
    }
'''

ast.parse(module_source)

if MODULE_PATH.exists():

    backup = V2_ROOT / "tests" / "pdf_reporter_previous.py"

    if not backup.exists():
        backup.write_bytes(MODULE_PATH.read_bytes())

MODULE_PATH.write_text(
    module_source,
    encoding="utf-8"
)

print("Module:", MODULE_PATH)
print("Lines:", len(module_source.splitlines()))
print("Size:", MODULE_PATH.stat().st_size, "bytes")
print("PASS — Python syntax verified.")


print("\n2. RUNNING ISOLATED PDF MODULE TESTS")
print("-" * 70)

test_code = r'''
from pathlib import Path
from copy import deepcopy
from PIL import Image, ImageDraw
import hashlib
import json

from src.llm_reporter import LocalLLMReporter
from src.pdf_reporter import generate_pdf_report

ROOT = Path.cwd()
TESTS_DIR = ROOT / "tests"

result = {
    "ensemble_prediction": "mature",
    "agreement": "2/3",
    "predictions": {
        "ViT-B/16": {
            "predicted_class": "normal",
            "confidence": 0.4481,
            "probabilities": {
                "normal": 0.4481,
                "immature": 0.1213,
                "mature": 0.4306
            },
            "inference_ms": 1427.04
        },
        "Swin-Tiny": {
            "predicted_class": "mature",
            "confidence": 0.4646,
            "probabilities": {
                "normal": 0.3042,
                "immature": 0.2312,
                "mature": 0.4646
            },
            "inference_ms": 457.14
        },
        "MobileViT-S": {
            "predicted_class": "mature",
            "confidence": 0.4931,
            "probabilities": {
                "normal": 0.0772,
                "immature": 0.4297,
                "mature": 0.4931
            },
            "inference_ms": 298.11
        }
    }
}

original = Image.new(
    "RGB",
    (224, 224),
    (11, 23, 41)
)

draw = ImageDraw.Draw(original)

draw.ellipse(
    (20, 65, 204, 159),
    fill=(220, 225, 230)
)

draw.ellipse(
    (72, 72, 152, 152),
    fill=(65, 105, 130)
)

draw.ellipse(
    (92, 92, 132, 132),
    fill=(25, 30, 40)
)

heatmap = Image.new(
    "RGB",
    (224, 224),
    (0, 0, 130)
)

draw = ImageDraw.Draw(heatmap)

for radius in range(90, 0, -1):

    intensity = 1 - radius / 90

    color = (
        int(255 * intensity),
        int(180 * (1 - intensity)),
        int(130 * (1 - intensity))
    )

    draw.ellipse(
        (
            112 - radius,
            112 - radius,
            112 + radius,
            112 + radius
        ),
        fill=color
    )

overlay = Image.blend(
    original,
    heatmap,
    alpha=0.45
)

xai = {
    "overlay": overlay,
    "heatmap": heatmap,
    "class_index": 2
}

print("\nTEST 1 — LIVE GENAI PDF REPORT")

with LocalLLMReporter(gpu_layers=99) as reporter:

    genai = reporter.generate(result)

assert genai["source"] == "LLM", genai["error"]

report = generate_pdf_report(
    result=result,
    original_image=original,
    xai_result=xai,
    genai=genai,
    image_name="Synthetic software-test image",
    synthetic=True,
    output_dir=TESTS_DIR
)

assert Path(report["path"]).is_file()
assert report["report_source"] == "LLM"
assert report["xai_available"] is True
assert report["xai_target"] == "Mature Cataract"

print(json.dumps(report, indent=4))

print("PASS — Live GenAI PDF generated.")


print("\nTEST 2 — FALLBACK PDF WITHOUT GRAD-CAM")

fallback = generate_pdf_report(
    result=result,
    original_image=original,
    xai_result=None,
    genai=None,
    image_name="Synthetic fallback test",
    synthetic=True,
    output_dir=TESTS_DIR
)

assert Path(fallback["path"]).is_file()
assert fallback["report_source"] == "DETERMINISTIC_FALLBACK"
assert fallback["xai_available"] is False

print(json.dumps(fallback, indent=4))

print("PASS — Fallback PDF generated.")


print("\nTEST 3 — INVALID ENSEMBLE REJECTION")

invalid = deepcopy(result)
invalid["ensemble_prediction"] = "normal"

try:

    generate_pdf_report(
        result=invalid,
        original_image=original,
        synthetic=True,
        output_dir=TESTS_DIR
    )

except ValueError:

    print("PASS — Invalid ensemble rejected.")

else:

    raise AssertionError("Invalid ensemble was accepted.")


print("\nTEST 4 — GRAD-CAM TARGET MISMATCH")

invalid_xai = dict(xai)
invalid_xai["class_index"] = 0

mismatch = generate_pdf_report(
    result=result,
    original_image=original,
    xai_result=invalid_xai,
    genai=genai,
    synthetic=True,
    output_dir=TESTS_DIR
)

assert mismatch["xai_available"] is False

print("PASS — Mismatched Grad-CAM target suppressed.")


print("\nTEST 5 — PDF INTEGRITY")

records = {}

for name, item in [
    ("genai_report", report),
    ("fallback_report", fallback),
    ("xai_mismatch_report", mismatch)
]:

    path = Path(item["path"])
    data = path.read_bytes()

    assert data.startswith(b"%PDF-")
    assert data.rstrip().endswith(b"%%EOF")
    assert len(data) > 1000

    records[name] = {
        **item,
        "sha256": hashlib.sha256(data).hexdigest()
    }

    print("PASS —", name)
    print("File:", path)
    print("Size:", len(data), "bytes")

results_path = TESTS_DIR / "pdf_reporter_tests.json"

results_path.write_text(
    json.dumps(records, indent=4),
    encoding="utf-8"
)

print("\nResults:", results_path)
print("\nPASS — ALL PDF MODULE TESTS COMPLETED")
'''

process = subprocess.run(
    [
        sys.executable,
        "-u",
        "-c",
        test_code
    ],
    cwd=str(V2_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    timeout=600
)

print(process.stdout)

if process.stderr:

    print("\nDIAGNOSTIC OUTPUT")
    print(process.stderr)

assert process.returncode == 0, (
    "PDF reporting module test failed."
)


print("\n3. VERIFYING GENERATED ARTIFACTS")
print("-" * 70)

test_results = json.loads(
    (
        V2_ROOT
        / "tests"
        / "pdf_reporter_tests.json"
    ).read_text(encoding="utf-8")
)

for name, record in test_results.items():

    path = Path(record["path"])

    assert path.is_file()

    print(name)
    print("Path:", path)
    print("Size:", path.stat().st_size, "bytes")
    print("Report source:", record["report_source"])
    print()


print("\n" + "=" * 70)
print("PASS — NETRASCAN.AI V2 PDF MODULE CREATED")
print("Live GenAI reporting verified.")
print("Deterministic fallback verified.")
print("Grad-CAM target verification completed.")
print("Original application source unchanged.")
print("Original V1 project untouched.")
print("SSD access: NONE")
print("=" * 70)

### 72. V2 Integration Preflight

In [ ]:
# STEP 72 — V2 Integration Preflight
# Cell 15 — V2 Integration Preflight

from pathlib import Path
import ast
import hashlib

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

print("NETRASCAN.AI V2 — INTEGRATION PREFLIGHT")
print("=" * 70)

required_files = [
    "app.py",
    "src/inference.py",
    "src/explainability.py",
    "src/llm_reporter.py",
    "src/pdf_reporter.py",
    "llm/manifest.json"
]

print("\n1. VERIFYING APPLICATION COMPONENTS")
print("-" * 70)

for relative_path in required_files:
    path = V2_ROOT / relative_path
    assert path.is_file(), f"Missing: {path}"
    print(f"PASS — {relative_path}")

print("\n2. INSPECTING REPORTING MODULE INTERFACES")
print("-" * 70)

for filename in ["llm_reporter.py", "pdf_reporter.py"]:
    path = V2_ROOT / "src" / filename
    tree = ast.parse(path.read_text(encoding="utf-8"))

    print(f"\nMODULE: {filename}")

    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            print(f"\nClass: {node.name}")

            for method in node.body:
                if isinstance(method, ast.FunctionDef):
                    if not method.name.startswith("_") or method.name == "__init__":
                        print(
                            f"    {method.name}"
                            f"({ast.unparse(method.args)})"
                        )

        elif isinstance(node, ast.FunctionDef):
            if not node.name.startswith("_"):
                print(
                    f"Function: {node.name}"
                    f"({ast.unparse(node.args)})"
                )

print("\n3. CHECKING UI BACKGROUND")
print("-" * 70)

assets = V2_ROOT / "assets"

images = [
    path for path in assets.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
]

if images:
    from PIL import Image

    for path in images:
        with Image.open(path) as img:
            print(f"Image: {path.name}")
            print(f"Resolution: {img.width} x {img.height}")
            print(f"Path: {path}")
            print(f"Size: {path.stat().st_size / 1024**2:.2f} MiB")
else:
    print("WARNING — No background image found.")

print("\n4. VERIFYING GRADIO INTERFACE")
print("-" * 70)

import gradio as gr
import inspect

print("Gradio version:", gr.__version__)
print("Blocks:", inspect.signature(gr.Blocks))
print("Launch:", inspect.signature(gr.Blocks.launch))

print("\n5. RECORDING ORIGINAL APPLICATION HASH")
print("-" * 70)

app_path = V2_ROOT / "app.py"

print("SHA-256:", hashlib.sha256(
    app_path.read_bytes()
).hexdigest())

print("\n" + "=" * 70)
print("PASS — INTEGRATION PREFLIGHT COMPLETED")
print("Application source unchanged.")
print("Vision models not loaded.")
print("LLM server not started.")
print("Original V1 project untouched.")
print("SSD access: NONE")
print("=" * 70)

### 73. Complete V2 Application Integration

In [ ]:
# STEP 73 — Complete V2 Application Integration
# Cell 16 — Complete V2 Application Integration

from pathlib import Path
import ast
import shutil
import hashlib

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

APP_PATH = V2_ROOT / "app.py"
BACKUP_PATH = V2_ROOT / "tests" / "app_v1_backup.py"

print("NETRASCAN.AI V2 — APPLICATION INTEGRATION")
print("=" * 70)

if not BACKUP_PATH.exists():
    shutil.copy2(APP_PATH, BACKUP_PATH)

print("PASS — Original application backed up.")

APP_SOURCE = r'''
from pathlib import Path
import atexit
import base64
import mimetypes
import threading
import hashlib
import html

import gradio as gr
import pandas as pd
import plotly.graph_objects as go

from src.inference import NetraScanInference
from src.explainability import NetraScanXAI
from src.llm_reporter import LocalLLMReporter
from src.pdf_reporter import generate_pdf_report

APP_ROOT = Path(__file__).resolve().parent
REPORTS_DIR = APP_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

CLASS_LABELS = {
    "normal": "Normal",
    "immature": "Immature Cataract",
    "mature": "Mature Cataract",
    "inconclusive": "Inconclusive"
}

MODEL_NAMES = [
    "ViT-B/16",
    "Swin-Tiny",
    "MobileViT-S"
]

MODEL_COLORS = [
    "#22d3ee",
    "#818cf8",
    "#34d399"
]

engine = None
xai_engine = None
reporter = LocalLLMReporter()
pipeline_lock = threading.RLock()

atexit.register(reporter.close)


def initialize():
    global engine, xai_engine

    if engine is None:
        engine = NetraScanInference()

    if xai_engine is None:
        xai_engine = NetraScanXAI(engine)

    return engine, xai_engine


def image_fingerprint(image):
    image = image.convert("RGB")

    digest = hashlib.sha256()
    digest.update(str(image.size).encode())
    digest.update(image.tobytes())

    return digest.hexdigest()


def create_probability_chart(predictions):
    fig = go.Figure()

    for model, color in zip(MODEL_NAMES, MODEL_COLORS):
        probabilities = predictions[model]["probabilities"]

        fig.add_trace(
            go.Bar(
                name=model,
                x=["Normal", "Immature", "Mature"],
                y=[
                    probabilities["normal"] * 100,
                    probabilities["immature"] * 100,
                    probabilities["mature"] * 100
                ],
                marker_color=color,
                hovertemplate="%{y:.2f}%<extra></extra>"
            )
        )

    fig.update_layout(
        template="plotly_dark",
        barmode="group",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color="#e2e8f0"),
        margin=dict(l=45, r=20, t=25, b=45),
        height=340,
        legend=dict(
            orientation="h",
            y=1.12,
            x=0
        ),
        yaxis=dict(
            title="Probability (%)",
            range=[0, 105],
            gridcolor="rgba(148,163,184,0.15)"
        )
    )

    return fig


def run_analysis(image):
    if image is None:
        raise gr.Error("Upload or capture an image first.")

    with pipeline_lock:
        model_engine, explanation_engine = initialize()

        original = image.convert("RGB").copy()
        result = model_engine.predict(original)

        ensemble = CLASS_LABELS[result["ensemble_prediction"]]
        agreement = result["agreement"]

        summary = f"""
### Ensemble classification

# {ensemble}

**Model agreement:** {agreement}

Research classification only — not a clinical diagnosis.
"""

        rows = []

        for name in MODEL_NAMES:
            prediction = result["predictions"][name]

            rows.append([
                name,
                CLASS_LABELS[prediction["predicted_class"]],
                round(prediction["confidence"] * 100, 2),
                prediction["inference_ms"]
            ])

        table = pd.DataFrame(
            rows,
            columns=[
                "Architecture",
                "Prediction",
                "Confidence (%)",
                "Latency (ms)"
            ]
        )

        chart = create_probability_chart(
            result["predictions"]
        )

        xai_result = None
        overlay = None
        heatmap = None

        try:
            candidate = explanation_engine.generate(original)

            expected_class = result["predictions"][
                "Swin-Tiny"
            ]["predicted_class"]

            expected_index = [
                "normal",
                "immature",
                "mature"
            ].index(expected_class)

            if int(candidate["class_index"]) != expected_index:
                raise ValueError("Grad-CAM target mismatch.")

            xai_result = candidate
            overlay = candidate["overlay"]
            heatmap = candidate["heatmap"]

            xai_status = (
                f"**Grad-CAM target:** "
                f"{CLASS_LABELS[expected_class]}\n\n"
                "Explains the Swin-Tiny prediction, "
                "not the complete ensemble decision."
            )

        except Exception as exc:
            print("Grad-CAM unavailable:", exc)

            xai_status = (
                "Grad-CAM unavailable or target verification "
                "failed. No explainability conclusions presented."
            )

        state = {
            "result": result,
            "original_image": original,
            "fingerprint": image_fingerprint(original),
            "xai_result": xai_result
        }

        return (
            original,
            summary,
            table,
            chart,
            overlay,
            heatmap,
            xai_status,
            state,
            "Analysis complete. Generate the GenAI report below.",
            None
        )


def create_report(state, current_image):
    if state is None or current_image is None:
        raise gr.Error("Complete image analysis first.")

    if image_fingerprint(current_image) != state["fingerprint"]:
        raise gr.Error("Image changed. Run analysis again.")

    with pipeline_lock:
        result = state["result"]

        genai = reporter.generate(result)

        report = generate_pdf_report(
            result=result,
            original_image=state["original_image"],
            xai_result=state["xai_result"],
            genai=genai,
            image_name="Uploaded or captured image",
            synthetic=False,
            output_dir=REPORTS_DIR
        )

        source = report["report_source"]

        if source == "LLM":
            source_label = (
                "Local GenAI — Qwen2.5-3B-Instruct"
            )
        else:
            source_label = "Deterministic fallback"

        interpretation = html.escape(
            report["interpretation"]
        )

        preview = f"""
### Research interpretation

**Generation source:** {source_label}

{interpretation}

**Report ID:** `{report["report_id"]}`

**Grad-CAM included:** {
    "Yes" if report["xai_available"] else "No"
}

---

Academic research prototype only. Model confidence
does not represent clinical certainty.
"""

        return preview, report["path"]


def reset_analysis():
    return (
        None,
        "### Awaiting image analysis",
        None,
        None,
        None,
        None,
        "Grad-CAM will appear after analysis.",
        None,
        "Generate a report after completing analysis.",
        None
    )


def clear_all():
    return (None,) + reset_analysis()


def shutdown_backend():
    with pipeline_lock:
        reporter.close()

    return "Local GenAI backend stopped. It can restart on demand."


BACKGROUND = None

for filename in [
    "background.png",
    "background.jpg",
    "background.jpeg",
    "background.webp"
]:
    candidate = APP_ROOT / "assets" / filename

    if candidate.is_file():
        mime = mimetypes.guess_type(candidate.name)[0]
        encoded = base64.b64encode(
            candidate.read_bytes()
        ).decode("ascii")

        BACKGROUND = f"data:{mime};base64,{encoded}"
        break


CUSTOM_CSS = """
body, .gradio-container {
    background-color: #050c19 !important;
    color: #e2e8f0 !important;
}

.gradio-container {
    max-width: 1500px !important;
    margin: auto !important;
    min-height: 100vh;
    background-image:
        radial-gradient(
            ellipse at 10% 10%,
            rgba(8,145,178,0.20),
            transparent 45%
        ),
        radial-gradient(
            ellipse at 90% 70%,
            rgba(79,70,229,0.15),
            transparent 45%
        ) !important;
    background-attachment: fixed !important;
}

#hero {
    padding: 30px;
    border: 1px solid rgba(34,211,238,0.30);
    border-radius: 18px;
    background: linear-gradient(
        110deg,
        rgba(7,18,35,0.94),
        rgba(8,47,73,0.78)
    );
    margin-bottom: 16px;
}

#hero h1 {
    font-size: clamp(32px, 5vw, 54px);
    letter-spacing: -1.8px;
    color: #f8fafc;
    margin: 0;
}

#hero h1 span {
    color: #22d3ee;
}

#hero p {
    color: #94a3b8;
    margin: 8px 0;
}

.hero-tag {
    display: inline-block;
    color: #67e8f9;
    border: 1px solid rgba(34,211,238,0.35);
    border-radius: 20px;
    padding: 5px 12px;
    margin: 10px 6px 0 0;
    font-size: 12px;
}

.glass-panel {
    background: rgba(10,22,40,0.88) !important;
    border: 1px solid rgba(34,211,238,0.18) !important;
    border-radius: 16px !important;
    padding: 16px !important;
    backdrop-filter: blur(14px);
}

#ensemble-panel {
    background: rgba(8,47,73,0.40) !important;
    border: 1px solid rgba(34,211,238,0.30) !important;
    border-radius: 12px !important;
    padding: 16px !important;
}

#ensemble-panel h1 {
    color: #67e8f9 !important;
}

#analyze-button, #report-button {
    background: linear-gradient(
        110deg,
        #0891b2,
        #2563eb
    ) !important;
    color: white !important;
    border: none !important;
    font-weight: 700 !important;
    min-height: 46px;
}

#research-footer {
    text-align: center;
    padding: 20px;
    color: #94a3b8;
    font-size: 12px;
}

footer {
    opacity: 0.6;
}
"""

if BACKGROUND:
    CUSTOM_CSS += f"""
    .gradio-container {{
        background-image:
            linear-gradient(
                rgba(5,12,25,0.72),
                rgba(5,12,25,0.85)
            ),
            url("{BACKGROUND}") !important;
        background-size: cover !important;
        background-position: center !important;
    }}
    """


THEME = gr.themes.Soft(
    primary_hue="cyan",
    secondary_hue="blue",
    neutral_hue="slate"
).set(
    body_background_fill="#050c19",
    body_background_fill_dark="#050c19",
    body_text_color="#e2e8f0",
    body_text_color_dark="#e2e8f0",
    block_background_fill="#0b1628",
    block_background_fill_dark="#0b1628",
    block_label_text_color="#cbd5e1",
    block_label_text_color_dark="#cbd5e1",
    block_title_text_color="#e2e8f0",
    block_title_text_color_dark="#e2e8f0",
    input_background_fill="#101e32",
    input_background_fill_dark="#101e32"
)


with gr.Blocks(
    title="NetraScan.AI V2",
    analytics_enabled=False
) as demo:

    prediction_state = gr.State(None)

    gr.HTML("""
    <div id="hero">
        <h1>NetraScan<span>.AI</span></h1>
        <p>Three Transformers. One Vision.</p>
        <p>
            Multi-transformer cataract classification,
            explainable AI and local generative intelligence.
        </p>
        <span class="hero-tag">VERSION 2.0</span>
        <span class="hero-tag">LOCAL GENAI</span>
        <span class="hero-tag">RESEARCH PROTOTYPE</span>
    </div>
    """)

    with gr.Row():

        with gr.Column(
            scale=1,
            elem_classes=["glass-panel"]
        ):
            gr.Markdown("## 01 — Image Acquisition")

            image_input = gr.Image(
                sources=["upload", "webcam"],
                type="pil",
                label="Upload or capture an eye image",
                height=300
            )

            analyze_button = gr.Button(
                "Run Transformer Analysis",
                variant="primary",
                elem_id="analyze-button"
            )

            clear_button = gr.Button("Clear Analysis")

            image_preview = gr.Image(
                label="Analyzed input",
                interactive=False,
                height=230
            )

        with gr.Column(
            scale=2,
            elem_classes=["glass-panel"]
        ):
            gr.Markdown("## 02 — Transformer Intelligence")

            ensemble_output = gr.Markdown(
                "### Awaiting image analysis",
                elem_id="ensemble-panel"
            )

            prediction_table = gr.Dataframe(
                headers=[
                    "Architecture",
                    "Prediction",
                    "Confidence (%)",
                    "Latency (ms)"
                ],
                interactive=False
            )

            probability_chart = gr.Plot(
                label="Class probability distribution"
            )

    with gr.Column(elem_classes=["glass-panel"]):

        gr.Markdown("## 03 — Explainable AI")

        with gr.Row():

            overlay_output = gr.Image(
                label="Swin-Tiny Grad-CAM overlay",
                interactive=False,
                height=300
            )

            heatmap_output = gr.Image(
                label="Activation heatmap",
                interactive=False,
                height=300
            )

        xai_message = gr.Markdown(
            "Grad-CAM will appear after analysis."
        )

    with gr.Column(elem_classes=["glass-panel"]):

        gr.Markdown("## 04 — Generative AI Research Report")

        gr.Markdown(
            "**Qwen2.5-3B-Instruct · Q4_K_M · Local inference**"
        )

        report_button = gr.Button(
            "Generate AI Research Report",
            variant="primary",
            elem_id="report-button"
        )

        report_preview = gr.Markdown(
            "Generate a report after completing analysis."
        )

        report_download = gr.File(
            label="Download PDF research report",
            interactive=False
        )

        stop_button = gr.Button(
            "Release Local LLM Resources"
        )

        backend_status = gr.Markdown()

    gr.HTML("""
    <div id="research-footer">
        NetraScan.AI V2 |
        ViT-B/16 + Swin-Tiny + MobileViT-S |
        Academic Research Only<br>
        Not clinically validated for medical diagnosis.
    </div>
    """)

    analysis_outputs = [
        image_preview,
        ensemble_output,
        prediction_table,
        probability_chart,
        overlay_output,
        heatmap_output,
        xai_message,
        prediction_state,
        report_preview,
        report_download
    ]

    analyze_button.click(
        fn=reset_analysis,
        outputs=analysis_outputs,
        queue=False
    ).then(
        fn=run_analysis,
        inputs=[image_input],
        outputs=analysis_outputs,
        concurrency_id="pipeline",
        concurrency_limit=1
    )

    report_button.click(
        fn=create_report,
        inputs=[prediction_state, image_input],
        outputs=[report_preview, report_download],
        concurrency_id="pipeline",
        concurrency_limit=1
    )

    image_input.change(
        fn=reset_analysis,
        outputs=analysis_outputs,
        concurrency_id="pipeline",
        concurrency_limit=1
    )

    clear_button.click(
        fn=clear_all,
        outputs=[image_input] + analysis_outputs,
        concurrency_id="pipeline",
        concurrency_limit=1
    )

    stop_button.click(
        fn=shutdown_backend,
        outputs=[backend_status],
        concurrency_id="pipeline",
        concurrency_limit=1
    )


if __name__ == "__main__":
    try:
        demo.queue(default_concurrency_limit=1)

        demo.launch(
            server_name="127.0.0.1",
            server_port=7860,
            inbrowser=True,
            share=False,
            show_error=True,
            allowed_paths=[str(REPORTS_DIR)],
            theme=THEME,
            css=CUSTOM_CSS
        )

    finally:
        reporter.close()
'''

ast.parse(APP_SOURCE)

print("PASS — New application syntax verified.")

TEMP_PATH = V2_ROOT / "app.py.tmp"
TEMP_PATH.write_text(APP_SOURCE, encoding="utf-8")
TEMP_PATH.replace(APP_PATH)

print("\nAPPLICATION DETAILS")
print("-" * 70)

print("Application:", APP_PATH)
print("Backup:", BACKUP_PATH)
print("Lines:", len(APP_SOURCE.splitlines()))
print("Size:", APP_PATH.stat().st_size, "bytes")

print("SHA-256:", hashlib.sha256(
    APP_PATH.read_bytes()
).hexdigest())

print("\nINTEGRATED COMPONENTS")
print("-" * 70)

for component in [
    "Three-transformer inference",
    "Majority voting",
    "Probability visualization",
    "Swin-Tiny Grad-CAM",
    "Local Qwen GenAI",
    "PDF reporting",
    "Image fingerprint verification",
    "Deterministic fallback",
    "LLM resource cleanup",
    "Dark clinical UI",
    "Automatic background detection"
]:
    print("PASS — Source integrated:", component)

print("\n" + "=" * 70)
print("PASS — V2 APPLICATION SOURCE CREATED")
print("Runtime testing pending.")
print("Vision models not loaded.")
print("LLM server not started.")
print("Original V1 project untouched.")
print("SSD access: NONE")
print("=" * 70)

### 74. Integrated Application Runtime Verification

In [ ]:
# STEP 74 — Integrated Application Runtime Verification
# Cell 17 — Integrated Application Runtime Verification

from pathlib import Path
import subprocess
import sys
import json
import time

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

LOG_PATH = V2_ROOT / "tests" / "integrated_runtime.log"

print("=" * 70)
print("NETRASCAN.AI V2 — INTEGRATED RUNTIME VERIFICATION")
print("=" * 70)

test_code = r'''
from pathlib import Path
from PIL import Image
from unittest.mock import patch
import numpy as np
import subprocess
import hashlib
import json
import time
import torch
import gradio as gr

ROOT = Path.cwd()
TESTS_DIR = ROOT / "tests"

records = {}


def gpu_snapshot(label):

    process = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.used,memory.free",
            "--format=csv,noheader,nounits"
        ],
        capture_output=True,
        text=True,
        timeout=15
    )

    if process.returncode != 0:
        print(label, "GPU telemetry unavailable.")
        return None

    values = process.stdout.strip().splitlines()[0].split(",")

    snapshot = {
        "gpu": values[0].strip(),
        "total_mib": int(values[1]),
        "used_mib": int(values[2]),
        "free_mib": int(values[3])
    }

    print(label)
    print(json.dumps(snapshot, indent=4))

    return snapshot


print("\n1. BASELINE GPU MEMORY")
print("-" * 70)

records["baseline_gpu"] = gpu_snapshot("Before application import")


print("\n2. IMPORTING COMPLETE V2 APPLICATION")
print("-" * 70)

started = time.perf_counter()

import app

from src.llm_reporter import verify_inference_result

records["import_seconds"] = round(
    time.perf_counter() - started, 3
)

assert isinstance(app.demo, gr.Blocks)
assert app.engine is None
assert app.xai_engine is None
assert app.reporter.process is None

print("PASS — Application imported.")
print("PASS — Gradio components constructed.")
print("PASS — Lazy initialization preserved.")
print("Import time:", records["import_seconds"], "seconds")

config = app.demo.get_config_file()

print("Gradio components:", len(config["components"]))
print("Gradio event dependencies:", len(config["dependencies"]))
print("Background loaded:", app.BACKGROUND is not None)


try:

    print("\n3. COMPLETE VISION INFERENCE")
    print("-" * 70)

    rng = np.random.default_rng(42)

    image = Image.fromarray(
        rng.integers(
            0,
            256,
            size=(224, 224, 3),
            dtype=np.uint8
        )
    )

    started = time.perf_counter()

    outputs = app.run_analysis(image)

    records["vision_pipeline_seconds"] = round(
        time.perf_counter() - started, 3
    )

    assert len(outputs) == 10

    (
        preview,
        summary,
        table,
        chart,
        overlay,
        heatmap,
        xai_status,
        state,
        report_preview,
        report_download
    ) = outputs

    assert state is not None
    assert len(table) == 3
    assert len(chart.data) == 3
    assert report_download is None

    result = state["result"]

    agreement = verify_inference_result(result)

    assert state["fingerprint"] == app.image_fingerprint(image)

    print("Ensemble:", result["ensemble_prediction"])
    print("Agreement:", agreement)
    print("\nIndividual predictions:")
    print(table.to_string(index=False))

    print("\nPASS — All three transformer predictions verified.")
    print("PASS — Majority voting verified.")
    print("PASS — Probability chart constructed.")
    print("PASS — Image fingerprint verified.")

    records["inference_result"] = result

    print(
        "Complete vision pipeline:",
        records["vision_pipeline_seconds"],
        "seconds"
    )


    print("\n4. GRAD-CAM VERIFICATION")
    print("-" * 70)

    assert state["xai_result"] is not None
    assert isinstance(overlay, Image.Image)
    assert isinstance(heatmap, Image.Image)

    cam = state["xai_result"]["cam"]

    assert np.isfinite(cam).all()
    assert cam.ndim == 2

    expected_class = result["predictions"][
        "Swin-Tiny"
    ]["predicted_class"]

    expected_index = [
        "normal",
        "immature",
        "mature"
    ].index(expected_class)

    assert int(
        state["xai_result"]["class_index"]
    ) == expected_index

    print("Grad-CAM target:", expected_class)
    print("Activation map shape:", cam.shape)
    print("Overlay resolution:", overlay.size)
    print("Heatmap resolution:", heatmap.size)
    print("PASS — Real Grad-CAM computation verified.")

    records["gradcam"] = {
        "target": expected_class,
        "shape": list(cam.shape),
        "finite": True
    }

    records["vision_gpu"] = gpu_snapshot(
        "After vision model initialization"
    )


    print("\n5. STALE IMAGE PROTECTION")
    print("-" * 70)

    changed_image = Image.new(
        "RGB",
        (224, 224),
        (255, 0, 0)
    )

    try:

        app.create_report(
            state,
            changed_image
        )

    except gr.Error:

        print("PASS — Changed image rejected.")

    else:

        raise AssertionError(
            "Stale analysis was accepted."
        )

    assert app.reporter.process is None

    print("PASS — Invalid request did not initialize LLM.")


    print("\n6. SIMULTANEOUS VISION AND GENAI INFERENCE")
    print("-" * 70)

    started = time.perf_counter()

    genai = app.reporter.generate(result)

    records["cold_genai_seconds"] = round(
        time.perf_counter() - started, 3
    )

    print(json.dumps(genai, indent=4))

    assert genai["source"] == "LLM", genai["error"]
    assert genai["llm_accepted"]

    records["genai"] = genai

    print("PASS — Local Qwen generation with vision models loaded.")

    records["combined_gpu"] = gpu_snapshot(
        "Vision models + local Qwen"
    )

    assert app.reporter.process is not None
    assert app.reporter.process.poll() is None

    llm_pid = app.reporter.process.pid

    print("LLM process ID:", llm_pid)


    print("\n7. INTEGRATED PDF REPORT CALLBACK")
    print("-" * 70)

    from src.pdf_reporter import generate_pdf_report

    def synthetic_pdf_wrapper(*args, **kwargs):

        kwargs["synthetic"] = True
        kwargs["image_name"] = "Synthetic runtime verification image"
        kwargs["output_dir"] = TESTS_DIR

        return generate_pdf_report(*args, **kwargs)

    started = time.perf_counter()

    with patch.object(
        app,
        "generate_pdf_report",
        side_effect=synthetic_pdf_wrapper
    ):

        preview_text, pdf_path = app.create_report(
            state,
            image
        )

    records["warm_report_seconds"] = round(
        time.perf_counter() - started, 3
    )

    pdf_path = Path(pdf_path)

    assert pdf_path.is_file()
    assert "Local GenAI" in preview_text
    assert app.reporter.process.pid == llm_pid

    pdf_bytes = pdf_path.read_bytes()

    assert pdf_bytes.startswith(b"%PDF-")
    assert pdf_bytes.rstrip().endswith(b"%%EOF")

    print("PASS — Integrated report callback completed.")
    print("PASS — Existing LLM server reused.")
    print("PASS — PDF file verified.")

    print("PDF:", pdf_path)
    print("PDF size:", len(pdf_bytes), "bytes")

    print(
        "Warm report generation:",
        records["warm_report_seconds"],
        "seconds"
    )

    records["pdf"] = {
        "path": str(pdf_path),
        "size_bytes": len(pdf_bytes),
        "sha256": hashlib.sha256(pdf_bytes).hexdigest(),
        "synthetic": True
    }


    print("\n8. VISION INFERENCE WITH LLM STILL LOADED")
    print("-" * 70)

    started = time.perf_counter()

    second_outputs = app.run_analysis(image)

    records["warm_vision_seconds"] = round(
        time.perf_counter() - started, 3
    )

    second_result = second_outputs[7]["result"]

    verify_inference_result(second_result)

    assert second_result["ensemble_prediction"] == (
        result["ensemble_prediction"]
    )

    assert second_result["agreement"] == result["agreement"]

    assert second_outputs[7]["xai_result"] is not None

    assert app.reporter.process.pid == llm_pid

    print("PASS — Vision inference remains operational.")
    print("PASS — Grad-CAM remains operational.")
    print("PASS — Local LLM remains loaded.")

    print(
        "Warm vision pipeline:",
        records["warm_vision_seconds"],
        "seconds"
    )

    records["combined_gpu_after_vision"] = gpu_snapshot(
        "After repeated vision inference"
    )


    print("\n9. APPLICATION STATE RESET")
    print("-" * 70)

    reset = app.reset_analysis()

    assert len(reset) == 10
    assert reset[7] is None
    assert reset[9] is None

    cleared = app.clear_all()

    assert len(cleared) == 11
    assert cleared[0] is None
    assert cleared[8] is None
    assert cleared[10] is None

    print("PASS — Analysis state reset.")
    print("PASS — Previous report download cleared.")
    print("PASS — Input clearing verified.")


    print("\n10. LLM RESOURCE CLEANUP")
    print("-" * 70)

    status = app.shutdown_backend()

    assert app.reporter.process is None

    time.sleep(1)

    print(status)
    print("PASS — Local LLM process stopped.")

    records["after_llm_shutdown_gpu"] = gpu_snapshot(
        "After LLM shutdown"
    )

    records["torch_memory"] = {
        "allocated_mib": round(
            torch.cuda.memory_allocated() / 1024**2, 2
        ),
        "reserved_mib": round(
            torch.cuda.memory_reserved() / 1024**2, 2
        ),
        "peak_allocated_mib": round(
            torch.cuda.max_memory_allocated() / 1024**2, 2
        )
    }

    print("\nPyTorch memory statistics:")
    print(json.dumps(records["torch_memory"], indent=4))

    records["status"] = "PASS"

finally:

    app.reporter.close()

    output_path = TESTS_DIR / "integrated_runtime.json"

    output_path.write_text(
        json.dumps(records, indent=4),
        encoding="utf-8"
    )

    print("\nRuntime results:", output_path)


print("\n" + "=" * 70)
print("PASS — COMPLETE V2 RUNTIME VERIFIED")
print("=" * 70)
'''

started = time.perf_counter()

process = subprocess.run(
    [
        sys.executable,
        "-u",
        "-c",
        test_code
    ],
    cwd=str(V2_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    timeout=600
)

LOG_PATH.write_text(
    process.stdout + "\n" + process.stderr,
    encoding="utf-8"
)

print(process.stdout)

if process.stderr:
    print("\nDIAGNOSTIC OUTPUT")
    print(process.stderr)

print("\nTotal test duration:", round(
    time.perf_counter() - started, 2
), "seconds")

print("Runtime log:", LOG_PATH)

assert process.returncode == 0, (
    "Integrated runtime verification failed. "
    "Review diagnostic output."
)

print("\n11. FINAL GPU MEMORY AFTER PROCESS EXIT")
print("-" * 70)

gpu_check = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.used,memory.free",
        "--format=csv"
    ],
    capture_output=True,
    text=True
)

print(gpu_check.stdout)

print("=" * 70)
print("PASS — NETRASCAN.AI V2 INTEGRATED RUNTIME")
print("Browser interface not launched.")
print("Original V1 project untouched.")
print("SSD access: NONE")
print("=" * 70)

### 75. Launch NetraScan.AI V2

In [ ]:
# STEP 75 — Launch NetraScan.AI V2
# Cell 18 — Launch NetraScan.AI V2

from pathlib import Path
import subprocess
import requests
import webbrowser
import time
import sys
import socket

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

APP_PATH = V2_ROOT / "app.py"
LOG_PATH = V2_ROOT / "tests" / "live_application.log"

HOST = "127.0.0.1"
PORT = 7860
URL = f"http://{HOST}:{PORT}"

print("=" * 70)
print("NETRASCAN.AI V2 — LIVE APPLICATION LAUNCH")
print("=" * 70)

assert APP_PATH.is_file()

print("\n1. CHECKING APPLICATION PORT")
print("-" * 70)

with socket.socket(
    socket.AF_INET,
    socket.SOCK_STREAM
) as sock:

    assert sock.connect_ex((HOST, PORT)) != 0, (
        f"Port {PORT} is already occupied. "
        "Close the existing application first."
    )

print(f"PASS — Port {PORT} available.")

print("\n2. STARTING APPLICATION")
print("-" * 70)

started = time.perf_counter()

with LOG_PATH.open("w", encoding="utf-8") as log:

    app_process = subprocess.Popen(
        [
            sys.executable,
            "-u",
            str(APP_PATH)
        ],
        cwd=str(V2_ROOT),
        stdout=log,
        stderr=subprocess.STDOUT,
        creationflags=getattr(
            subprocess,
            "CREATE_NEW_PROCESS_GROUP",
            0
        )
    )

print("Application PID:", app_process.pid)
print("Python:", sys.executable)
print("Application:", APP_PATH)

print("\n3. WAITING FOR GRADIO SERVER")
print("-" * 70)

session = requests.Session()
session.trust_env = False

ready = False

try:

    while time.perf_counter() - started < 120:

        if app_process.poll() is not None:

            raise RuntimeError(
                "Application exited unexpectedly.\n"
                + LOG_PATH.read_text(
                    encoding="utf-8",
                    errors="replace"
                )
            )

        try:

            response = session.get(
                URL,
                timeout=3
            )

            if response.status_code == 200:

                ready = True
                break

        except requests.RequestException:
            pass

        time.sleep(1)

    if not ready:

        raise TimeoutError(
            "Gradio startup timed out."
        )

except BaseException:

    if app_process.poll() is None:
        app_process.terminate()

        try:
            app_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            app_process.kill()
            app_process.wait()

    raise

finally:

    session.close()

print("PASS — Gradio server responding.")

print(
    "Startup time:",
    round(time.perf_counter() - started, 3),
    "seconds"
)

print("\n4. APPLICATION READY")
print("-" * 70)

print("Local URL:", URL)
print("Application PID:", app_process.pid)
print("Runtime log:", LOG_PATH)

print(
    "Background image:",
    "AVAILABLE"
    if any(
        (V2_ROOT / "assets" / name).is_file()
        for name in [
            "background.png",
            "background.jpg",
            "background.jpeg",
            "background.webp"
        ]
    )
    else "DEFAULT NAVY GRADIENT"
)

print("\n" + "=" * 70)
print("PASS — NETRASCAN.AI V2 IS RUNNING")
print("Vision models will initialize on first analysis.")
print("Qwen will initialize on first report request.")
print("Original V1 project untouched.")
print("SSD access: NONE")
print("=" * 70)

### 76. Stop NetraScan.AI V2

In [ ]:
# STEP 76 — Stop NetraScan.AI V2
# Cell 18A — Stop NetraScan.AI V2

import subprocess

if app_process.poll() is None:
    subprocess.run([
        "taskkill",
        "/PID",
        str(app_process.pid),
        "/T",
        "/F"
    ])

    app_process.wait(timeout=15)
    print("PASS — NetraScan.AI V2 stopped.")
else:
    print("Application is already stopped.")

### 77. Final UI Corrections and Application Shutdown

In [ ]:
# STEP 77 — Final UI Corrections and Application Shutdown
# Cell 19 — Final UI Corrections and Application Shutdown

from pathlib import Path
import ast
import shutil
import hashlib
import socket

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

APP_PATH = V2_ROOT / "app.py"
BACKUP_PATH = V2_ROOT / "tests" / "app_v2_before_ui_patch.py"

print("=" * 70)
print("NETRASCAN.AI V2 — FINAL UI CORRECTIONS")
print("=" * 70)

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    assert sock.connect_ex(("127.0.0.1", 7860)) != 0, (
        "Stop the running application first."
    )

print("PASS — Application port available.")

if not BACKUP_PATH.exists():
    shutil.copy2(APP_PATH, BACKUP_PATH)

source = BACKUP_PATH.read_text(encoding="utf-8")

protected_files = [
    "src/inference.py",
    "src/explainability.py",
    "src/llm_reporter.py",
    "src/pdf_reporter.py"
]

original_hashes = {
    name: hashlib.sha256(
        (V2_ROOT / name).read_bytes()
    ).hexdigest()
    for name in protected_files
}


def replace_once(old, new):
    global source

    assert source.count(old) == 1, (
        f"Expected exactly one source match: {old[:100]}"
    )

    source = source.replace(old, new, 1)


print("\n1. ADDING APPLICATION SHUTDOWN")
print("-" * 70)

replace_once(
    "import atexit",
    "import atexit\nimport os\nimport time"
)

shutdown_code = '''
shutdown_requested = threading.Event()


def stop_application():

    if shutdown_requested.is_set():
        return "Application shutdown already requested."

    shutdown_requested.set()

    def shutdown_worker():

        time.sleep(1.5)

        try:
            with pipeline_lock:
                reporter.close()

            demo.close()

        finally:
            os._exit(0)

    threading.Thread(
        target=shutdown_worker,
        daemon=True
    ).start()

    return (
        "Application shutting down. "
        "You may now close this browser tab."
    )


'''

replace_once(
    "BACKGROUND = None",
    shutdown_code + "BACKGROUND = None"
)

replace_once(
    '    with gr.Row():\n\n        with gr.Column(',
    '''    with gr.Row(elem_id="application-controls"):

        application_status = gr.Markdown(
            "**Application status:** Online"
        )

        exit_button = gr.Button(
            "Stop Application",
            variant="stop",
            elem_id="exit-button"
        )

    with gr.Row():

        with gr.Column('''
)

replace_once(
    "    stop_button.click(",
    '''    exit_button.click(
        fn=stop_application,
        outputs=[application_status],
        concurrency_id="pipeline",
        concurrency_limit=1
    )

    stop_button.click('''
)

replace_once(
    "    if image is None:\n",
    '''    if shutdown_requested.is_set():
        raise gr.Error("Application is shutting down.")

    if image is None:
'''
)

replace_once(
    "    if state is None or current_image is None:",
    '''    if shutdown_requested.is_set():
        raise gr.Error("Application is shutting down.")

    if state is None or current_image is None:'''
)

print("PASS — Stop Application button integrated.")
print("PASS — LLM cleanup included.")
print("PASS — New analysis blocked during shutdown.")


print("\n2. FIXING PROBABILITY CHART")
print("-" * 70)

replace_once(
    "margin=dict(l=45, r=20, t=25, b=45)",
    "margin=dict(l=45, r=20, t=65, b=85)"
)

replace_once(
    '''            orientation="h",
            y=1.12,
            x=0''',
    '''            orientation="h",
            y=-0.23,
            x=0.5,
            xanchor="center",
            yanchor="top"'''
)

replace_once(
    "height=340,",
    "height=390,"
)

print("PASS — Legend moved below probability chart.")


print("\n3. IMPROVING AGREEMENT DISPLAY")
print("-" * 70)

replace_once(
    '        agreement = result["agreement"]',
    '''        agreement = result["agreement"]

        agreement_message = {
            "3/3": "Unanimous model agreement.",
            "2/3": "Majority prediction — model disagreement present.",
            "1/3": "Inconclusive — no majority agreement."
        }[agreement]'''
)

replace_once(
    "**Model agreement:** {agreement}",
    """**Model agreement:** {agreement}

**Agreement status:** {agreement_message}"""
)

print("PASS — Agreement status added.")
print("PASS — Majority voting algorithm unchanged.")


print("\n4. REFINING INTERFACE STYLING")
print("-" * 70)

replace_once(
    "rgba(10,22,40,0.88)",
    "rgba(10,22,40,0.76)"
)

replace_once(
    "rgba(34,211,238,0.18)",
    "rgba(34,211,238,0.28)"
)

extra_css = '''
#application-controls {
    align-items: center !important;
    margin-bottom: 12px !important;
}

#exit-button {
    background: linear-gradient(
        110deg,
        #991b1b,
        #dc2626
    ) !important;
    color: #ffffff !important;
    border: 1px solid rgba(248,113,113,0.45) !important;
    border-radius: 10px !important;
    font-weight: 700 !important;
    max-width: 220px !important;
    min-height: 42px !important;
}

#exit-button:hover {
    background: #b91c1c !important;
    box-shadow: 0 0 18px rgba(239,68,68,0.22);
}

.glass-panel {
    box-shadow: 0 8px 30px rgba(0,0,0,0.16);
}

#hero {
    box-shadow: 0 8px 35px rgba(8,145,178,0.08);
}

'''

replace_once(
    "if BACKGROUND:\n",
    "CUSTOM_CSS += " + repr(extra_css) + "\n\nif BACKGROUND:\n"
)

print("PASS — UI styling updated.")


print("\n5. CHECKING BACKGROUND ASSETS")
print("-" * 70)

backgrounds = [
    V2_ROOT / "assets" / name
    for name in [
        "background.png",
        "background.jpg",
        "background.jpeg",
        "background.webp"
    ]
]

available = [
    path for path in backgrounds
    if path.is_file()
]

if available:
    print("Background:", available[0])
else:
    print("Custom background not found.")
    print("Default navy gradient will remain active.")


print("\n6. VERIFYING UPDATED APPLICATION")
print("-" * 70)

ast.parse(source)

print("PASS — Python syntax verified.")

for name, original_hash in original_hashes.items():

    current_hash = hashlib.sha256(
        (V2_ROOT / name).read_bytes()
    ).hexdigest()

    assert current_hash == original_hash

    print(f"PASS — {name} unchanged.")

temp_path = V2_ROOT / "app.py.tmp"

temp_path.write_text(
    source,
    encoding="utf-8"
)

temp_path.replace(APP_PATH)

print("\nApplication:", APP_PATH)
print("Backup:", BACKUP_PATH)
print("Updated lines:", len(source.splitlines()))
print("Size:", APP_PATH.stat().st_size, "bytes")

print("\nSHA-256:", hashlib.sha256(
    APP_PATH.read_bytes()
).hexdigest())

print("\n" + "=" * 70)
print("PASS — FINAL UI SOURCE UPDATED")
print("Shutdown button added.")
print("Probability chart legend corrected.")
print("Agreement warnings added.")
print("All backend modules preserved.")
print("Runtime verification pending.")
print("Original V1 project untouched.")
print("SSD access: NONE")
print("=" * 70)

### 78. Final Application Launch

In [ ]:
# STEP 78 — Final Application Launch
# Cell 20 — Final Application Launch

from pathlib import Path
import subprocess
import requests
import socket
import sys
import time

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

APP_PATH = V2_ROOT / "app.py"
LOG_PATH = V2_ROOT / "tests" / "final_ui_runtime.log"

HOST = "127.0.0.1"
PORT = 7860
URL = f"http://{HOST}:{PORT}"

print("=" * 70)
print("NETRASCAN.AI V2 — FINAL UI VERIFICATION")
print("=" * 70)

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    assert sock.connect_ex((HOST, PORT)) != 0, (
        "Port 7860 is already occupied."
    )

print("PASS — Application port available.")

started = time.perf_counter()

with LOG_PATH.open("w", encoding="utf-8") as log:

    app_process = subprocess.Popen(
        [
            sys.executable,
            "-u",
            str(APP_PATH)
        ],
        cwd=str(V2_ROOT),
        stdout=log,
        stderr=subprocess.STDOUT,
        creationflags=getattr(
            subprocess,
            "CREATE_NEW_PROCESS_GROUP",
            0
        )
    )

print("Application PID:", app_process.pid)

session = requests.Session()
session.trust_env = False

ready = False

try:

    while time.perf_counter() - started < 120:

        if app_process.poll() is not None:
            raise RuntimeError(
                LOG_PATH.read_text(
                    encoding="utf-8",
                    errors="replace"
                )
            )

        try:
            response = session.get(URL, timeout=3)

            if response.status_code == 200:
                ready = True
                break

        except requests.RequestException:
            pass

        time.sleep(1)

    if not ready:
        raise TimeoutError("Application startup timed out.")

    print("PASS — Gradio server responding.")

    config = session.get(
        URL + "/config",
        timeout=10
    )

    config.raise_for_status()
    config = config.json()

    buttons = [
        component.get("props", {}).get("value")
        for component in config["components"]
        if component.get("type") == "button"
    ]

    assert "Stop Application" in buttons
    assert "Release Local LLM Resources" in buttons
    assert "Generate AI Research Report" in buttons

    print("PASS — Stop Application button registered.")
    print("PASS — LLM resource release button registered.")
    print("PASS — GenAI report button registered.")

except BaseException:

    if app_process.poll() is None:
        app_process.terminate()

        try:
            app_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            app_process.kill()
            app_process.wait()

    raise

finally:
    session.close()

print("\nStartup time:", round(
    time.perf_counter() - started, 3
), "seconds")

print("URL:", URL)
print("Log:", LOG_PATH)

print("\n" + "=" * 70)
print("PASS — UPDATED APPLICATION RUNNING")
print("Manual shutdown verification pending.")
print("Original V1 project untouched.")
print("SSD access: NONE")
print("=" * 70)

### 79. Benchmark Data Consolidation

In [ ]:
# STEP 79 — Benchmark Data Consolidation
# Cell 21 — Benchmark Data Consolidation

from pathlib import Path
import json
import hashlib

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

TESTS_DIR = V2_ROOT / "tests"
BENCHMARK_DIR = V2_ROOT / "benchmarks"

BENCHMARK_DIR.mkdir(exist_ok=True)

print("=" * 75)
print("NETRASCAN.AI V2 — BENCHMARK CONSOLIDATION")
print("=" * 75)

assert TESTS_DIR.exists()

benchmark_files = [
    "llm_benchmark.json",
    "structured_generation.json",
    "grounded_generation.json",
    "validator_regression.json",
    "pdf_environment.json",
    "pdf_reporter_tests.json",
    "integrated_runtime.json"
]

consolidated = {}
missing = []

for filename in benchmark_files:

    path = TESTS_DIR / filename

    print("\n" + "-" * 75)
    print("FILE:", filename)
    print("-" * 75)

    if not path.exists():
        missing.append(filename)
        print("WARNING — File not found.")
        continue

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    consolidated[path.stem] = data

    print("Size:", path.stat().st_size, "bytes")
    print("SHA-256:", hashlib.sha256(
        path.read_bytes()
    ).hexdigest())

    print("\nRecorded benchmark data:")
    print(json.dumps(data, indent=2, ensure_ascii=False))

print("\n" + "=" * 75)
print("APPLICATION SHUTDOWN VERIFICATION")
print("=" * 75)

import socket

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    sock.settimeout(2)
    port_released = sock.connect_ex(("127.0.0.1", 7860)) != 0

process_stopped = (
    app_process.poll() is not None
    if "app_process" in globals()
    else None
)

print("Port 7860 released:", port_released)
print("Application process stopped:", process_stopped)

print("\n" + "=" * 75)
print("BENCHMARK SUMMARY")
print("=" * 75)

print("Expected files:", len(benchmark_files))
print("Available files:", len(consolidated))
print("Missing files:", len(missing))

if missing:
    for filename in missing:
        print("Missing:", filename)

print("\nAdditional JSON files:")

for path in sorted(TESTS_DIR.glob("*.json")):
    if path.name not in benchmark_files:
        print(path.name)

print("\nBenchmark directory:", BENCHMARK_DIR)

print("\n" + "=" * 75)
print("PASS — BENCHMARK DISCOVERY COMPLETE")
print("Original benchmark files preserved.")
print("No estimated metrics introduced.")
print("No private eye images accessed.")
print("No model modifications.")
print("SSD access: NONE")
print("=" * 75)

### 80. Publication-Ready Benchmark Visualization

In [ ]:
# STEP 80 — Publication-Ready Benchmark Visualization
# Cell 21B — Publication-Ready Benchmark Visualization

from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

TESTS_DIR = V2_ROOT / "tests"
BENCHMARK_DIR = V2_ROOT / "benchmarks"
FIGURES_DIR = BENCHMARK_DIR / "figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def load_json(name):
    return json.loads(
        (TESTS_DIR / name).read_text(encoding="utf-8")
    )


runtime = load_json("integrated_runtime.json")
llm = load_json("llm_benchmark.json")
structured = load_json("structured_generation.json")
grounded = load_json("grounded_generation.json")
validator = load_json("validator_regression.json")

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.15,
    "axes.axisbelow": True
})

NAVY = "#142B49"
BLUE = "#2563EB"
CYAN = "#0891B2"
GREEN = "#059669"
ORANGE = "#D97706"
GRAY = "#94A3B8"
RED = "#DC2626"

generated_files = []


def save_figure(fig, name):

    fig.text(
        0.5, 0.015,
        "NetraScan.AI V2 | RTX 3080 12 GB | Research Prototype",
        ha="center",
        fontsize=9,
        color="#64748B"
    )

    fig.tight_layout(rect=[0, 0.055, 1, 1])

    for extension in ["png", "svg"]:

        path = FIGURES_DIR / f"{name}.{extension}"

        fig.savefig(
            path,
            dpi=300,
            bbox_inches="tight"
        )

        generated_files.append(path)

    plt.show()
    plt.close(fig)

    print("PASS —", name)


print("=" * 75)
print("NETRASCAN.AI V2 — BENCHMARK VISUALIZATION")
print("=" * 75)


# 01 — GPU Memory Consumption

memory_labels = [
    "System\nBaseline",
    "Vision\nModels",
    "Vision +\nQwen",
    "After LLM\nShutdown"
]

memory_values = [
    runtime["baseline_gpu"]["used_mib"],
    runtime["vision_gpu"]["used_mib"],
    runtime["combined_gpu"]["used_mib"],
    runtime["after_llm_shutdown_gpu"]["used_mib"]
]

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(
    memory_labels,
    np.array(memory_values) / 1024,
    color=[GRAY, BLUE, CYAN, GREEN],
    width=0.58
)

ax.bar_label(
    bars,
    fmt="%.2f GiB",
    padding=6,
    fontweight="bold"
)

ax.axhline(
    12,
    linestyle="--",
    color=RED,
    alpha=0.65,
    label="GPU capacity: 12 GiB"
)

ax.set_ylim(0, 13.5)
ax.set_ylabel("Total Device VRAM Used (GiB)")
ax.set_title("GPU Memory Utilization Across Pipeline Stages")
ax.legend(loc="upper left")

save_figure(fig, "01_gpu_memory_utilization")


# 02 — Cold vs Warm Pipeline Latency

labels = ["Vision Pipeline", "GenAI Report"]

cold = [
    runtime["vision_pipeline_seconds"] * 1000,
    runtime["cold_genai_seconds"] * 1000
]

warm = [
    runtime["warm_vision_seconds"] * 1000,
    runtime["warm_report_seconds"] * 1000
]

x = np.arange(len(labels))
width = 0.32

fig, ax = plt.subplots(figsize=(10, 6))

b1 = ax.bar(
    x - width / 2,
    cold,
    width,
    label="Cold",
    color=NAVY
)

b2 = ax.bar(
    x + width / 2,
    warm,
    width,
    label="Warm",
    color=CYAN
)

ax.bar_label(b1, fmt="%.0f ms", padding=5)
ax.bar_label(b2, fmt="%.0f ms", padding=5)

ax.set_xticks(x, labels)
ax.set_ylim(0, max(cold) * 1.22)
ax.set_ylabel("Latency (milliseconds)")
ax.set_title("Cold and Warm Runtime Comparison")
ax.legend()

ax.text(
    0.5, -0.12,
    "Cold GenAI excludes PDF creation; warm report includes PDF.",
    transform=ax.transAxes,
    ha="center",
    fontsize=9
)

save_figure(fig, "02_cold_warm_latency")


# 03 — Individual Transformer Inference

predictions = runtime["inference_result"]["predictions"]

model_names = list(predictions.keys())

model_latency = [
    predictions[name]["inference_ms"]
    for name in model_names
]

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    model_names,
    model_latency,
    color=[BLUE, CYAN, GREEN],
    height=0.5
)

ax.bar_label(
    bars,
    fmt="%.2f ms",
    padding=7,
    fontweight="bold"
)

ax.invert_yaxis()
ax.set_xlim(0, max(model_latency) * 1.28)
ax.set_xlabel("Inference Latency (milliseconds)")
ax.set_title("Individual Transformer Inference Latency")

save_figure(fig, "03_transformer_inference_latency")


# 04 — Local LLM Generation Throughput

throughput_labels = [
    "Standalone\nFree-form",
    "Grounded\nMajority",
    "Grounded\nUnanimous",
    "Grounded\nInconclusive",
    "Integrated\nPipeline"
]

throughput_values = [
    llm["generation_tokens_per_second"],
    grounded["majority_agreement"]["timings"][
        "predicted_per_second"
    ],
    grounded["unanimous_agreement"]["timings"][
        "predicted_per_second"
    ],
    grounded["inconclusive"]["timings"][
        "predicted_per_second"
    ],
    runtime["genai"]["tokens_per_second"]
]

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(
    throughput_labels,
    throughput_values,
    color=[GRAY, BLUE, CYAN, GREEN, NAVY],
    width=0.58
)

ax.bar_label(bars, fmt="%.2f", padding=6)

ax.set_ylim(0, max(throughput_values) * 1.25)
ax.set_ylabel("Generated Tokens per Second")
ax.set_title("Qwen2.5-3B-Instruct Generation Throughput")

save_figure(fig, "04_llm_generation_throughput")


# 05 — Narrative Validation Regression

structured_accepted = sum(
    bool(case["llm_accepted"])
    for case in structured.values()
)

grounded_accepted = validator["original_accepted"]
corrected_accepted = validator["corrected_accepted"]
total_valid = validator["total_valid_samples"]

validation_labels = [
    "Structured\nGeneration",
    "Grounded\nOriginal Validator",
    "Grounded\nCorrected Validator"
]

validation_values = [
    structured_accepted,
    grounded_accepted,
    corrected_accepted
]

validation_totals = [
    len(structured),
    total_valid,
    total_valid
]

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(
    validation_labels,
    [
        accepted / total * 100
        for accepted, total in zip(
            validation_values,
            validation_totals
        )
    ],
    color=[RED, ORANGE, GREEN],
    width=0.55
)

ax.bar_label(
    bars,
    labels=[
        f"{accepted}/{total}"
        for accepted, total in zip(
            validation_values,
            validation_totals
        )
    ],
    padding=7,
    fontweight="bold"
)

ax.set_ylim(0, 120)
ax.set_ylabel("Narrative Acceptance (%)")
ax.set_title("GenAI Narrative Validation Regression")

save_figure(fig, "05_narrative_validation")


# 06 — Adversarial Validation Tests

adversarial = validator["adversarial_results"]

attack_names = [
    case["name"]
    for case in adversarial
]

attack_results = [
    int(case["correctly_rejected"])
    for case in adversarial
]

fig, ax = plt.subplots(figsize=(12, 7))

bars = ax.barh(
    attack_names,
    attack_results,
    color=[
        GREEN if result else RED
        for result in attack_results
    ],
    height=0.55
)

ax.bar_label(
    bars,
    labels=[
        "REJECTED" if result else "MISSED"
        for result in attack_results
    ],
    padding=7,
    fontweight="bold"
)

ax.set_xlim(0, 1.35)
ax.set_xticks([])
ax.invert_yaxis()
ax.set_title("Adversarial Narrative Validation Tests")

save_figure(fig, "06_adversarial_validation")


# Sanitized Benchmark Summary

baseline = runtime["baseline_gpu"]["used_mib"]
vision = runtime["vision_gpu"]["used_mib"]
combined = runtime["combined_gpu"]["used_mib"]
released = runtime["after_llm_shutdown_gpu"]["used_mib"]

summary = {
    "project": "NetraScan.AI V2",
    "hardware": {
        "gpu": runtime["baseline_gpu"]["gpu"],
        "gpu_capacity_mib": runtime["baseline_gpu"]["total_mib"],
        "cpu": "Intel Core i7-12700K",
        "ram_gib": 64
    },
    "llm": {
        "model": llm["model"],
        "quantization": llm["quantization"],
        "backend": llm["backend"],
        "standalone_startup_seconds": llm["startup_seconds"],
        "standalone_generation_tokens_per_second":
            llm["generation_tokens_per_second"],
        "integrated_generation_tokens_per_second":
            runtime["genai"]["tokens_per_second"]
    },
    "runtime": {
        "import_seconds": runtime["import_seconds"],
        "cold_vision_seconds": runtime["vision_pipeline_seconds"],
        "warm_vision_seconds": runtime["warm_vision_seconds"],
        "cold_genai_seconds": runtime["cold_genai_seconds"],
        "warm_report_seconds": runtime["warm_report_seconds"],
        "warm_combined_stage_seconds": round(
            runtime["warm_vision_seconds"]
            + runtime["warm_report_seconds"], 3
        ),
        "individual_model_latency_ms": {
            name: predictions[name]["inference_ms"]
            for name in model_names
        }
    },
    "gpu_memory": {
        "baseline_mib": baseline,
        "vision_loaded_mib": vision,
        "combined_loaded_mib": combined,
        "after_llm_shutdown_mib": released,
        "vision_incremental_mib": vision - baseline,
        "llm_incremental_mib": combined - vision,
        "combined_incremental_mib": combined - baseline,
        "llm_released_mib": combined - released,
        "torch_memory": runtime["torch_memory"]
    },
    "validation": {
        "structured_accepted": structured_accepted,
        "structured_total": len(structured),
        "grounded_original_accepted": grounded_accepted,
        "grounded_corrected_accepted": corrected_accepted,
        "grounded_total": total_valid,
        "adversarial_rejected": validator["adversarial_rejected"],
        "adversarial_total": validator["total_adversarial_samples"]
    },
    "limitations": [
        "Runtime measurements are individual recorded runs.",
        "LLM workloads differ in prompt length and caching.",
        "Cold GenAI timing excludes PDF creation.",
        "Warm report timing includes PDF creation.",
        "Validation uses a small designed synthetic test suite.",
        "GPU memory values include background device usage.",
        "Clinical validation has not been established."
    ]
}

summary_path = BENCHMARK_DIR / "benchmark_summary.json"

summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

assert len(generated_files) == 12

for path in generated_files:
    assert path.exists()
    assert path.stat().st_size > 0

print("\n" + "=" * 75)
print("GENERATED BENCHMARK ARTIFACTS")
print("=" * 75)

for path in generated_files:
    print(
        path.name,
        "|",
        round(path.stat().st_size / 1024, 2),
        "KB"
    )

print("\nSummary:", summary_path)
print("Figures generated:", len(generated_files))
print("Original benchmark records preserved.")
print("Private photographs excluded.")
print("Absolute local paths excluded from summary.")
print("SSD access: NONE")

print("\nPASS — BENCHMARK VISUALIZATION COMPLETE")
print("=" * 75)

### Recorded integrated desktop measurements
| Metric | Recorded value |
|---|---:|
| Baseline GPU memory used | 2458 MiB |
| Vision pipeline resident | 3451 MiB |
| Vision + local LLM resident | 5750 MiB |
| After LLM shutdown | 3448 MiB |
| Cold vision pipeline | 2.179 s |
| Warm vision pipeline | 0.079 s |
| Integrated generation throughput | 134.79 tokens/s |
| Warm report generation | 0.336 s |

Single-run RTX 3080 measurements from the original integrated test. Cold and warm report operations have different scopes. No cross-device performance comparison is included.

## 10. Independent portable V2 deployment
Copy the verified V1 runtime and V2 application assets into a separate SSD directory. Preserve the original V1 installation. The following deployment cells require an existing V1 runtime and a completed V2 workspace.

### 81. Portable SSD Preflight Inspection

In [ ]:
# STEP 81 — Portable SSD Preflight Inspection
# Cell 22A — Portable SSD Preflight Inspection

from pathlib import Path
import shutil
import string

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

print("=" * 75)
print("NETRASCAN.AI V2 — PORTABLE DEPLOYMENT PREFLIGHT")
print("=" * 75)

detected = []

for letter in string.ascii_uppercase:

    drive = Path(f"{letter}:/")

    if not drive.exists():
        continue

    portable_v1 = drive / "NetraScan.AI_PORTABLE"

    if portable_v1.is_dir():

        detected.append(portable_v1)

        usage = shutil.disk_usage(drive)

        print("\nPortable V1 detected:", portable_v1)
        print("Drive:", drive)
        print("Total capacity:", round(usage.total / 1024**3, 2), "GiB")
        print("Used space:", round(usage.used / 1024**3, 2), "GiB")
        print("Free space:", round(usage.free / 1024**3, 2), "GiB")

assert len(detected) == 1, (
    "Expected exactly one existing portable V1 installation."
)

PORTABLE_V1 = detected[0]
SSD_ROOT = PORTABLE_V1.parent
PORTABLE_V2 = SSD_ROOT / "NetraScan.AI_V2_PORTABLE"


def directory_size(path):

    return sum(
        file.stat().st_size
        for file in path.rglob("*")
        if file.is_file()
    )


print("\n" + "=" * 75)
print("EXISTING PORTABLE V1 STRUCTURE")
print("=" * 75)

for path in sorted(PORTABLE_V1.iterdir()):

    size = (
        directory_size(path)
        if path.is_dir()
        else path.stat().st_size
    )

    print(
        "[DIR]" if path.is_dir() else "[FILE]",
        path.name,
        "|",
        round(size / 1024**3, 3),
        "GiB"
    )

print("\n" + "=" * 75)
print("DESKTOP V2 STRUCTURE")
print("=" * 75)

for path in sorted(V2_ROOT.iterdir()):

    size = (
        directory_size(path)
        if path.is_dir()
        else path.stat().st_size
    )

    print(
        "[DIR]" if path.is_dir() else "[FILE]",
        path.name,
        "|",
        round(size / 1024**3, 3),
        "GiB"
    )

print("\n" + "=" * 75)
print("DEPLOYMENT TARGET")
print("=" * 75)

print("V1:", PORTABLE_V1)
print("V2:", PORTABLE_V2)
print("V2 destination already exists:", PORTABLE_V2.exists())

print("\n" + "=" * 75)
print("PASS — SSD PREFLIGHT INSPECTION COMPLETE")
print("No files copied.")
print("No files deleted.")
print("No SSD modifications.")
print("Existing V1 preserved.")
print("=" * 75)

### 82. Portable Runtime Compatibility Inspection

In [ ]:
# STEP 82 — Portable Runtime Compatibility Inspection
# Cell 22B — Portable Runtime Compatibility Inspection

from pathlib import Path
import subprocess
import json
import sys

PORTABLE_V1 = Path(r"E:\NetraScan.AI_PORTABLE")
PORTABLE_V2 = Path(r"E:\NetraScan.AI_V2_PORTABLE")

RUNTIME_DIR = PORTABLE_V1 / "runtime"

print("=" * 75)
print("NETRASCAN.AI V2 — PORTABLE RUNTIME INSPECTION")
print("=" * 75)

assert RUNTIME_DIR.is_dir()
assert not PORTABLE_V2.exists()

print("\n1. PORTABLE PYTHON EXECUTABLES")
print("-" * 75)

executables = list(RUNTIME_DIR.rglob("python.exe"))

for path in executables:
    print(path)

assert executables, "Portable Python executable not found."

print("\n2. EXISTING V1 LAUNCHER")
print("-" * 75)

launcher = PORTABLE_V1 / "START_NETRASCAN.bat"

print(launcher.read_text(
    encoding="utf-8",
    errors="replace"
))

print("\n3. RUNTIME DIRECTORY STRUCTURE")
print("-" * 75)

for path in sorted(RUNTIME_DIR.iterdir()):
    print(
        "[DIR]" if path.is_dir() else "[FILE]",
        path.name
    )

packages = [
    "torch",
    "torchvision",
    "timm",
    "transformers",
    "gradio",
    "numpy",
    "pandas",
    "Pillow",
    "grad-cam",
    "requests",
    "opencv-python",
    "plotly",
    "reportlab"
]

inspection_code = f"""
import sys
import json
import importlib.metadata as metadata

packages = {packages!r}

versions = {{}}

for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "NOT INSTALLED"

print(json.dumps({{
    "python": sys.version,
    "executable": sys.executable,
    "packages": versions
}}))
"""

print("\n4. PORTABLE PACKAGE VERSIONS")
print("-" * 75)

for executable in executables:

    result = subprocess.run(
        [
            str(executable),
            "-B",
            "-c",
            inspection_code
        ],
        capture_output=True,
        text=True,
        timeout=60,
        cwd=str(PORTABLE_V1)
    )

    print("\nExecutable:", executable)

    if result.returncode == 0:

        try:
            details = json.loads(result.stdout)
            print(json.dumps(details, indent=2))
        except json.JSONDecodeError:
            print(result.stdout)

    else:
        print("WARNING — Runtime inspection failed.")
        print(result.stderr)

print("\n5. DESKTOP PACKAGE VERSIONS")
print("-" * 75)

desktop_result = subprocess.run(
    [sys.executable, "-B", "-c", inspection_code],
    capture_output=True,
    text=True,
    timeout=60
)

print(desktop_result.stdout)

print("\n6. V2 LLM MANIFEST")
print("-" * 75)

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

manifest = V2_ROOT / "llm" / "manifest.json"

assert manifest.is_file()

print(manifest.read_text(encoding="utf-8"))

print("\n" + "=" * 75)
print("PORTABLE RUNTIME INSPECTION COMPLETE")
print("No packages installed.")
print("No files copied.")
print("Existing V1 preserved.")
print("V2 destination not created yet.")
print("=" * 75)

### 83. Independent Portable V2 Deployment

In [ ]:
# STEP 83 — Independent Portable V2 Deployment
# Cell 22C — Independent Portable V2 Deployment

from pathlib import Path
import shutil
import hashlib
import time
import json

V2_ROOT = Path(
    r"C:\NetraScanWorkspace\NetraScan_V2"
)

PORTABLE_V1 = Path(r"E:\NetraScan.AI_PORTABLE")
PORTABLE_V2 = Path(r"E:\NetraScan.AI_V2_PORTABLE")

print("=" * 75)
print("NETRASCAN.AI V2 — PORTABLE DEPLOYMENT")
print("=" * 75)

assert PORTABLE_V1.is_dir()
assert V2_ROOT.is_dir()
assert not PORTABLE_V2.exists()
assert PORTABLE_V1.resolve() != PORTABLE_V2.resolve()


def sha256_file(path):

    digest = hashlib.sha256()

    with path.open("rb") as f:

        while chunk := f.read(8 * 1024 * 1024):
            digest.update(chunk)

    return digest.hexdigest()


def directory_size(path):

    return sum(
        file.stat().st_size
        for file in path.rglob("*")
        if file.is_file()
    )


def file_manifest(path):

    return {
        file.relative_to(path).as_posix(): (
            file.stat().st_size,
            file.stat().st_mtime_ns
        )
        for file in path.rglob("*")
        if file.is_file()
    }


print("\n1. RECORDING V1 INTEGRITY")
print("-" * 75)

v1_before = file_manifest(PORTABLE_V1)

v1_app_hash = sha256_file(PORTABLE_V1 / "app.py")
v1_launcher_hash = sha256_file(
    PORTABLE_V1 / "START_NETRASCAN.bat"
)

print("V1 files recorded:", len(v1_before))
print("PASS — V1 integrity baseline recorded.")


print("\n2. CHECKING STORAGE REQUIREMENTS")
print("-" * 75)

copy_plan = {
    "runtime": PORTABLE_V1 / "runtime",
    "models": V2_ROOT / "models",
    "llm": V2_ROOT / "llm",
    "src": V2_ROOT / "src",
    "assets": V2_ROOT / "assets",
    "config": V2_ROOT / "config",
    "benchmarks": V2_ROOT / "benchmarks"
}

for name, source in copy_plan.items():
    assert source.is_dir(), f"Missing source: {name}"

required_bytes = sum(
    directory_size(source)
    for source in copy_plan.values()
)

free_bytes = shutil.disk_usage(
    PORTABLE_V2.parent
).free

print("Required:", round(required_bytes / 1024**3, 3), "GiB")
print("Available:", round(free_bytes / 1024**3, 3), "GiB")

assert free_bytes > required_bytes + 2 * 1024**3

print("PASS — Sufficient SSD storage.")


print("\n3. COPYING PORTABLE COMPONENTS")
print("-" * 75)

PORTABLE_V2.mkdir()

deployment_start = time.perf_counter()

copy_results = {}

for name, source in copy_plan.items():

    destination = PORTABLE_V2 / name

    print(f"\nCopying {name}...", flush=True)

    started = time.perf_counter()

    shutil.copytree(
        source,
        destination,
        copy_function=shutil.copy2,
        ignore=shutil.ignore_patterns(
            "__pycache__",
            "*.pyc",
            "*.pyo"
        )
    )

    elapsed = time.perf_counter() - started

    copied_bytes = directory_size(destination)

    speed = (
        copied_bytes / 1024**2 / elapsed
        if elapsed > 0 else 0
    )

    copy_results[name] = {
        "size_bytes": copied_bytes,
        "seconds": round(elapsed, 3),
        "average_copy_mib_per_second": round(speed, 2)
    }

    print("Size:", round(copied_bytes / 1024**3, 3), "GiB")
    print("Time:", round(elapsed, 2), "seconds")
    print("Average copy rate:", round(speed, 2), "MiB/s")
    print(f"PASS — {name} copied.")


print("\n4. COPYING APPLICATION FILES")
print("-" * 75)

for filename in [
    "app.py",
    "README.md",
    "requirements.txt"
]:

    source = V2_ROOT / filename

    assert source.is_file()

    shutil.copy2(
        source,
        PORTABLE_V2 / filename
    )

    print("PASS —", filename)

for name in ["reports", "logs", "tests"]:
    (PORTABLE_V2 / name).mkdir(exist_ok=True)

print("PASS — Empty output directories created.")
print("Existing personal reports not copied.")


print("\n5. VERIFYING MODEL CHECKPOINTS")
print("-" * 75)

checkpoints = list(
    (V2_ROOT / "models").rglob("*.pth")
)

assert len(checkpoints) == 3

for source in checkpoints:

    relative = source.relative_to(V2_ROOT)
    destination = PORTABLE_V2 / relative

    assert destination.is_file()

    assert sha256_file(source) == sha256_file(destination)

    print("PASS —", relative.as_posix())


print("\n6. VERIFYING QWEN MODEL AND BACKEND")
print("-" * 75)

manifest_path = PORTABLE_V2 / "llm" / "manifest.json"

manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

gguf = PORTABLE_V2 / manifest["model_path"]
backend = PORTABLE_V2 / manifest["backend_executable"]

assert gguf.is_file()
assert backend.is_file()

assert gguf.stat().st_size == manifest["model_size_bytes"]

assert sha256_file(gguf) == manifest["model_sha256"]

print("PASS — Qwen GGUF SHA-256 verified.")

backend_source = V2_ROOT / manifest["backend_executable"]

assert sha256_file(backend) == sha256_file(backend_source)

print("PASS — llama.cpp executable SHA-256 verified.")


print("\n7. VERIFYING APPLICATION MODULES")
print("-" * 75)

source_files = [
    "app.py",
    "src/inference.py",
    "src/explainability.py",
    "src/llm_reporter.py",
    "src/pdf_reporter.py"
]

for filename in source_files:

    assert sha256_file(
        V2_ROOT / filename
    ) == sha256_file(
        PORTABLE_V2 / filename
    )

    print("PASS —", filename)


print("\n8. VERIFYING V1 WAS NOT MODIFIED")
print("-" * 75)

v1_after = file_manifest(PORTABLE_V1)

assert v1_before == v1_after

assert sha256_file(
    PORTABLE_V1 / "app.py"
) == v1_app_hash

assert sha256_file(
    PORTABLE_V1 / "START_NETRASCAN.bat"
) == v1_launcher_hash

print("PASS — V1 file inventory unchanged.")
print("PASS — V1 application hash unchanged.")
print("PASS — V1 launcher hash unchanged.")


print("\n9. FINAL DEPLOYMENT SUMMARY")
print("-" * 75)

elapsed = time.perf_counter() - deployment_start

portable_python = (
    PORTABLE_V2 / "runtime" / "python" / "python.exe"
)

assert portable_python.is_file()

deployment_summary = {
    "status": "FILES_DEPLOYED",
    "copy_results": copy_results,
    "deployment_seconds": round(elapsed, 3),
    "portable_size_bytes": directory_size(PORTABLE_V2),
    "vision_checkpoints_verified": len(checkpoints),
    "qwen_sha256_verified": True,
    "backend_executable_verified": True,
    "v1_inventory_unchanged": True,
    "v1_app_hash_unchanged": True,
    "v1_launcher_hash_unchanged": True,
    "runtime_dependencies_pending": [
        "requests",
        "reportlab"
    ]
}

summary_path = (
    PORTABLE_V2 / "tests" / "portable_deployment.json"
)

summary_path.write_text(
    json.dumps(deployment_summary, indent=2),
    encoding="utf-8"
)

print("Portable V2:", PORTABLE_V2)
print("Python:", portable_python)

print("Total size:", round(
    directory_size(PORTABLE_V2) / 1024**3, 3
), "GiB")

print("Deployment time:", round(elapsed, 2), "seconds")

print("\n" + "=" * 75)
print("PASS — PORTABLE V2 FILE DEPLOYMENT COMPLETE")
print("All three transformer checkpoints verified.")
print("Qwen GGUF verified.")
print("Application modules verified.")
print("Existing V1 preserved.")
print("Independent Python runtime copied.")
print("Dependency installation pending.")
print("Portable launcher creation pending.")
print("Runtime verification pending.")
print("=" * 75)

### 84. Install portable reporting dependencies

In [ ]:
# STEP 84 — Install portable reporting dependencies
# Cell 22D — Portable V2 Dependency Installation

from pathlib import Path
import subprocess
import sys
import json
import os

PORTABLE_V1 = Path(r"E:\NetraScan.AI_PORTABLE")
PORTABLE_V2 = Path(r"E:\NetraScan.AI_V2_PORTABLE")

PORTABLE_PYTHON = (
    PORTABLE_V2 / "runtime" / "python" / "python.exe"
)

SITE_PACKAGES = (
    PORTABLE_V2 / "runtime" / "python"
    / "Lib" / "site-packages"
)

print("=" * 75)
print("NETRASCAN.AI V2 — PORTABLE DEPENDENCIES")
print("=" * 75)

assert PORTABLE_PYTHON.is_file()
assert SITE_PACKAGES.is_dir()
assert PORTABLE_V2.resolve() in SITE_PACKAGES.resolve().parents
assert PORTABLE_V1.resolve() not in SITE_PACKAGES.resolve().parents

env = os.environ.copy()

env.pop("PYTHONPATH", None)
env.pop("PYTHONHOME", None)

env["PYTHONNOUSERSITE"] = "1"
env["PYTHONDONTWRITEBYTECODE"] = "1"

protected = [
    "torch",
    "torchvision",
    "timm",
    "gradio",
    "numpy",
    "pandas",
    "Pillow",
    "grad-cam",
    "opencv-python",
    "plotly"
]

inspection_code = f"""
import json
import importlib.metadata as m

versions = {{}}

for package in {protected!r}:
    versions[package] = m.version(package)

print(json.dumps(versions))
"""


def inspect_protected():

    result = subprocess.run(
        [
            str(PORTABLE_PYTHON),
            "-B",
            "-c",
            inspection_code
        ],
        capture_output=True,
        text=True,
        check=True,
        env=env
    )

    return json.loads(result.stdout)


print("\n1. RECORDING PROTECTED PACKAGE VERSIONS")
print("-" * 75)

before = inspect_protected()

for name, version in before.items():
    print(name, ":", version)


print("\n2. INSTALLING MISSING PACKAGES")
print("-" * 75)

packages = [
    "requests==2.32.3",
    "reportlab==4.4.4"
]

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--target",
    str(SITE_PACKAGES),
    "--no-deps",
    "--disable-pip-version-check"
] + packages

result = subprocess.run(
    command,
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

assert result.returncode == 0

print("PASS — Requests and ReportLab installed.")



dependencies = [
    "urllib3",
    "certifi",
    "charset-normalizer",
    "idna"
]

transitive_inspection_code = f"""
import json
import importlib.metadata as m

versions = {{}}

for package in {dependencies!r}:
    try:
        versions[package] = m.version(package)
    except m.PackageNotFoundError:
        versions[package] = None

print(json.dumps(versions))
"""

result = subprocess.run(
    [str(PORTABLE_PYTHON), "-B", "-c", transitive_inspection_code],
    capture_output=True,
    text=True,
    check=True,
    env=env
)

portable_versions = json.loads(result.stdout)

missing = []

print("\n1. CHECKING TRANSITIVE DEPENDENCIES")
print("-" * 75)

for package in dependencies:

    desktop_version = metadata.version(package)
    portable_version = portable_versions[package]

    print(
        package,
        "| Desktop:", desktop_version,
        "| Portable:", portable_version
    )

    if portable_version is None:
        missing.append(f"{package}=={desktop_version}")


print("\n2. INSTALLING MISSING DEPENDENCIES")
print("-" * 75)

if missing:

    print("Installing:", missing)

    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--target",
        str(SITE_PACKAGES),
        "--no-deps",
        "--disable-pip-version-check",
        "--progress-bar",
        "off"
    ] + missing

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        env=env
    )

    print(result.stdout)

    if result.stderr:
        print(result.stderr)

    assert result.returncode == 0

else:
    print("All transitive dependencies already installed.")

### 85. Verify portable imports, CUDA, PDF generation and package integrity

In [ ]:
# STEP 85 — Verify portable imports, CUDA, PDF generation and package integrity
env["MPLBACKEND"] = "Agg"
# Cell 22D.2 — Corrected Dependency Integrity Verification

import subprocess
import json

print("=" * 75)
print("NETRASCAN.AI V2 — FINAL DEPENDENCY VERIFICATION")
print("=" * 75)

expected_versions = {
    "torch": "2.9.0+cu128",
    "torchvision": "0.24.0+cu128",
    "timm": "1.0.22",
    "gradio": "6.32.0",
    "numpy": "2.4.6",
    "pandas": "2.3.3",
    "Pillow": "13.0.0",
    "grad-cam": "1.5.7",
    "opencv-python": "5.0.0.93",
    "plotly": "6.9.0"
}

protected_inspection_code = f"""
import json
import importlib.metadata as m

versions = {{
    package: m.version(package)
    for package in {list(expected_versions)!r}
}}

print(json.dumps(versions))
"""

print("\n1. VERIFYING PROTECTED PACKAGE VERSIONS")
print("-" * 75)

result = subprocess.run(
    [
        str(PORTABLE_PYTHON),
        "-B",
        "-c",
        protected_inspection_code
    ],
    capture_output=True,
    text=True,
    check=True,
    env=env,
    cwd=str(PORTABLE_V2)
)

actual_versions = json.loads(result.stdout)

mismatches = []

for package, expected in expected_versions.items():

    actual = actual_versions.get(package)

    if actual == expected:
        print(f"PASS — {package} {actual} unchanged.")
    else:
        mismatches.append(package)
        print(
            f"FAIL — {package}: "
            f"expected {expected}, found {actual}"
        )

assert not mismatches, f"Version mismatches: {mismatches}"


print("\n2. VERIFYING COMPLETE PORTABLE IMPORTS")
print("-" * 75)

final_smoke_code = """
import sys
import io
import torch
import torchvision
import timm
import gradio
import numpy
import pandas
import cv2
import plotly
import requests
import reportlab
import urllib3
import certifi
import charset_normalizer
import idna
import pytorch_grad_cam

from reportlab.pdfgen import canvas

print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available()

print("GPU:", torch.cuda.get_device_name(0))

tensor = torch.ones((256, 256), device="cuda")
output = tensor @ tensor

torch.cuda.synchronize()

assert output[0, 0].item() == 256.0

print("PASS — CUDA tensor computation.")

print("Gradio:", gradio.__version__)
print("Requests:", requests.__version__)
print("ReportLab:", reportlab.Version)

buffer = io.BytesIO()

pdf = canvas.Canvas(buffer)
pdf.drawString(50, 750, "NetraScan.AI V2 Portable")
pdf.save()

assert buffer.getvalue().startswith(b"%PDF")

print("PASS — PDF generation.")
print("PASS — All required imports.")
"""


# Cell 22D.3 — Portable Matplotlib Backend Correction

import subprocess
import json

print("=" * 75)
print("NETRASCAN.AI V2 — PORTABLE BACKEND CORRECTION")
print("=" * 75)

print("\n1. CORRECTING SUBPROCESS ENVIRONMENT")
print("-" * 75)

print("Previous backend:", env.get("MPLBACKEND"))

env["MPLBACKEND"] = "Agg"
env["PYTHONNOUSERSITE"] = "1"
env["PYTHONDONTWRITEBYTECODE"] = "1"

env.pop("PYTHONPATH", None)
env.pop("PYTHONHOME", None)

print("Corrected backend:", env["MPLBACKEND"])

print("PASS — Headless Matplotlib backend configured.")


print("\n2. REPEATING COMPLETE PORTABLE SMOKE TEST")
print("-" * 75)

result = subprocess.run(
    [
        str(PORTABLE_PYTHON),
        "-B",
        "-c",
        final_smoke_code
    ],
    capture_output=True,
    text=True,
    env=env,
    cwd=str(PORTABLE_V2),
    timeout=180
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

assert result.returncode == 0, (
    "Portable smoke test failed. Inspect traceback above."
)

print("PASS — Complete portable smoke test.")


print("\n3. VERIFYING PROTECTED PACKAGE VERSIONS")
print("-" * 75)

result = subprocess.run(
    [
        str(PORTABLE_PYTHON),
        "-B",
        "-c",
        protected_inspection_code
    ],
    capture_output=True,
    text=True,
    check=True,
    env=env,
    cwd=str(PORTABLE_V2)
)

actual_versions = json.loads(result.stdout)

assert actual_versions == expected_versions

for package, version in actual_versions.items():
    print(f"PASS — {package} {version} unchanged.")


print("\n4. SAVING FINAL VERIFICATION")
print("-" * 75)

summary = {
    "status": "PASS",
    "requests": "2.32.3",
    "reportlab": "4.4.4",
    "additional_dependencies_installed": [
        "urllib3==2.4.0",
        "charset-normalizer==3.4.1"
    ],
    "matplotlib_backend": "Agg",
    "protected_packages_expected": expected_versions,
    "protected_packages_actual": actual_versions,
    "protected_packages_unchanged": True,
    "cuda_computation_test": True,
    "pdf_smoke_test": True,
    "http_import_test": True,
    "complete_import_test": True
}

summary_path = (
    PORTABLE_V2 / "tests" / "portable_dependencies.json"
)

summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("PASS — Final verification saved.")

print("\n" + "=" * 75)
print("PASS — PORTABLE V2 DEPENDENCIES VERIFIED")
print("Matplotlib backend corrected.")
print("Grad-CAM imports verified.")
print("CUDA computation verified.")
print("All protected package versions unchanged.")
print("No additional installations.")
print("V1 installation untouched.")
print("Portable launcher creation pending.")
print("=" * 75)

### 86. Create standalone Windows launcher

In [ ]:
# STEP 86 — Create standalone Windows launcher
# Cell 22E — Standalone Portable Launcher Creation

from pathlib import Path
import subprocess
import hashlib
import json
import os

PORTABLE_V1 = Path(r"E:\NetraScan.AI_PORTABLE")
PORTABLE_V2 = Path(r"E:\NetraScan.AI_V2_PORTABLE")

LAUNCHER_PATH = PORTABLE_V2 / "START_NETRASCAN_V2.bat"

print("=" * 75)
print("NETRASCAN.AI V2 — PORTABLE LAUNCHER CREATION")
print("=" * 75)

assert PORTABLE_V2.is_dir()

v1_launcher = PORTABLE_V1 / "START_NETRASCAN.bat"

v1_hash_before = hashlib.sha256(
    v1_launcher.read_bytes()
).hexdigest()

print("\n1. VERIFYING REQUIRED COMPONENTS")
print("-" * 75)

required_files = [
    "runtime/python/python.exe",
    "app.py",
    "src/inference.py",
    "src/explainability.py",
    "src/llm_reporter.py",
    "src/pdf_reporter.py",
    "llm/manifest.json",
    "llm/backend/llama-server.exe",
    "llm/weights/qwen2.5-3b-instruct-q4_k_m.gguf",
    "models/vit_v4/vit_v4_best.pth",
    "models/swin_v4/swin_v4_best.pth",
    "models/mobilevit_v4/mobilevit_v4_best.pth"
]

for filename in required_files:

    assert (PORTABLE_V2 / filename).is_file(), filename

    print("PASS —", filename)


print("\n2. CREATING PORTABLE BAT LAUNCHER")
print("-" * 75)

launcher_content = r"""@echo off
setlocal
title NetraScan.AI V2 - Portable Launcher

cd /d "%~dp0"

set "MPLBACKEND=Agg"
set "PYTHONNOUSERSITE=1"
set "PYTHONDONTWRITEBYTECODE=1"
set "PYTHONPATH="
set "PYTHONHOME="
set "GRADIO_ANALYTICS_ENABLED=False"
set "HF_HUB_OFFLINE=1"
set "TRANSFORMERS_OFFLINE=1"

echo ==========================================================
echo                    NetraScan.AI V2
echo              Three Transformers. One Vision.
echo ==========================================================
echo.
echo Portable AI Research Prototype
echo Vision Transformers + Grad-CAM + Local GenAI
echo.
echo Project: %CD%
echo.

if not exist "runtime\python\python.exe" goto missing_runtime
if not exist "app.py" goto missing_app
if not exist "llm\backend\llama-server.exe" goto missing_llm
if not exist "llm\weights\qwen2.5-3b-instruct-q4_k_m.gguf" goto missing_llm

if not exist "reports" mkdir "reports"
if not exist "logs" mkdir "logs"

echo Checking application port...

"runtime\python\python.exe" -B -c "import socket; s=socket.socket(); s.settimeout(2); r=s.connect_ex(('127.0.0.1',7860)); s.close(); raise SystemExit(1 if r==0 else 0)"

if errorlevel 1 goto port_busy

echo PASS - Port 7860 available.
echo.
echo Starting NetraScan.AI V2...
echo.
echo Application URL:
echo http://127.0.0.1:7860
echo.
echo Your browser should open automatically.
echo Keep this window open while using NetraScan.AI.
echo Use Stop Application to shut down safely.
echo.
echo Runtime log: logs\application.log
echo.

"runtime\python\python.exe" -B -u "app.py" > "logs\application.log" 2>&1

set "APP_EXIT=%ERRORLEVEL%"

echo.
echo ==========================================================

if "%APP_EXIT%"=="0" (
    echo NetraScan.AI V2 stopped successfully.
) else (
    echo Application exited with code %APP_EXIT%.
    echo.
    echo Check logs\application.log for details.
)

echo ==========================================================
pause
exit /b %APP_EXIT%

:missing_runtime
echo ERROR: Portable Python runtime not found.
goto failed

:missing_app
echo ERROR: Application source not found.
goto failed

:missing_llm
echo ERROR: Local GenAI components not found.
goto failed

:port_busy
echo ERROR: Port 7860 is already occupied.
echo Close the existing application before restarting.
goto failed

:failed
pause
exit /b 1
"""

with LAUNCHER_PATH.open(
    "w",
    encoding="ascii",
    newline="\r\n"
) as f:
    f.write(launcher_content)

print("PASS — Portable BAT launcher created.")
print("Launcher:", LAUNCHER_PATH)


print("\n3. VERIFYING DRIVE-LETTER INDEPENDENCE")
print("-" * 75)

assert 'cd /d "%~dp0"' in launcher_content
assert 'set "MPLBACKEND=Agg"' in launcher_content
assert 'set "PYTHONPATH="' in launcher_content
assert 'set "PYTHONHOME="' in launcher_content

for letter in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":

    assert f"{letter}:\\" not in launcher_content
    assert f"{letter}:/" not in launcher_content

print("PASS — No hardcoded drive letters.")
print("PASS — Relative runtime and application paths.")
print("PASS — Headless Matplotlib configuration.")
print("PASS — Offline model settings.")


print("\n4. VERIFYING APPLICATION IMPORT FROM ANOTHER DIRECTORY")
print("-" * 75)

portable_env = os.environ.copy()

portable_env.pop("PYTHONPATH", None)
portable_env.pop("PYTHONHOME", None)

portable_env.update({
    "MPLBACKEND": "Agg",
    "PYTHONNOUSERSITE": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
    "GRADIO_ANALYTICS_ENABLED": "False",
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1"
})

portable_python = (
    PORTABLE_V2 / "runtime" / "python" / "python.exe"
)

import_test = f"""
import sys
from pathlib import Path

root = Path({str(PORTABLE_V2)!r})
sys.path.insert(0, str(root))

import app

assert app.APP_ROOT.resolve() == root.resolve()
assert app.engine is None
assert app.reporter.process is None

print("Application root:", app.APP_ROOT)
print("Working directory:", Path.cwd())
print("PASS — Application imported successfully.")
print("PASS — Application root resolved independently.")
print("PASS — Vision models remain lazily initialized.")
print("PASS — Local LLM remains lazily initialized.")

app.reporter.close()
"""

result = subprocess.run(
    [
        str(portable_python),
        "-B",
        "-c",
        import_test
    ],
    cwd=str(PORTABLE_V2.parent),
    env=portable_env,
    capture_output=True,
    text=True,
    timeout=180
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

assert result.returncode == 0


print("\n5. VERIFYING ORIGINAL V1 LAUNCHER")
print("-" * 75)

v1_hash_after = hashlib.sha256(
    v1_launcher.read_bytes()
).hexdigest()

assert v1_hash_before == v1_hash_after

print("PASS — Original V1 launcher unchanged.")


print("\n6. SAVING LAUNCHER VERIFICATION")
print("-" * 75)

summary = {
    "status": "PASS",
    "launcher": LAUNCHER_PATH.name,
    "relative_paths": True,
    "hardcoded_drive_letters": False,
    "matplotlib_backend": "Agg",
    "offline_model_settings": True,
    "application_import_verified": True,
    "lazy_initialization_verified": True,
    "v1_launcher_unchanged": True,
    "live_application_test_pending": True
}

summary_path = (
    PORTABLE_V2 / "tests" / "portable_launcher.json"
)

summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("PASS — Launcher verification saved.")

print("\n" + "=" * 75)
print("PASS — PORTABLE V2 LAUNCHER CREATED")
print("Independent Python runtime ready.")
print("Application import verified.")
print("Drive-independent launcher configured.")
print("Original V1 launcher preserved.")
print("Live portable application test pending.")
print("=" * 75)

## 11. Deployment acceptance and limitations
The portable application was manually reported working on desktop and laptop. The launcher resolves its directory dynamically, uses embedded Python and logs startup output. Full reproducibility requires the external artifacts and compatible NVIDIA drivers. This notebook does not publish private reports or laptop performance measurements.